# REM sleep without atonia across open sleep cohorts

Everything runs in the cloud; nothing is downloaded to your computer. Every
dataset used here is open (no application, no credential):

| cohort | what | license |
|---|---|---|
| CAP | 22 RBD patients + healthy sleepers + other sleep disorders (Italy) | ODC-By |
| HMC | 151 clinical PSGs, Haaglanden Medisch Centrum (Netherlands) | CC BY 4.0 |
| Challenge 2018 | 994 clinical PSGs, Massachusetts General Hospital (USA) | ODC-By |

Each night is streamed from PhysioNet's open mirror, reduced to a 1-2 MB file
on this Colab machine and copied to Google Drive (`MyDrive/rbd-rwa/packs`),
and the raw file is deleted. If the session disconnects, run the cells again:
nights already copied to Drive are restored and skipped.

Run order: **Runtime -> Run all**. The first cell asks for access to Drive;
if nobody answers within 3 minutes the run continues without it.

In [ ]:
# Work happens on this machine's disk; finished nights are ALSO copied to
# Google Drive when it is connected. (Colab's Drive connection can drop out
# under heavy use, so it is never the working copy.)
import os
from google.colab import drive
try:
    drive.mount('/content/drive', timeout_ms=180000)
    os.makedirs('/content/drive/MyDrive/rbd-rwa', exist_ok=True)
    MIRROR = '/content/drive/MyDrive/rbd-rwa'
except Exception as e:
    print('Drive not connected:', e, '- results stay on this Colab machine for now')
    MIRROR = ''
PACK_MIRROR = f'--mirror {MIRROR}/packs' if MIRROR else ''
RESULT_MIRROR = f'--mirror {MIRROR}/results' if MIRROR else ''
os.environ['RBD_DURABLE_DIR'] = '/content/durable'
!pip -q install wfdb pyedflib
!mkdir -p /content/rbd-rwa/src /content/durable
%cd /content/rbd-rwa

In [ ]:
%%writefile /content/rbd-rwa/src/__init__.py

In [ ]:
%%writefile /content/rbd-rwa/src/config.py
"""Paths that work identically on Colab, locally, or anywhere else.

The guiding rule: RAW recordings are disposable, DERIVED data is precious.
Raw PSG is hundreds of MB per night and lives in scratch (Colab's ephemeral
disk); the extracted envelope + hypnogram is a few MB and lives in a durable
directory (Google Drive), so a disconnected session never loses real work.
"""
import os
from pathlib import Path


def _first_existing(*candidates):
    for c in candidates:
        if c and Path(c).exists():
            return Path(c)
    return None


ON_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

# Durable storage: Drive on Colab, the project folder otherwise.
DRIVE = _first_existing(
    os.environ.get("RBD_DRIVE_DIR"),
    "/content/drive/MyDrive",
)
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DURABLE = Path(os.environ.get("RBD_DURABLE_DIR",
                              (DRIVE / "rbd-rwa") if DRIVE else PROJECT_ROOT / "data"))

# Scratch: fast, large, and wiped when the session ends. Raw EDFs only.
SCRATCH = Path(os.environ.get("RBD_SCRATCH_DIR",
                              "/content/scratch" if ON_COLAB else PROJECT_ROOT / "data" / "raw"))

DERIVED   = DURABLE / "derived"      # one compact .npz per recording
MANIFESTS = DURABLE / "manifests"    # progress records, so a restart resumes
FIGURES   = DURABLE / "figures"
RESULTS   = DURABLE / "results"

for d in (DERIVED, MANIFESTS, FIGURES, RESULTS, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)

# Envelope sampling rate. Phasic RWA bursts are >= 0.1 s, so 50 Hz (20 ms bins)
# gives 5 samples across the shortest event a method needs to see, while being
# ~100x smaller than the raw signal.
ENV_FS = 50

# EMG band. 10 Hz high-pass removes movement/drift and ECG baseline; 100 Hz is
# the conventional upper edge, lowered automatically when Nyquist forbids it.
EMG_BAND = (10.0, 100.0)

# A second, common band that every recording can supply. Chin EMG in CAP is
# sampled anywhere from 100 to 512 Hz, and amplitude depends on bandwidth, so
# comparing amplitudes ACROSS recordings needs the same band for all of them.
# 45 Hz is the highest edge a 100 Hz recording allows (Nyquist minus 5).
HARMONIZED_BAND = (10.0, 45.0)


def describe():
    print(f"on Colab : {ON_COLAB}")
    print(f"durable  : {DURABLE}   (derived data, survives disconnects)")
    print(f"scratch  : {SCRATCH}   (raw EDFs, disposable)")

In [ ]:
%%writefile /content/rbd-rwa/src/edf.py
"""Minimal EDF/EDF+ reader that returns each channel at its NATIVE rate.

Why not MNE: `mne.io.read_raw_edf` resamples every channel to the highest rate
in the file. CAP stores chin EMG at 256-512 Hz but leg EMG at 128 Hz, so the
leg channels silently came back upsampled to 512 Hz. Reading the samples
directly keeps each channel's true rate, reads only the channels asked for,
and needs nothing beyond numpy - which matters on hosted notebooks where
installing packages is not always possible.

Values are returned in microvolts when the header's physical dimension is a
voltage. Some files carry no unit at all (raw converter counts); those are
returned unscaled and flagged, because an absolute amplitude threshold means
nothing for them.
"""
from dataclasses import dataclass
from pathlib import Path

import numpy as np

_TO_UV = {"uv": 1.0, "µv": 1.0, "mv": 1e3, "v": 1e6, "nv": 1e-3}


@dataclass
class Channel:
    index: int
    label: str
    dim: str
    fs: float
    n_per_record: int
    gain: float          # physical units per digital unit
    offset: float        # physical value at digital zero
    prefilter: str
    transducer: str

    @property
    def uv_scale(self):
        """Factor from physical units to microvolts, or None if not a voltage."""
        return _TO_UV.get(self.dim.strip().lower().replace("μ", "µ"))


@dataclass
class EdfHeader:
    path: Path
    n_records: int
    record_s: float
    header_bytes: int
    start_date: str
    start_time: str
    channels: list

    @property
    def duration_s(self):
        return self.n_records * self.record_s

    def find(self, label):
        for c in self.channels:
            if c.label == label:
                return c
        raise KeyError(label)


def read_header(path) -> EdfHeader:
    path = Path(path)
    with open(path, "rb") as f:
        h = f.read(256)
        ns = int(h[252:256])
        s = f.read(256 * ns)

    def fields(width, start):
        return [s[start + i * width: start + (i + 1) * width]
                .decode("latin-1").strip() for i in range(ns)], start + width * ns

    labels, o = fields(16, 0)
    trans, o = fields(80, o)
    dims, o = fields(8, o)
    pmin, o = fields(8, o)
    pmax, o = fields(8, o)
    dmin, o = fields(8, o)
    dmax, o = fields(8, o)
    pref, o = fields(80, o)
    nsamp, o = fields(8, o)

    header_bytes = int(h[184:192])
    n_records = int(h[236:244])
    record_s = float(h[244:252])
    per_rec = [int(x) for x in nsamp]
    # Records actually present. The header may say "-1" (unknown), or claim
    # more records than a truncated file holds; trust the file size.
    on_disk = (path.stat().st_size - header_bytes) // (2 * sum(per_rec))
    n_records = on_disk if n_records < 0 else min(n_records, on_disk)

    chans = []
    for i in range(ns):
        p0, p1, d0, d1 = (float(pmin[i]), float(pmax[i]),
                          float(dmin[i]), float(dmax[i]))
        gain = (p1 - p0) / (d1 - d0) if d1 != d0 else 1.0
        chans.append(Channel(
            index=i, label=labels[i], dim=dims[i],
            fs=per_rec[i] / record_s if record_s > 0 else float("nan"),
            n_per_record=per_rec[i], gain=gain, offset=p0 - gain * d0,
            prefilter=pref[i], transducer=trans[i]))
    return EdfHeader(path, n_records, record_s, header_bytes,
                     h[168:176].decode("ascii", "ignore").strip(),
                     h[176:184].decode("ascii", "ignore").strip(), chans)


def read_channel(hdr: EdfHeader, channel):
    """
    One channel as float64 at its native rate, in microvolts when the unit is
    a voltage. `channel` is a Channel, an index, or a label (first match).
    Returns (signal, fs, unit_known).
    """
    if isinstance(channel, Channel):
        ch = channel
    elif isinstance(channel, int):
        ch = hdr.channels[channel]
    else:
        ch = hdr.find(channel)
    per_rec = [c.n_per_record for c in hdr.channels]
    rec_len = sum(per_rec)
    start = sum(per_rec[:ch.index])
    raw = np.memmap(hdr.path, dtype="<i2", mode="r", offset=hdr.header_bytes,
                    shape=(hdr.n_records, rec_len))
    x = np.asarray(raw[:, start:start + ch.n_per_record], dtype=np.float64).ravel()
    del raw
    x = x * ch.gain + ch.offset
    k = ch.uv_scale
    if k is not None:
        x *= k
    return x, ch.fs, k is not None

In [ ]:
%%writefile /content/rbd-rwa/src/ecg.py
"""Remove the heartbeat from chin EMG.

The chin electrodes pick up the ECG. Each QRS complex adds a ~50-100 ms
deflection to the chin signal, so in quiet REM sleep - exactly where RWA is
measured - every heartbeat can look like a small muscle burst. In CAP's
healthy controls the chin amplitude at the R-peak is up to 2.5x the
surrounding level, enough to cross the 2x-background threshold of SINBAR.
Automated RWA tools remove it (RBDtector uses template subtraction), and so
do we:

  1. find R-peaks on the ECG channel
  2. average the filtered chin EMG around them, in 5-minute blocks, into a
     heartbeat template (a median, so genuine muscle bursts do not leak in)
  3. subtract the template at every beat
"""
import numpy as np

ECG_BIPOLAR = ("ECG1-ECG2", "EKG1-EKG2")
ECG_PAIRS = (("ECG1", "ECG2"), ("EKG1", "EKG2"))
ECG_PREFIX = ("ECG", "EKG")


def _norm(label):
    return label.upper().replace(" ", "")


def find_ecg(hdr):
    """The ECG source: a channel, a (positive, negative) pair, or None."""
    chans = hdr.channels
    for c in chans:
        if any(p in _norm(c.label) for p in ECG_BIPOLAR):
            return c
    lab = {_norm(c.label): c for c in chans}
    for a, b in ECG_PAIRS:
        if a in lab and b in lab and lab[a].fs == lab[b].fs:
            return (lab[a], lab[b])
    return next((c for c in chans if _norm(c.label).startswith(ECG_PREFIX)), None)


def r_peaks(ecg, fs, window_s=30.0):
    """
    R-peak times in seconds. Band-pass 5-20 Hz, orient so the QRS points up,
    then keep peaks >= 0.3 s apart (<= 200 bpm) that exceed 35% of the local
    99th percentile, computed in `window_s` windows so that slow changes in
    ECG amplitude through the night do not break detection.
    """
    from scipy.signal import butter, sosfiltfilt, find_peaks
    if fs < 50 or len(ecg) < fs * 60:
        return np.empty(0)
    sos = butter(3, [5.0, min(20.0, fs / 2 - 1)], btype="band", fs=fs, output="sos")
    y = sosfiltfilt(sos, np.asarray(ecg, dtype=np.float64))
    if abs(np.percentile(y, 0.1)) > abs(np.percentile(y, 99.9)):
        y = -y
    w = int(window_s * fs)
    n = int(np.ceil(len(y) / w))
    level = np.array([np.percentile(y[i * w:(i + 1) * w], 99) for i in range(n)])
    height = np.repeat(0.35 * level, w)[: len(y)]
    peaks, _ = find_peaks(y, height=height, distance=int(0.3 * fs))
    return peaks / fs


def plausible(peaks_s, duration_s):
    """True when the detected beats look like a heart rate, not noise."""
    if len(peaks_s) < 100:
        return False
    rate = 60.0 * len(peaks_s) / duration_s
    rr = np.diff(peaks_s)
    return 35.0 <= rate <= 130.0 and np.mean((rr > 0.33) & (rr < 2.0)) > 0.9


def subtract_heartbeat(x, fs, peaks_s, pre_s=0.10, post_s=0.15, block_s=300.0):
    """
    Subtract a local heartbeat template from a FILTERED signal. Returns the
    cleaned copy. Segments never overlap because beats are >= 0.3 s apart.
    """
    from scipy.signal.windows import tukey
    x = np.array(x, dtype=np.float64, copy=True)
    a, b = int(round(pre_s * fs)), int(round(post_s * fs))
    idx = np.round(np.asarray(peaks_s) * fs).astype(np.int64)
    idx = idx[(idx - a >= 0) & (idx + b <= len(x))]
    if len(idx) < 20:
        return x
    seg = idx[:, None] + np.arange(-a, b)[None, :]
    beats = x[seg]
    taper = tukey(a + b, alpha=0.3)
    overall = np.median(beats, axis=0)
    block = (idx / fs // block_s).astype(np.int64)
    for k in np.unique(block):
        m = block == k
        tmpl = np.median(beats[m], axis=0) if m.sum() >= 20 else overall
        x[seg[m]] -= tmpl * taper
    return x


def artifact_ratio(env, env_fs, peaks_s):
    """
    Heartbeat-locked envelope profile: the median at each lag around the
    R-peak, peak (-40..+80 ms) over baseline (-500..-300 ms). ~1 means no
    heartbeat artifact; 2.5 means the beat dominates quiet EMG. Medians at
    each lag (not a maximum over noisy single beats) keep this unbiased.
    """
    i = np.round(np.asarray(peaks_s) * env_fs).astype(np.int64)
    lags = np.arange(-int(0.5 * env_fs), int(0.1 * env_fs) + 1)
    i = i[(i + lags[0] >= 0) & (i + lags[-1] < len(env))]
    if len(i) < 50:
        return float("nan")
    prof = np.array([np.median(env[i + k]) for k in lags])
    t = lags / env_fs
    base = np.median(prof[(t >= -0.5) & (t <= -0.3)])
    peak = prof[(t >= -0.04) & (t <= 0.08)].max()
    return float(peak / base) if base > 0 else float("nan")

In [ ]:
%%writefile /content/rbd-rwa/src/fetch.py
"""Resumable downloads. PhysioNet's TLS drops connections mid-transfer, so
every request retries and partial files resume with an HTTP Range header.

Open PhysioNet databases are mirrored byte-for-byte on AWS Open Data
(s3://physionet-open), which is far faster from cloud notebooks than
physionet.org, so the mirror is tried first and PhysioNet is the fallback."""
import time
import urllib.request
from pathlib import Path

CAP_MIRRORS = (
    "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/",
    "https://physionet.org/files/capslpdb/1.0.0/",
)
CAP_BASE = CAP_MIRRORS[-1]


def remote_size(url, tries=5):
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method="HEAD")
            return int(urllib.request.urlopen(req, timeout=45)
                       .headers.get("Content-Length", 0))
        except Exception:
            if i == tries - 1:
                return None
            time.sleep(3)


def download(url, dest: Path, tries=10, quiet=False):
    """Fetch `url` to `dest`, resuming if a partial file is already there."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    total = remote_size(url)
    if total is None:
        raise IOError(f"cannot reach {url}")
    if dest.exists() and dest.stat().st_size == total:
        return dest
    for i in range(tries):
        have = dest.stat().st_size if dest.exists() else 0
        if have == total:
            return dest
        if have > total:                          # stale file from another source
            dest.unlink()
            have = 0
        try:
            headers = {"Range": f"bytes={have}-"} if have else {}
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "ab") as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
        except Exception as e:
            if not quiet:
                print(f"    {dest.name}: {type(e).__name__}, resuming "
                      f"({dest.stat().st_size / 1e6 if dest.exists() else 0:.0f} MB)")
            time.sleep(3)
    got = dest.stat().st_size if dest.exists() else 0
    if got != total:
        raise IOError(f"{dest.name}: got {got} of {total} bytes")
    return dest


def download_any(name, dest: Path, mirrors=CAP_MIRRORS):
    """Try each mirror in turn; the first one that completes wins."""
    last = None
    for base in mirrors:
        try:
            return download(base + name, dest, quiet=True)
        except Exception as e:
            last = e
    raise IOError(f"{name}: all mirrors failed ({last})")


def cap_records():
    """All CAP record names, grouped by cohort prefix."""
    txt = None
    for base in CAP_MIRRORS:
        try:
            txt = urllib.request.urlopen(base + "RECORDS", timeout=45).read().decode()
            break
        except Exception:
            continue
    if txt is None:
        raise IOError("cannot fetch the CAP RECORDS list from any mirror")
    recs = [r.replace(".edf", "") for r in txt.split()]
    groups = {}
    for r in recs:
        key = "".join(c for c in r if not c.isdigit())
        groups.setdefault(key, []).append(r)
    return groups


def cap_demographics():
    """
    Age and sex of every CAP subject, from the database's gender-age.xlsx,
    keyed by record name. The sheet has no header and two typos ("SBD" for
    sdb, "RDB11" for rbd11). Read with the standard library, so no Excel
    package is needed. Returns {} if no mirror answers.
    """
    import io
    import zipfile
    import xml.etree.ElementTree as ET
    raw = None
    for base in CAP_MIRRORS:
        try:
            raw = urllib.request.urlopen(base + "gender-age.xlsx", timeout=60).read()
            break
        except Exception:
            continue
    if raw is None:
        return {}
    z = zipfile.ZipFile(io.BytesIO(raw))
    m = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    shared = ["".join(t.text or "" for t in si.iter(m + "t"))
              for si in ET.fromstring(z.read("xl/sharedStrings.xml")).iter(m + "si")]
    out = {}
    for row in ET.fromstring(z.read("xl/worksheets/sheet1.xml")).iter(m + "row"):
        vals = []
        for c in row.iter(m + "c"):
            v = c.find(m + "v")
            vals.append(shared[int(v.text)] if c.get("t") == "s"
                        else (v.text if v is not None else ""))
        if len(vals) < 3:
            continue
        rec = vals[0].strip().lower().replace("sbd", "sdb").replace("rdb", "rbd")
        try:
            out[rec] = {"sex": vals[1].strip().upper()[:1], "age": float(vals[2])}
        except ValueError:
            continue
    return out


def fetch_cap(record, scratch: Path):
    """Download one CAP recording (.edf + .txt) into scratch."""
    edf = download_any(record + ".edf", Path(scratch) / f"{record}.edf")
    ann = download_any(record + ".txt", Path(scratch) / f"{record}.txt")
    return edf, ann

In [ ]:
%%writefile /content/rbd-rwa/src/cap.py
"""Parse CAP Sleep Database annotations and align them to the EDF clock.

The annotation file is a RemLogic export whose epochs are stamped with WALL
CLOCK times ("23:07:19"), while the signal array starts at whatever time the
EDF header declares. Aligning them is the step most likely to go silently
wrong: an unnoticed offset shifts every sleep stage relative to the signal and
quietly corrupts every downstream measurement, so the offset is computed
explicitly here and reported.
"""
import datetime as dt
import re
from pathlib import Path

import numpy as np

# W / S1 / S2 / S3 / S4 / R / MT  ->  compact codes
STAGE_CODE = {"W": 0, "S1": 1, "S2": 2, "S3": 3, "S4": 3, "R": 4}
STAGE_NAME = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM", -1: "excluded"}
EPOCH_SEC = 30


def edf_start_datetime(edf_path: Path) -> dt.datetime:
    """Recording start from the EDF header (bytes 168-183)."""
    with open(edf_path, "rb") as f:
        head = f.read(184)
    d = head[168:176].decode("ascii", "ignore").strip()   # dd.mm.yy
    t = head[176:184].decode("ascii", "ignore").strip()   # hh.mm.ss
    day, mon, yr = (int(x) for x in re.split(r"[.\-/]", d))
    hh, mm, ss = (int(x) for x in re.split(r"[.\-:]", t))
    # EDF year is two digits: 85-99 -> 19xx, 00-84 -> 20xx (EDF spec clipping rule)
    yr += 1900 if yr >= 85 else 2000
    return dt.datetime(yr, mon, day, hh, mm, ss)


def parse_annotations(txt_path: Path):
    """Return (times, stages) for every scored 30 s epoch, in file order."""
    raw = Path(txt_path).read_bytes().decode("latin-1")
    lines = [l.rstrip("\r") for l in raw.split("\n")]

    # The table begins at the header row; column order varies between files.
    hdr_i = next(i for i, l in enumerate(lines) if l.startswith("Sleep Stage"))
    cols = [c.strip() for c in lines[hdr_i].split("\t")]
    i_stage = cols.index("Sleep Stage")
    i_time = next(i for i, c in enumerate(cols) if c.startswith("Time"))
    i_event = cols.index("Event") if "Event" in cols else None

    times, stages = [], []
    for line in lines[hdr_i + 1:]:
        parts = line.split("\t")
        if len(parts) <= max(i_stage, i_time):
            continue
        # Only sleep-stage rows; CAP files also carry MCAP-A1/A2/A3 arousal events
        if i_event is not None and i_event < len(parts):
            if not parts[i_event].strip().upper().startswith("SLEEP"):
                continue
        stage = parts[i_stage].strip().upper()
        tm = parts[i_time].strip()
        m = re.match(r"(\d{1,2})[:.](\d{2})[:.](\d{2})", tm)
        if not m:
            continue
        times.append(tuple(int(g) for g in m.groups()))
        stages.append(STAGE_CODE.get(stage, -1))
    return times, np.array(stages, dtype=np.int8)


def build_hypnogram(edf_path: Path, txt_path: Path, n_seconds: float):
    """
    Hypnogram on the EDF's own timeline: one code per 30 s epoch from recording
    start. Returns (hypnogram, info).
    """
    start = edf_start_datetime(edf_path)
    times, stages = parse_annotations(txt_path)
    if len(times) == 0:
        raise ValueError(f"{Path(txt_path).name}: no scored epochs found")

    # Walk the clock forward, adding a day whenever it wraps past midnight.
    day = 0
    prev = None
    offsets = []
    for hh, mm, ss in times:
        cur = hh * 3600 + mm * 60 + ss
        if prev is not None and cur < prev - 3600:   # wrapped midnight
            day += 1
        prev = cur
        stamp = dt.datetime(start.year, start.month, start.day, hh, mm, ss) \
            + dt.timedelta(days=day)
        offsets.append((stamp - start).total_seconds())

    offsets = np.array(offsets)
    # The first annotation may precede the EDF start by a day if the recording
    # itself began before midnight and the header date is the following day.
    if offsets[0] < -12 * 3600:
        offsets += 24 * 3600

    n_epochs = int(np.floor(n_seconds / EPOCH_SEC))
    hyp = np.full(n_epochs, -1, dtype=np.int8)
    idx = np.round(offsets / EPOCH_SEC).astype(int)
    keep = (idx >= 0) & (idx < n_epochs)
    hyp[idx[keep]] = stages[keep]

    info = {
        "edf_start": start.isoformat(),
        "first_annotation_offset_s": float(offsets[0]),
        "n_annotated": int(len(stages)),
        "n_placed": int(keep.sum()),
        "n_dropped_outside_signal": int((~keep).sum()),
        "n_epochs": int(n_epochs),
        "counts": {STAGE_NAME[k]: int((hyp == k).sum())
                   for k in sorted(STAGE_NAME) if (hyp == k).sum()},
    }
    return hyp, info

In [ ]:
%%writefile /content/rbd-rwa/src/extract.py
"""Turn a raw overnight PSG into a small derived file.

One CAP recording is ~370 MB; what RWA quantification needs is the EMG
amplitude envelope and the hypnogram, a few MB. Extracting and discarding the
raw signal is what makes the project runnable on hosted notebooks, where disk
is ephemeral and sessions end without warning.

Per recording we keep:
    chin  (5, n_bins) float16 - chin EMG amplitude per 20 ms bin, heartbeat
              removed: RMS and mean-rectified in the native band (10-100 Hz
              or as high as the sampling rate allows), the same two in the
              common 10-45 Hz band shared by every recording, and finally the
              native-band RMS WITHOUT heartbeat removal, for comparison
    legs  (n_legs, n_bins) float16 - leg EMG RMS, native band
    hypnogram (n_epochs,) int8   - 30 s sleep stages on the EDF timeline
    meta  json - channels, native rates, bands, units, alignment

TIMING. Bins are exactly 1/ENV_FS seconds long whatever the source rate, so a
bin index converts to time without drift. (An earlier version used a whole
number of samples per bin; at 512 Hz that made 51.2 bins per second instead
of 50 and shifted the hypnogram by ~11 minutes by the end of a night.)
"""
import json
from pathlib import Path

import numpy as np

from .config import ENV_FS, EMG_BAND, HARMONIZED_BAND, DERIVED
from .cap import build_hypnogram, EPOCH_SEC
from .edf import read_header, read_channel
from .ecg import find_ecg, r_peaks, plausible, subtract_heartbeat, artifact_ratio

# Chin rows, all with the heartbeat removed except the last, which keeps it
# so the effect of heartbeat removal can itself be measured.
CHIN_ROWS = ("rms", "mav", "rms_h", "mav_h", "rms_with_ecg")
F16_MAX = 65000.0     # float16 tops out at 65504; clip (and count) above this

# How chin and leg EMG are labeled across CAP's recording systems. Bipolar
# labels win over split electrodes, which win over a lone referential name.
CHIN_BIPOLAR = ("EMG1-EMG2", "EMG-EMG", "CHIN1-CHIN2")
CHIN_PAIRS = (("EMG1", "EMG2"), ("CHIN1", "CHIN2"))
CHIN_SINGLE = ("CHIN", "MILO", "EMG", "MENT", "SUBM")   # MILO = mylohyoid
LEG_BIPOLAR = ("DX1-DX2", "SX1-SX2")                    # DX/SX = right/left
LEG_CONTAINS = ("TIB",)
LEG_PAIRS = (("DX1", "DX2"), ("SX1", "SX2"))


def _norm(label):
    return label.upper().replace(" ", "")


def _pair(chans, a, b):
    lab = {_norm(c.label): c for c in chans}
    if a in lab and b in lab and lab[a].fs == lab[b].fs:
        return (lab[a], lab[b])
    return None


def find_emg(hdr):
    """
    Locate chin and leg EMG. Each source is either a single channel or a
    (positive, negative) pair to be subtracted. Returns (chin, legs).
    """
    chans = hdr.channels
    chin = next((c for c in chans if _norm(c.label) in CHIN_BIPOLAR
                 or any(p in _norm(c.label) for p in CHIN_BIPOLAR)), None)
    if chin is None:
        chin = next((p for a, b in CHIN_PAIRS if (p := _pair(chans, a, b))), None)
    if chin is None:
        chin = next((c for name in CHIN_SINGLE for c in chans
                     if _norm(c.label) == name), None)

    legs = [c for c in chans if any(p in _norm(c.label) for p in LEG_BIPOLAR)]
    legs += [c for c in chans if any(p in _norm(c.label) for p in LEG_CONTAINS)
             and c not in legs]
    if not legs:
        legs = [p for a, b in LEG_PAIRS if (p := _pair(chans, a, b))]
    return chin, legs


def read_source(hdr, src):
    """Signal for a channel or a (positive, negative) pair of channels."""
    if isinstance(src, tuple):
        a, fs, ka = read_channel(hdr, src[0])
        b, _, kb = read_channel(hdr, src[1])
        return a - b, fs, ka and kb, f"{src[0].label}-{src[1].label}", src[0]
    x, fs, k = read_channel(hdr, src)
    return x, fs, k, src.label, src


def emg_band(fs, band=EMG_BAND):
    """The EMG pass-band actually usable at this sampling rate."""
    lo, hi = band
    hi = min(hi, fs / 2.0 - 5.0)
    if hi <= lo:
        raise ValueError(f"sampling rate {fs} Hz too low for EMG band {band}")
    return lo, hi


def bandpass(sig, fs, band=EMG_BAND, mains=None):
    """Band-pass (and optionally notch) the EMG. Returns (filtered, (lo, hi))."""
    from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

    lo, hi = emg_band(fs, band)
    sos = butter(4, [lo, hi], btype="band", fs=fs, output="sos")
    x = sosfiltfilt(sos, np.asarray(sig, dtype=np.float64))
    if mains:
        # Mains and its harmonics that fall inside the pass-band
        for f0 in np.arange(mains, hi, mains):
            b, a = iirnotch(f0, Q=30.0, fs=fs)
            x = filtfilt(b, a, x)
    return x, (lo, hi)


def envelope(sig, fs, env_fs=ENV_FS, band=EMG_BAND, mains=None, beats_s=None):
    """
    Band-pass the EMG, subtract the heartbeat if R-peak times are given, then
    reduce it to amplitude per bin of exactly 1/env_fs seconds.
    Returns (rms, mav, (lo, hi)).
    """
    x, (lo, hi) = bandpass(sig, fs, band, mains)
    if beats_s is not None and len(beats_s):
        x = subtract_heartbeat(x, fs, beats_s)
    rms, mav = bin_envelope(x, fs, env_fs)
    return rms, mav, (lo, hi)


def bin_envelope(x, fs, env_fs=ENV_FS):
    """RMS and mean-rectified amplitude of an already filtered signal, per bin
    of exactly 1/env_fs seconds."""
    n_bins = int(np.floor(len(x) * env_fs / fs))
    if n_bins == 0:
        raise ValueError("signal shorter than one envelope bin")
    edges = np.floor(np.arange(n_bins + 1) * fs / env_fs).astype(np.int64)
    counts = np.diff(edges)
    rms = np.sqrt(np.add.reduceat(x * x, edges[:-1]) / counts)
    mav = np.add.reduceat(np.abs(x), edges[:-1]) / counts
    return rms.astype(np.float32), mav.astype(np.float32)


def _describe(label, ch, fs, unit_known, band, x):
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    return {"label": label, "fs": fs, "band": list(band),
            "unit": ch.dim, "unit_known": bool(unit_known),
            "transducer": ch.transducer, "prefilter": ch.prefilter,
            # share of samples sitting at the recording's largest magnitude:
            # well above ~1e-4 means the recorder was clipping
            "clipped_frac": float(np.mean(np.abs(x) >= 0.999 * peak)) if peak > 0 else 0.0}


def extract_recording(edf_path: Path, txt_path: Path, record: str,
                      out_dir: Path = None, overwrite=False, mains=50.0):
    """Extract one CAP recording to a compact .npz. Returns the output path."""
    out_dir = Path(out_dir or DERIVED)
    out = out_dir / f"{record}.npz"
    if out.exists() and not overwrite:
        return out

    hdr = read_header(edf_path)
    chin_src, leg_srcs = find_emg(hdr)
    if chin_src is None:
        raise ValueError(f"{record}: no chin EMG channel among "
                         f"{[c.label for c in hdr.channels]}")

    # Heartbeats, for removing the ECG from the chin EMG
    ecg_src = find_ecg(hdr)
    beats, ecg_label = np.empty(0), None
    if ecg_src is not None:
        e, fs_e, _, ecg_label, _ = read_source(hdr, ecg_src)
        beats = r_peaks(e, fs_e)
        if not plausible(beats, len(e) / fs_e):
            beats = np.empty(0)
        del e

    x, fs, unit_known, label, ch = read_source(hdr, chin_src)
    rms, mav, band = envelope(x, fs, mains=mains, beats_s=beats)
    rms_h, mav_h, band_h = envelope(x, fs, band=HARMONIZED_BAND, mains=mains,
                                    beats_s=beats)
    rms_raw, _, _ = envelope(x, fs, mains=mains)
    chin_meta = _describe(label, ch, fs, unit_known, band, x)
    chin_meta["band_harmonized"] = list(band_h)
    chin_meta["ecg"] = {
        "channel": ecg_label, "beats": int(len(beats)),
        "heart_rate_bpm": round(60.0 * len(beats) / (len(x) / fs), 1) if len(beats) else None,
        "artifact_ratio_before": round(artifact_ratio(rms_raw, ENV_FS, beats), 3),
        "artifact_ratio_after": round(artifact_ratio(rms, ENV_FS, beats), 3),
    }
    chin_rows = [rms, mav, rms_h, mav_h, rms_raw]
    del x

    leg_rows, leg_meta = [], []
    for src in leg_srcs:
        x, fs, k, label, ch = read_source(hdr, src)
        r, _, b = envelope(x, fs, mains=mains)
        leg_rows.append(r)
        leg_meta.append(_describe(label, ch, fs, k, b, x))
        del x

    n = min(len(r) for r in chin_rows + leg_rows)   # sources can differ by a bin
    chin = np.stack([r[:n] for r in chin_rows])
    legs = np.stack([r[:n] for r in leg_rows]) if leg_rows \
        else np.zeros((0, n), np.float32)

    n_seconds = n / ENV_FS
    hyp, align = build_hypnogram(edf_path, txt_path, n_seconds)

    meta = {
        "record": record,
        "chin_channel": chin_meta["label"],
        "leg_channels": [m["label"] for m in leg_meta],
        "chin_info": chin_meta,
        "leg_info": leg_meta,
        "chin_rows": list(CHIN_ROWS),
        "env_fs": ENV_FS,
        "mains_notch": mains,
        "duration_s": float(n_seconds),
        "epoch_sec": EPOCH_SEC,
        "alignment": align,
    }
    # Anything above float16's range is an artifact (tens of millivolts);
    # clip it rather than store inf, and record how much there was.
    meta["bins_clipped_f16"] = int((chin > F16_MAX).sum() + (legs > F16_MAX).sum())
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name(out.stem + ".tmp.npz")
    np.savez_compressed(tmp, chin=np.minimum(chin, F16_MAX).astype(np.float16),
                        legs=np.minimum(legs, F16_MAX).astype(np.float16),
                        hypnogram=hyp, meta=json.dumps(meta))
    tmp.replace(out)                             # never leave a half-written file
    return out


def load_derived(path):
    """
    Read a derived file back. Returns a dict with float32 arrays:
    chin_rms, chin_mav, chin_rms_h, chin_mav_h, legs (n_legs, n) and
    hypnogram, plus meta.
    """
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    chin = z["chin"].astype(np.float32)
    d = {f"chin_{name}": chin[i] for i, name in enumerate(meta["chin_rows"])}
    d["legs"] = z["legs"].astype(np.float32)
    d["hypnogram"] = z["hypnogram"]
    d["meta"] = meta
    return d


def summarise(path):
    d = load_derived(path)
    meta, hyp = d["meta"], d["hypnogram"]
    rem = int((hyp == 4).sum())
    ci = meta["chin_info"]
    legs = "/".join(f"{m['fs']:g}" for m in meta["leg_info"]) or "-"
    unit = "" if ci["unit_known"] else "  NO-UNITS"
    e = ci.get("ecg", {})
    ecg = (f"ECG artifact x{e['artifact_ratio_before']:.2f}->x{e['artifact_ratio_after']:.2f}"
           if e.get("beats") else "no ECG")
    return (f"{meta['record']:<8} {meta['duration_s']/3600:>5.1f} h  "
            f"chin {ci['label']} @ {ci['fs']:g} Hz{unit}  legs {legs} Hz  "
            f"REM {rem} epochs ({rem*0.5:.0f} min)  {ecg}")

In [ ]:
%%writefile /content/rbd-rwa/src/pack.py
"""Compact per-recording storage: full detail for REM, summaries for the rest.

RWA is measured in REM sleep. Everything else a method needs from the rest of
the night is a background level, which 1-second summaries of N2/N3 provide.
Keeping 20 ms resolution for REM epochs only shrinks a night from ~15 MB to
~2 MB, which is what makes a thousand-night cohort fit in Google Drive.

A pack holds, for chin EMG rows ROWS (all heartbeat-cleaned unless named
otherwise):
    rem        (n_rows, n_rem_epochs, bins_per_epoch) float16
    rem_legs   (n_legs, n_rem_epochs, bins_per_epoch) float16   (RMS)
    rem_epochs (n_rem_epochs,) int32  - epoch numbers in the night
    nrem_1s    (n_rows, n_seconds) float16 - 1 s RMS (rms rows) or mean
               (mav rows) over N2/N3
    hypnogram  (n_epochs,) int8       - the whole night, for context
    epoch_flags {name: (n_epochs,) int8} - e.g. respiratory event, arousal
    meta       json
"""
import json

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)
ROWS = ("rms", "mav", "rms_h", "mav_h", "rms_with_ecg")
F16_MAX = 65000.0


def _one_second(env, env_fs, how):
    n = len(env) // env_fs
    blocks = env[: n * env_fs].reshape(n, env_fs)
    return blocks.mean(axis=1) if how == "mean" else np.sqrt((blocks ** 2).mean(axis=1))


def build_pack(chin, legs, hypnogram, env_fs, epoch_sec, meta, epoch_flags=None):
    """
    chin: dict row -> full-night envelope (float32), for the rows in ROWS that
    exist. legs: list of full-night RMS envelopes. Returns a pack dict.
    """
    per = int(round(env_fs * epoch_sec))
    n_bins = min(len(v) for v in chin.values())
    n_ep = min(len(hypnogram), n_bins // per)
    hyp = np.asarray(hypnogram[:n_ep], dtype=np.int8)
    # the standard rows first, then any others (e.g. noise-ladder copies)
    rows = [r for r in ROWS if r in chin] + [r for r in chin if r not in ROWS]

    rem_ep = np.flatnonzero(hyp == REM_CODE).astype(np.int32)
    idx = rem_ep[:, None] * per + np.arange(per)[None, :]
    rem = np.stack([chin[r][idx] for r in rows]) if len(rem_ep) \
        else np.zeros((len(rows), 0, per), np.float32)
    rem_legs = np.stack([l[idx] for l in legs]) if (legs and len(rem_ep)) \
        else np.zeros((len(legs or []), len(rem_ep), per), np.float32)

    nrem_mask = np.isin(hyp, NREM_CODES).repeat(per)
    nrem_1s = []
    for r in rows:
        env = chin[r][: n_ep * per][nrem_mask]
        nrem_1s.append(_one_second(env, env_fs, "mean" if r.startswith("mav") else "rms"))
    nrem_1s = np.stack(nrem_1s) if nrem_1s else np.zeros((0, 0), np.float32)

    meta = dict(meta, rows=rows, env_fs=env_fs, epoch_sec=epoch_sec,
                n_rem_epochs=int(len(rem_ep)), n_epochs=int(n_ep))
    return {"rem": rem, "rem_legs": rem_legs, "rem_epochs": rem_ep,
            "nrem_1s": nrem_1s, "hypnogram": hyp,
            "epoch_flags": {k: np.asarray(v[:n_ep], np.int8)
                            for k, v in (epoch_flags or {}).items()},
            "meta": meta}


def save_pack(path, pack):
    f16 = lambda a: np.minimum(np.nan_to_num(a, nan=0.0), F16_MAX).astype(np.float16)
    extra = {f"flag_{k}": v for k, v in pack["epoch_flags"].items()}
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, rem=f16(pack["rem"]), rem_legs=f16(pack["rem_legs"]),
                        rem_epochs=pack["rem_epochs"], nrem_1s=f16(pack["nrem_1s"]),
                        hypnogram=pack["hypnogram"], meta=json.dumps(pack["meta"]),
                        **extra)
    tmp.replace(path)
    return path


def load_pack(path):
    z = np.load(path, allow_pickle=False)
    return {"rem": z["rem"].astype(np.float32), "rem_legs": z["rem_legs"].astype(np.float32),
            "rem_epochs": z["rem_epochs"], "nrem_1s": z["nrem_1s"].astype(np.float32),
            "hypnogram": z["hypnogram"],
            "epoch_flags": {k[5:]: z[k] for k in z.files if k.startswith("flag_")},
            "meta": json.loads(str(z["meta"]))}


def pack_from_full(d):
    """Convert a full-night derived file (see extract.load_derived) to a pack."""
    meta = d["meta"]
    chin = {r: d[f"chin_{r}"] for r in ROWS if f"chin_{r}" in d}
    legs = [row for row in d["legs"]] if len(d["legs"]) else []
    m = {k: v for k, v in meta.items() if k not in ("env_fs", "epoch_sec")}
    m["unit_known"] = bool(meta.get("chin_info", {}).get("unit_known", True))
    return build_pack(chin, legs, d["hypnogram"], meta["env_fs"], meta["epoch_sec"], m)

In [ ]:
%%writefile /content/rbd-rwa/src/cohorts.py
"""Open polysomnography cohorts, each reduced to the same compact pack.

Every cohort here is downloadable with no application and no credential:

  cap     CAP Sleep Database (PhysioNet, ODC-By). 22 RBD patients plus
          healthy sleepers and other sleep disorders. Italy, 50 Hz mains.
  chal18  PhysioNet/CinC Challenge 2018 training set (PhysioNet, ODC-By).
          994 clinical PSGs from Massachusetts General Hospital with sleep
          stages, respiratory events and arousals, age and sex. USA, 60 Hz.
  hmc     Haaglanden Medisch Centrum sleep staging database (PhysioNet,
          CC BY 4.0). 151 clinical PSGs. The Netherlands, 50 Hz.

External cohorts (paper/preregistration_external.md):

  boas    Bitbrain Open Access Sleep dataset (OpenNeuro ds005555, CC0). 128
          nights of 100 adults from the general population, Micromed SD plus,
          chin EMG at 256 Hz, no ECG. Spain, 50 Hz.
  dodh    Dreem Open Dataset, healthy (Zenodo 15900394, MIT). 25 healthy
          volunteers, Compumedics Siesta, 250 Hz. France, 50 Hz.
  dodo    Dreem Open Dataset, obstructive sleep apnea (same). 55 patients,
          Somnomedics Somno HD, 250 Hz. Stanford, USA, 60 Hz.

A cohort adapter knows how to list records, fetch one record's files into
scratch, and read from them: chin EMG, ECG, leg EMG (if any), the hypnogram in
30 s epochs from the start of the signal, per-epoch event flags and any
demographics. `extract_pack` turns that into a pack (see pack.py).
"""
import csv
import io
import urllib.request
from pathlib import Path

import numpy as np

from .config import ENV_FS, HARMONIZED_BAND
from .ecg import r_peaks, plausible, artifact_ratio
from .extract import envelope
from .fetch import download_any
from .pack import build_pack

EPOCH_SEC = 30
PHYSIONET = ("https://physionet-open.s3.amazonaws.com/{db}/",
             "https://physionet.org/files/{db}/")


def _mirrors(db):
    return tuple(m.format(db=db) for m in PHYSIONET)


def _read_text(db, name):
    for base in _mirrors(db):
        try:
            return urllib.request.urlopen(base + name, timeout=60).read().decode("latin-1")
        except Exception:
            continue
    raise IOError(f"cannot fetch {db}/{name}")


EEG_PREFERRED = ("C4-M1", "C3-M2", "C4-A1", "C3-A2", "C4A1", "C3A2")


def _central_eeg(channels):
    """A central EEG channel of an EDF (C4 or C3, against a mastoid if possible)."""
    norm = lambda s: s.upper().replace(" ", "").replace("EEG", "")
    for want in EEG_PREFERRED:
        for c in channels:
            if norm(c.label) == want:
                return c
    return next((c for c in channels if norm(c.label).startswith(("C4", "C3"))), None)


# ══════════════════════════════════════════════════════════════════════════════
#  CAP (streamed like the other cohorts, for the noise ladder; the main CAP
#  pipeline goes through run_extract.py and is repacked by run_cohort.py)
# ══════════════════════════════════════════════════════════════════════════════

class CAP:
    name, mains = "cap", 50.0

    def records(self):
        from .fetch import cap_records
        return [r for group in cap_records().values() for r in group]

    def demographics(self):
        return {}

    def fetch(self, rec, scratch):
        from .fetch import fetch_cap
        edf, txt = fetch_cap(rec, scratch)
        return {"edf": edf, "txt": txt}

    def read(self, rec, files):
        from .cap import build_hypnogram
        from .ecg import find_ecg
        from .edf import read_header, read_channel
        from .extract import find_emg, read_source
        hdr = read_header(files["edf"])
        chin_src, _ = find_emg(hdr)
        if chin_src is None:
            raise ValueError(f"{rec}: no chin EMG among {[c.label for c in hdr.channels]}")
        chin, fs, unit_known, label, _ = read_source(hdr, chin_src)
        ecg_src = find_ecg(hdr)
        ecg, fs_e = read_source(hdr, ecg_src)[:2] if ecg_src is not None else (None, None)
        eeg_ch = _central_eeg(hdr.channels)
        eeg, fs_g = read_channel(hdr, eeg_ch)[:2] if eeg_ch else (None, None)
        hyp, _ = build_hypnogram(files["edf"], files["txt"], len(chin) / fs)
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=label,
                    ecg=ecg, fs_ecg=fs_e, legs=[], hypnogram=hyp, flags={}, extra={},
                    eeg=eeg, fs_eeg=fs_g, eeg_label=eeg_ch.label if eeg_ch else None)


# ══════════════════════════════════════════════════════════════════════════════
#  CHALLENGE 2018 (MGH)
# ══════════════════════════════════════════════════════════════════════════════

class Chal18:
    name, db, mains = "chal18", "challenge-2018/1.0.0", 60.0
    STAGES = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}

    def records(self):
        return [r.strip("/").split("/")[-1]
                for r in _read_text(self.db, "training/RECORDS").split()]

    def demographics(self):
        rows = csv.DictReader(io.StringIO(_read_text(self.db, "age-sex.csv")))
        return {r["Record"]: {"age": float(r["Age"]), "sex": r["Sex"]} for r in rows}

    def fetch(self, rec, scratch):
        out = {}
        for ext in ("hea", "mat", "arousal"):
            out[ext] = download_any(f"training/{rec}/{rec}.{ext}",
                                    Path(scratch) / f"{rec}.{ext}", _mirrors(self.db))
        return out

    def read(self, rec, files):
        import wfdb
        base = str(files["hea"])[:-4]
        names = wfdb.rdheader(base).sig_name
        eeg_name = next((n for n in ("C4-M1", "C3-M2") if n in names), None)
        sig = wfdb.rdrecord(base, channel_names=[n for n in ("Chin1-Chin2", "ECG", eeg_name)
                                                 if n in names])
        col = {n: i for i, n in enumerate(sig.sig_name)}

        def channel(name):                        # microvolts, and whether units are known
            if name not in col:
                return None, False
            unit = sig.units[col[name]].lower()
            return sig.p_signal[:, col[name]] * (1e3 if unit == "mv" else 1.0), \
                unit in ("uv", "mv")

        chin, unit_known = channel("Chin1-Chin2")
        ecg, _ = channel("ECG")
        eeg, _ = channel(eeg_name)
        fs = float(sig.fs)
        n_ep = int(len(chin) / fs // EPOCH_SEC)

        ann = wfdb.rdann(base, "arousal")
        hyp = np.full(n_ep, -1, np.int8)
        stage_marks = [(s, self.STAGES[n]) for s, n in zip(ann.sample, ann.aux_note)
                       if n in self.STAGES]
        for (s0, code), nxt in zip(stage_marks, stage_marks[1:] + [(len(chin), None)]):
            a = int(round(s0 / fs / EPOCH_SEC))
            b = int(round(nxt[0] / fs / EPOCH_SEC))
            hyp[a:min(b, n_ep)] = code

        flags = {"resp": np.zeros(n_ep, np.int8), "arousal": np.zeros(n_ep, np.int8)}
        events = {"resp": [], "arousal": []}      # (start, end) in seconds
        open_ev = {}
        for s, note in zip(ann.sample, ann.aux_note):
            if note.startswith("("):
                open_ev[note[1:]] = s
            elif note.endswith(")") and note[:-1] in open_ev:
                label, s0 = note[:-1], open_ev.pop(note[:-1])
                kind = "resp" if label.startswith("resp_") else \
                    "arousal" if label.startswith("arousal_") else None
                if kind:
                    events[kind].append((s0 / fs, s / fs))
                    a = int(s0 / fs // EPOCH_SEC)
                    b = int(s / fs // EPOCH_SEC)
                    flags[kind][a:min(b + 1, n_ep)] = 1
        n_resp = sum(1 for n in ann.aux_note if n.startswith("(resp_"))
        sleep_h = np.isin(hyp, (1, 2, 3, 4)).sum() * EPOCH_SEC / 3600.0
        extra = {"ahi": round(n_resp / sleep_h, 1) if sleep_h > 0 else None}
        return dict(chin=chin, fs=fs, unit_known=unit_known,
                    chin_label="Chin1-Chin2", ecg=ecg, fs_ecg=fs, legs=[],
                    hypnogram=hyp, flags=flags, extra=extra, events=events,
                    eeg=eeg, fs_eeg=fs, eeg_label=eeg_name)


# ══════════════════════════════════════════════════════════════════════════════
#  HAAGLANDEN MEDISCH CENTRUM
# ══════════════════════════════════════════════════════════════════════════════

class HMC:
    name, db, mains = "hmc", "hmc-sleep-staging/1.1", 50.0
    STAGES = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}
    # v1.1 lists 151 recordings: SN001-SN154 minus SN014, SN064 and SN135,
    # which the release notes withdrew for erroneous, unfixable signal data.

    def records(self):
        return [Path(x).stem for x in _read_text(self.db, "RECORDS").split()]

    def demographics(self):
        return {}                       # only group-level age/sex are published

    def fetch(self, rec, scratch):
        return {"edf": download_any(f"recordings/{rec}.edf", Path(scratch) / f"{rec}.edf",
                                    _mirrors(self.db)),
                "txt": download_any(f"recordings/{rec}_sleepscoring.txt",
                                    Path(scratch) / f"{rec}_sleepscoring.txt",
                                    _mirrors(self.db))}

    def read(self, rec, files):
        from .edf import read_header, read_channel
        hdr = read_header(files["edf"])
        chin_ch = next(c for c in hdr.channels if "CHIN" in c.label.upper())
        ecg_ch = next((c for c in hdr.channels if "ECG" in c.label.upper()), None)
        eeg_ch = _central_eeg(hdr.channels)
        chin, fs, unit_known = read_channel(hdr, chin_ch)
        ecg, fs_e = (read_channel(hdr, ecg_ch)[:2]) if ecg_ch else (None, None)
        eeg, fs_g = (read_channel(hdr, eeg_ch)[:2]) if eeg_ch else (None, None)
        n_ep = int(len(chin) / fs // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        text = Path(files["txt"]).read_text(encoding="latin-1").splitlines()
        for line in text[1:]:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 5 or not parts[4].startswith("Sleep stage"):
                continue
            code = self.STAGES.get(parts[4].split()[-1])
            e = int(round(float(parts[2]) / EPOCH_SEC))
            if code is not None and 0 <= e < n_ep:
                hyp[e] = code
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=chin_ch.label,
                    ecg=ecg, fs_ecg=fs_e, legs=[], hypnogram=hyp, flags={}, extra={},
                    eeg=eeg, fs_eeg=fs_g, eeg_label=eeg_ch.label if eeg_ch else None)


class HMCDegraded(HMC):
    """
    HMC recordings re-acquired, in software, the way the MGH chin EMG was:
    optionally scaled by `gain`, resampled to `fs_out` and quantized to
    `lsb_uv` microvolt steps. MGH stores chin EMG at 200 Hz with 1 uV
    resolution, and its quiet-REM level (~0.6 uV) is below that step; HMC's is
    ~1.5 uV. Scaling HMC down to the MGH level first puts the signal where the
    step size can matter. Comparing HMC with its degraded copies isolates how
    much of a site difference that hardware alone can create.
    """
    def __init__(self, name, fs_out=None, lsb_uv=None, gain=None):
        self.name, self.fs_out, self.lsb_uv, self.gain = name, fs_out, lsb_uv, gain

    def read(self, rec, files):
        d = super().read(rec, files)
        x, fs = d["chin"], d["fs"]
        if self.gain:
            x = x * self.gain
        if self.fs_out and self.fs_out != fs:
            from fractions import Fraction
            from scipy.signal import resample_poly
            f = Fraction(int(self.fs_out), int(fs)).limit_denominator(1000)
            x = resample_poly(x, f.numerator, f.denominator)
            fs = float(self.fs_out)
        if self.lsb_uv:
            x = np.round(x / self.lsb_uv) * self.lsb_uv
        d.update(chin=x, fs=fs)
        return d


class Chal18Noisy(Chal18):
    """
    MGH recordings with white noise added to the chin EMG, the reverse test:
    MGH's quiet-REM level (~0.6 uV) raised to HMC's (~1.5 uV), as a noisier
    amplifier or higher electrode impedance would. The noise is seeded by
    record name, so reruns are identical.
    """
    def __init__(self, name, noise_uv):
        self.name, self.noise_uv = name, noise_uv

    def read(self, rec, files):
        import zlib
        d = super().read(rec, files)
        rng = np.random.default_rng(zlib.crc32(rec.encode()))
        d["chin"] = d["chin"] + rng.normal(0.0, self.noise_uv, len(d["chin"]))
        return d


# ══════════════════════════════════════════════════════════════════════════════
#  EXTERNAL COHORTS
# ══════════════════════════════════════════════════════════════════════════════

class BOAS:
    """
    Bitbrain Open Access Sleep dataset (OpenNeuro ds005555, CC0): adults from
    the general population, one or more nights each (participants.tsv 'pid'),
    Micromed SD plus PSG, bipolar chin EMG at 256 Hz, consensus AASM stages of
    three scorers in the events file (0 W, 1 N1, 2 N2, 3 N3, 4 REM; 8 and -2 =
    disconnection / artifact). There is no ECG channel: heartbeats are found
    on the chin EMG itself (see `_beats`).
    """
    name, mains = "boas", 50.0
    BASE = "https://s3.amazonaws.com/openneuro.org/ds005555/"

    def _participants(self):
        txt = urllib.request.urlopen(self.BASE + "participants.tsv", timeout=60).read().decode()
        return list(csv.DictReader(io.StringIO(txt), delimiter="\t"))

    def records(self):
        return [r["participant_id"] for r in self._participants()]

    def demographics(self):
        num = lambda v: float(v) if v not in (None, "", "n/a") else None
        return {r["participant_id"]: {"age": num(r.get("age")), "sex": r.get("sex"),
                                      "pid": r.get("pid")} for r in self._participants()}

    def fetch(self, rec, scratch):
        from .fetch import download
        stem = f"{rec}/eeg/{rec}_task-Sleep_acq-psg"
        return {"edf": download(self.BASE + stem + "_eeg.edf", Path(scratch) / f"{rec}.edf"),
                "tsv": download(self.BASE + stem + "_events.tsv", Path(scratch) / f"{rec}.tsv")}

    def read(self, rec, files):
        from .edf import read_header, read_channel
        hdr = read_header(files["edf"])
        chin_ch = next(c for c in hdr.channels if c.label.upper().endswith("EMG"))
        chin, fs, unit_known = read_channel(hdr, chin_ch)
        eeg_ch = next((c for c in hdr.channels if c.label.upper().endswith(("C4", "C3"))), None)
        eeg, fs_g = read_channel(hdr, eeg_ch)[:2] if eeg_ch else (None, None)
        n_ep = int(len(chin) / fs // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        with open(files["tsv"], encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                e, code = int(round(float(r["onset"]) / EPOCH_SEC)), int(float(r["stage_hum"]))
                if 0 <= e < n_ep and code in (0, 1, 2, 3, 4):
                    hyp[e] = code
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=chin_ch.label,
                    ecg=None, fs_ecg=None, beats_from_chin=True, legs=[], hypnogram=hyp,
                    flags={}, extra={}, eeg=eeg, fs_eeg=fs_g,
                    eeg_label=eeg_ch.label if eeg_ch else None)


class _RangeReader:
    """Byte ranges of one URL, with retries (Zenodo serves ~1.6 MB/s per connection,
    so large members are fetched as many ranges in parallel)."""
    def __init__(self, url):
        self.url = url

    def get(self, a, b, tries=8):
        import time
        for i in range(tries):
            try:
                req = urllib.request.Request(self.url, headers={"Range": f"bytes={a}-{b - 1}"})
                data = urllib.request.urlopen(req, timeout=300).read()
                if len(data) == b - a:
                    return data
            except Exception:
                pass
            time.sleep(min(60, 3 * 2 ** i))
        raise IOError(f"range {a}-{b} of {self.url} failed")


class _RangeFile(io.RawIOBase):
    """A seekable file over HTTP ranges - enough for zipfile to read a central directory."""
    def __init__(self, url, block=1 << 16):
        self.r, self.pos, self.block = _RangeReader(url), 0, block
        head = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        self.size = int(head.headers["Content-Length"])
        self.c0, self.c = -1, b""

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b""
        a, b = self.pos, self.pos + n
        if not (self.c0 <= a and b <= self.c0 + len(self.c)):
            self.c0 = a
            self.c = self.r.get(a, min(self.size, a + max(n, self.block)))
        self.pos = b
        return self.c[a - self.c0: b - self.c0]


class DOD:
    """
    Dreem Open Datasets (Zenodo 15900394, MIT license): one h5 file per night
    inside a zip per dataset, with signals/eeg/*, signals/emg/* and ECG in uV at
    250 Hz and a consensus hypnogram of five scorers (-1 not scored, 0 W, 1 N1,
    2 N2, 3 N3, 4 REM). A record's member is fetched with parallel byte-range
    requests and inflated locally.
    """
    URL = "https://zenodo.org/records/15900394/files/{}.zip?download=1"
    UNSCORED = {"5d8e93a5-d26a-436d-bc2c-e9527dc92cbf"}   # no consensus (release notes)
    FS = 250.0

    def __init__(self, name, mains, threads=8, chunk=32 << 20):
        self.name, self.mains, self.threads, self.chunk = name, mains, threads, chunk
        self._members = None

    def members(self):
        if self._members is None:
            import zipfile
            z = zipfile.ZipFile(_RangeFile(self.URL.format(self.name)))
            self._members = {Path(i.filename).stem: i for i in z.infolist()
                             if i.filename.endswith(".h5") and "__MACOSX" not in i.filename
                             and i.file_size > 1e6}
        return self._members

    def records(self):
        return sorted(r for r in self.members() if r not in self.UNSCORED)

    def demographics(self):
        return {}

    def fetch(self, rec, scratch):
        import struct
        import zlib
        from concurrent.futures import ThreadPoolExecutor
        info = self.members()[rec]
        rr = _RangeReader(self.URL.format(self.name))
        head = rr.get(info.header_offset, info.header_offset + 30)
        n, m = struct.unpack("<HH", head[26:30])
        start = info.header_offset + 30 + n + m
        parts = [(a, min(a + self.chunk, start + info.compress_size))
                 for a in range(start, start + info.compress_size, self.chunk)]
        out = Path(scratch) / f"{rec}.h5"
        inflate = zlib.decompressobj(-15) if info.compress_type == 8 else None
        with ThreadPoolExecutor(self.threads) as ex, open(out, "wb") as f:
            for i in range(0, len(parts), self.threads):          # in order, bounded memory
                for data in ex.map(lambda p: rr.get(*p), parts[i: i + self.threads]):
                    f.write(inflate.decompress(data) if inflate else data)
            if inflate:
                f.write(inflate.flush())
        if out.stat().st_size != info.file_size:
            raise IOError(f"{rec}: inflated {out.stat().st_size} of {info.file_size} bytes")
        return {"h5": out}

    def read(self, rec, files):
        import h5py
        is_ecg = lambda k: "ECG" in k.upper() or "EKG" in k.upper()
        with h5py.File(files["h5"], "r") as h:
            sig = h["signals"]
            keys = [k for k in sig["emg"].keys() if not is_ecg(k)]
            emg_name = next((k for k in keys if any(t in k.upper() for t in
                                                    ("CHIN", "EMG", "MENT", "SUBM"))), keys[0])
            chin = np.asarray(sig["emg"][emg_name][:], dtype=np.float64)
            ecg = None
            for group in sig.values():
                if isinstance(group, h5py.Group):
                    for k in group.keys():
                        if is_ecg(k):
                            ecg = np.asarray(group[k][:], dtype=np.float64)
            eeg_name = next((k for k in ("C3_M2", "C4_M1") if k in sig["eeg"]), None)
            eeg = np.asarray(sig["eeg"][eeg_name][:], dtype=np.float64) if eeg_name else None
            stages = np.asarray(h["hypnogram"][:])
        n_ep = int(len(chin) / self.FS // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        k = min(n_ep, len(stages))
        hyp[:k] = np.where(np.isin(stages[:k], (0, 1, 2, 3, 4)), stages[:k], -1)
        return dict(chin=chin, fs=self.FS, unit_known=True, chin_label=f"emg/{emg_name}",
                    ecg=ecg, fs_ecg=self.FS if ecg is not None else None, legs=[],
                    hypnogram=hyp, flags={}, extra={}, eeg=eeg, fs_eeg=self.FS,
                    eeg_label=eeg_name)


# 0.39 = median quiet-REM level MGH / HMC (0.59 / 1.52 uV). 1.5 uV of white noise
# at 200 Hz leaves ~1.4 uV in the 10-95 Hz band: sqrt(0.59^2 + 1.4^2) ~ 1.52 uV.
COHORTS = {"chal18": Chal18(), "hmc": HMC(),
           "hmc_q1uv": HMCDegraded("hmc_q1uv", lsb_uv=1.0),
           "hmc_mgh": HMCDegraded("hmc_mgh", fs_out=200, lsb_uv=1.0),
           "hmc_low": HMCDegraded("hmc_low", gain=0.39),
           "hmc_low_mgh": HMCDegraded("hmc_low_mgh", fs_out=200, lsb_uv=1.0, gain=0.39),
           "chal18_noise": Chal18Noisy("chal18_noise", noise_uv=1.5),
           "cap": CAP(),
           "boas": BOAS(), "dodh": DOD("dodh", mains=50.0), "dodo": DOD("dodo", mains=60.0)}


# ══════════════════════════════════════════════════════════════════════════════
#  COMMON: raw signals -> pack
# ══════════════════════════════════════════════════════════════════════════════

def _beats(d):
    """R-peak times (s) from the record's ECG, or none if they do not look real.
    Records without ECG may ask for the chin EMG itself to be searched for the
    heartbeat it carries (same detector, same plausibility test)."""
    if d.get("ecg") is not None:
        sig, fs = d["ecg"], d["fs_ecg"]
    elif d.get("beats_from_chin"):
        sig, fs = d["chin"], d["fs"]
    else:
        return np.empty(0)
    beats = r_peaks(sig, fs)
    return beats if plausible(beats, len(sig) / fs) else np.empty(0)


def extract_pack(cohort, rec, d, demo=None):
    """Envelopes, heartbeat removal and packing for one record's signals."""
    beats = _beats(d)
    x, fs, mains = d["chin"], d["fs"], cohort.mains
    rms, mav, band = envelope(x, fs, mains=mains, beats_s=beats)
    rms_h, mav_h, band_h = envelope(x, fs, band=HARMONIZED_BAND, mains=mains, beats_s=beats)
    rms_raw, _, _ = envelope(x, fs, mains=mains)
    legs = [envelope(l, fs_l, mains=mains)[0] for l, fs_l in d.get("legs", [])]
    meta = {
        "record": rec, "cohort": cohort.name, "chin_channel": d["chin_label"],
        "chin_fs": fs, "band": list(band), "band_harmonized": list(band_h),
        "unit_known": bool(d["unit_known"]), "mains_notch": mains,
        "ecg": {"beats": int(len(beats)),
                "artifact_ratio_before": round(artifact_ratio(rms_raw, ENV_FS, beats), 3),
                "artifact_ratio_after": round(artifact_ratio(rms, ENV_FS, beats), 3)},
        "duration_s": round(len(x) / fs, 1),
        **(demo or {}), **d.get("extra", {}),
    }
    chin = {"rms": rms, "mav": mav, "rms_h": rms_h, "mav_h": mav_h, "rms_with_ecg": rms_raw}
    return build_pack(chin, legs, d["hypnogram"], ENV_FS, EPOCH_SEC, meta, d.get("flags"))


# ══════════════════════════════════════════════════════════════════════════════
#  THE NOISE LADDER: the same night at several noise floors
# ══════════════════════════════════════════════════════════════════════════════

LADDER_UV = (0.5, 1.0, 1.5, 2.0, 3.0)   # in-band RMS of the added noise, uV
TARGETS_UV = (1.5, 2.0)                 # quiet-REM level after harmonization, uV
EEG_BAND = (40.0, 90.0)                 # where EEG is mostly the system's own noise


def quiet_level(rms, hypnogram, env_fs=ENV_FS):
    """5th percentile of 1 s RMS over REM: the background the relative
    criteria are multiples of ('p5_rem' in rwa.py)."""
    from .rwa import mini_epoch_rms
    per = int(round(env_fs * EPOCH_SEC))
    hyp = np.asarray(hypnogram)[: len(rms) // per]
    rem = np.flatnonzero(hyp == 4)
    if len(rem) == 0:
        return float("nan")
    r, _ = mini_epoch_rms(rms[(rem[:, None] * per + np.arange(per)[None, :]).reshape(-1)],
                          env_fs, 1.0)
    return float(np.percentile(r, 5)) if len(r) >= 10 else float("nan")


def eeg_floor(eeg, fs, hypnogram, mains, band=EEG_BAND):
    """The 40-90 Hz floor (5th percentile of 1 s RMS) of an EEG channel in N2/N3
    and in REM: an estimate of the recording system's own noise, measured on a
    channel other than the chin."""
    from .extract import bandpass, bin_envelope
    if eeg is None or not fs or fs / 2.0 - 5.0 < band[0] + 10.0:
        return {}
    x, (lo, hi) = bandpass(eeg, fs, band, mains)
    sec, _ = bin_envelope(x, fs, env_fs=1)
    n = min(len(sec), len(hypnogram) * EPOCH_SEC)
    stage = np.repeat(np.asarray(hypnogram), EPOCH_SEC)[:n]
    out = {"band": [lo, hi]}
    for name, codes in (("nrem", (2, 3)), ("rem", (4,))):
        v = sec[:n][np.isin(stage, codes)]
        out[f"floor_{name}_uv"] = round(float(np.percentile(v, 5)), 4) if len(v) >= 60 else None
    return out


def extract_ladder_pack(cohort, rec, d, demo=None):
    """
    The night's chin EMG as recorded (native band, heartbeat removed) and with
    white noise added in band: at each LADDER_UV level, and at the level that
    raises its quiet-REM level to each TARGETS_UV. Filtering and heartbeat
    removal are linear, so the signal and one unit-RMS noise track (seeded by
    record name) are filtered once and every copy is their sum - the same noise
    draw scaled, i.e. common random numbers across levels.
    """
    import zlib
    from .ecg import subtract_heartbeat
    from .extract import bandpass, bin_envelope
    beats = _beats(d)
    x, fs, mains, hyp = d["chin"], d["fs"], cohort.mains, d["hypnogram"]
    xf, band = bandpass(x, fs, mains=mains)
    if len(beats):
        xf = subtract_heartbeat(xf, fs, beats)
    rms, mav = bin_envelope(xf, fs)
    chin = {"rms": rms, "mav": mav}
    level = quiet_level(rms, hyp)
    ladder = {}
    if d["unit_known"] and np.isfinite(level):
        rng = np.random.default_rng(zlib.crc32(rec.encode()))
        nf, _ = bandpass(rng.normal(0.0, 1.0, len(x)), fs, mains=mains)
        edge = int(5 * fs)
        nf /= np.sqrt(np.mean(nf[edge:-edge] ** 2))
        for u in LADDER_UV:
            r, m = bin_envelope(xf + u * nf, fs)
            chin[f"rms_n{round(u * 100):03d}"], chin[f"mav_n{round(u * 100):03d}"] = r, m
            ladder[f"n{round(u * 100):03d}"] = {"added_uv": u, "level_uv": round(quiet_level(r, hyp), 3)}
        for t in TARGETS_UV:
            # The 5th percentile catches seconds where the noise itself dips, so
            # sqrt(t^2 - level^2) undershoots; rescale until within 1% of t.
            s, r, m, got = float(np.sqrt(max(t * t - level * level, 0.0))), rms, mav, level
            for _ in range(4):
                if s <= 0 or abs(got - t) <= 0.01 * t:
                    break
                r, m = bin_envelope(xf + s * nf, fs)
                got = quiet_level(r, hyp)
                if abs(got - t) > 0.01 * t:
                    s *= np.sqrt((t * t - level * level) / max(got * got - level * level, 1e-9))
            chin[f"rms_h{round(t * 100):03d}"], chin[f"mav_h{round(t * 100):03d}"] = r, m
            ladder[f"h{round(t * 100):03d}"] = {"added_uv": round(s, 3), "level_uv": round(got, 3)}
        del nf
    meta = {
        "record": rec, "cohort": cohort.name, "chin_channel": d["chin_label"],
        "chin_fs": fs, "band": list(band), "unit_known": bool(d["unit_known"]),
        "mains_notch": mains, "duration_s": round(len(x) / fs, 1),
        "level_uv": round(level, 4) if np.isfinite(level) else None, "ladder": ladder,
        "ecg": {"beats": int(len(beats))},
        "eeg": dict(eeg_floor(d.get("eeg"), d.get("fs_eeg"), hyp, mains),
                    channel=d.get("eeg_label")),
        **(demo or {}), **d.get("extra", {}),
    }
    return build_pack(chin, [], hyp, ENV_FS, EPOCH_SEC, meta, d.get("flags"))

In [ ]:
%%writefile /content/rbd-rwa/src/rwa.py
"""The published methods for quantifying REM sleep without atonia (RWA).

Each method reduces one night to one or more percentages (or an index) that are
compared against a diagnostic cut-off. They differ in epoch length, amplitude
thresholds, what counts as "background", and which cut-off applies:

  Montreal  Montplaisir et al., Mov Disord 2010;25:2044 (after Lapierre &
            Montplaisir, Neurology 1992). Tonic: 20 s epochs, >50% of the
            epoch at >= 2x background or > 10 uV; positive at >= 30%.
            Phasic: 2 s mini-epochs containing bursts > 4x background;
            positive at >= 15%.
  SINBAR    Frauscher et al., Sleep 2012;35:835. Phasic bursts 0.1-5 s at
            > 2x background; tonic as Montreal but on 30 s epochs; "any"
            activity in 3 s mini-epochs. Mentalis-only "any" cut-off 18.2%
            (100% specificity). The better-known 32% needs bilateral flexor
            digitorum superficialis EMG, which most recordings lack.
  AASM      AASM scoring manual. Tonic epoch: >= 50% of a 30 s epoch above
            the minimum NREM amplitude. Phasic epoch: >= 5 of its ten 3 s
            mini-epochs contain 0.1-5 s bursts >= 4x background. The manual
            gives no numeric cut-off for "excessive".
  RAI       REM Atonia Index, Ferri et al., J Sleep Res 2008 and Sleep Med
            2010;11:947 ("improved" computation: subtract the minimum of the
            surrounding 60 one-second mini-epochs). RAI < 0.8 is the commonly
            used abnormal range.

Five COMPUTERIZED indices (RAI among them) were compared on the same 146
recordings by Cesari et al., Sleep 2018;41(10):zsy133. This module puts the
visual-criteria methods and the RAI on one code path so they can be applied
unchanged to large clinical cohorts.

OPERATIONALIZING "BACKGROUND". The visual methods judge background by eye.
Here it is an explicit, named choice (`baseline_amplitude`), because it
changes every downstream percentage. Values still listed in a method's
`verify` tuple are secondary-source readings not yet checked against the
original paper.
"""
from dataclasses import dataclass, field

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)
# The 5th percentile of 1 s RMS values of band-limited white noise, relative to
# its RMS (0.910-0.913 for 10-95/100 Hz at 200, 256 and 512 Hz, simulated).
NOISE_P5_RATIO = 0.91


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED MACHINERY
# ══════════════════════════════════════════════════════════════════════════════

def mini_epoch_rms(env, env_fs, seconds, how="rms"):
    """
    Split an envelope into consecutive mini-epochs and summarize each: "rms"
    for an RMS envelope (gives the RMS of the underlying signal), "mean" for a
    mean-rectified envelope (gives its average rectified amplitude).
    """
    w = int(round(env_fs * seconds))
    n = len(env) // w
    if n == 0:
        return np.empty(0), w
    blocks = env[: n * w].reshape(n, w)
    if how == "mean":
        return blocks.mean(axis=1), w
    return np.sqrt((blocks ** 2).mean(axis=1)), w


def stage_mask_at(env_fs, hypnogram, epoch_sec, codes, n_samples):
    """Boolean mask over envelope samples for the given sleep-stage codes."""
    per = int(round(env_fs * epoch_sec))
    mask = np.zeros(n_samples, dtype=bool)
    for i, code in enumerate(hypnogram):
        if code in codes:
            a = i * per
            if a >= n_samples:
                break
            mask[a: min(a + per, n_samples)] = True
    return mask


def baseline_amplitude(env, env_fs, hypnogram, epoch_sec, how, seconds=1.0):
    """
    Background EMG level against which activity is judged, from 1 s windows.

    how = "min_nrem" : lowest window in NREM (the AASM wording; a literal
                       minimum is set by one window, so unstable)
          "p5_nrem"  : 5th percentile in NREM (robust reading of the same)
          "min_rem" / "p5_rem" : the same within REM itself, i.e. the atonic
                       floor of REM - closer to what a visual scorer calls
                       "background" when judging a REM epoch
    """
    codes = NREM_CODES if how.endswith("nrem") else (REM_CODE,)
    m = stage_mask_at(env_fs, hypnogram, epoch_sec, codes, len(env))
    if m.sum() < env_fs * seconds * 10:
        return np.nan
    r, _ = mini_epoch_rms(env[m], env_fs, seconds)
    if len(r) == 0:
        return np.nan
    return float(r.min() if how.startswith("min") else np.percentile(r, 5))


def runs_above(env, env_fs, threshold, min_s=0.1, max_s=None):
    """
    Boolean array marking samples inside runs of consecutive supra-threshold
    samples lasting at least `min_s` (and at most `max_s`, if given). The upper
    bound separates phasic bursts from sustained tonic activity.
    """
    above = env > threshold
    if not above.any():
        return np.zeros(len(env), dtype=bool)
    edges = np.diff(np.r_[0, above.astype(np.int8), 0])
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dur = (stops - starts) / env_fs
    keep = dur >= min_s
    if max_s is not None:
        keep &= dur <= max_s
    # runs never touch (a stop is always followed by a sample below threshold),
    # so +1 at each kept start and -1 at its stop, summed, marks the runs
    step = np.zeros(len(env) + 1, dtype=np.int32)
    step[starts[keep]] += 1
    step[stops[keep]] -= 1
    return np.cumsum(step[:-1]) > 0


def _blocks(mask_idx, width):
    """Consecutive, non-overlapping blocks of `width` indices from an index list."""
    n = len(mask_idx) // width
    return mask_idx[: n * width].reshape(n, width)


# ══════════════════════════════════════════════════════════════════════════════
#  METHOD DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Method:
    name: str
    tonic_epoch_s: float          # tonic scored on epochs of this length
    tonic_mult: float             # tonic threshold, multiples of background
    tonic_abs_uv: float = None    # "... or > X uV": an absolute alternative
    tonic_baseline: str = "p5_rem"
    phasic_mini_s: float = 3.0    # phasic scored on mini-epochs of this length
    phasic_mult: float = 4.0
    phasic_max_s: float = 5.0     # longest burst still counted as phasic
    phasic_baseline: str = "p5_rem"
    any_mini_s: float = None      # SINBAR "any" density on these mini-epochs
    any_mult: float = None
    phasic_epoch_rule: int = None # AASM: >= this many phasic mini-epochs / 30 s
    cutoffs: dict = field(default_factory=dict)   # metric -> % (positive if >=)
    verify: tuple = field(default_factory=tuple)
    # How "k x background" becomes an amplitude threshold (see `threshold`):
    # "relative" as published, "floor" corrected for the noise floor, "clamp"
    # with the background never taken below ref_floor_uv.
    mode: str = "relative"
    ref_floor_uv: float = None
    guard_mult: float = 1.5
    # Where the floor-corrected rule takes its noise estimate from: "rem" (the
    # background b itself, as pre-specified), "nrem" (the quiet N2/N3 level,
    # which cannot contain REM muscle activity) or "min" (the lower of the two).
    noise_from: str = "rem"


def threshold(method, k, b):
    """
    The amplitude (uV RMS) that "activity above k x background" means for a
    night whose background is b.

    relative  k b, as published. Because b is mostly the recording's own noise
              floor, the muscle activity this requires, sqrt(k^2 - 1) b, rises
              and falls with the noise of the recording system.
    floor     max(sqrt(n^2 + (k^2 - 1) B^2), g b) with B = ref_floor_uv and
              n = b / NOISE_P5_RATIO the noise RMS that b implies: the muscle
              activity required is fixed at what the published rule requires
              on a system whose background is B, and the night's noise is
              subtracted (in power) instead of multiplied. The guard g b keeps
              noise alone from being counted: with g = 1.5, white noise puts a
              0.1 s run above threshold in < 1% of mini-epochs (0 for tonic).
    clamp     k max(b, B): the published rule with the background never taken
              to be lower than B (an absolute floor, no subtraction).
    """
    if method.mode == "floor":
        B, n = method.ref_floor_uv, b / NOISE_P5_RATIO
        return max(float(np.sqrt(n * n + (k * k - 1.0) * B * B)), method.guard_mult * b)
    if method.mode == "clamp":
        return k * max(b, method.ref_floor_uv)
    return k * b


def floor_corrected(method, ref_uv, mode="floor", guard=1.5, noise_from="rem"):
    """A method's floor-corrected (or clamped) version, same name and cut-offs."""
    import dataclasses
    return dataclasses.replace(method, mode=mode, ref_floor_uv=float(ref_uv),
                               guard_mult=guard, noise_from=noise_from)


METHODS = {
    "montreal": Method(
        "Montreal", tonic_epoch_s=20.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=2.0, phasic_mult=4.0, phasic_max_s=15.0,
        cutoffs={"tonic_pct": 30.0, "phasic_pct": 15.0},
        verify=("phasic_max_s",),     # 10 s vs 15 s differs between sources
    ),
    "sinbar": Method(
        "SINBAR", tonic_epoch_s=30.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=3.0, phasic_mult=2.0, phasic_max_s=5.0,
        any_mini_s=3.0, any_mult=2.0,
        cutoffs={"any_pct": 18.2},    # mentalis only, 3 s mini-epochs
    ),
    "aasm": Method(
        "AASM", tonic_epoch_s=30.0, tonic_mult=1.0, tonic_baseline="p5_nrem",
        phasic_mini_s=3.0, phasic_mult=4.0, phasic_max_s=5.0,
        phasic_epoch_rule=5,
        cutoffs={},                   # the manual gives none
    ),
}


# ══════════════════════════════════════════════════════════════════════════════
#  SCORING A NIGHT (from a pack: REM epochs at full resolution, see pack.py)
# ══════════════════════════════════════════════════════════════════════════════

# band -> (RMS row, mean-rectified row) of the pack. Any other name X selects
# the rows "rms_X" and "mav_X" (the noise-ladder copies, see cohorts.py).
BANDS = {"native": ("rms", "mav"), "harmonized": ("rms_h", "mav_h"),
         "native_with_ecg": ("rms_with_ecg", None)}


def _select(pack, band, exclude=()):
    """REM epoch numbers and (n_epochs, bins) arrays for a band, minus any
    REM epoch carrying one of the `exclude` flags."""
    rows = pack["meta"]["rows"]
    rk, mk = BANDS.get(band, (f"rms_{band}", f"mav_{band}"))
    ep = pack["rem_epochs"]
    keep = np.ones(len(ep), bool)
    for name in exclude or ():
        flag = pack["epoch_flags"].get(name)
        if flag is not None and len(ep):
            keep &= flag[ep] == 0
    rms = pack["rem"][rows.index(rk)][keep] if rk in rows else None
    mav = pack["rem"][rows.index(mk)][keep] if mk in rows else None
    nrem = pack["nrem_1s"][rows.index(rk)] if (rk in rows and pack["nrem_1s"].size) \
        else np.empty(0)
    return ep[keep], rms, mav, nrem


def _segments(ep):
    """(start, stop) positions of runs of consecutive epoch numbers."""
    if len(ep) == 0:
        return []
    brk = np.flatnonzero(np.diff(ep) != 1) + 1
    return list(zip(np.r_[0, brk], np.r_[brk, len(ep)]))


def _runs(rem2d, ep, env_fs, thr, min_s, max_s):
    """runs_above within each stretch of consecutive REM epochs, so that a
    burst never bridges the gap between two separate REM periods."""
    out = np.zeros(rem2d.shape, bool)
    for a, b in _segments(ep):
        out[a:b] = runs_above(rem2d[a:b].reshape(-1), env_fs, thr, min_s,
                              max_s).reshape(b - a, -1)
    return out


def _background(rem2d, nrem_1s, env_fs, how):
    """
    Background from 1 s RMS values of REM or of N2/N3 (see baseline_amplitude).
    how = "<stat>_<stage>" with stat "min" or "pNN" (NN-th percentile) and stage
    "rem" or "nrem", e.g. "p5_rem" (the default), "p25_rem", "min_nrem".
    """
    stat, stage = how.split("_", 1)
    if stage == "nrem":
        vals = nrem_1s
    else:
        vals, _ = mini_epoch_rms(rem2d.reshape(-1), env_fs, 1.0)
    if len(vals) < 10:
        return np.nan
    if stat == "min":
        return float(vals.min())
    return float(np.percentile(vals, float(stat[1:])))


def _mini(mask2d, env_fs, seconds):
    """Split concatenated REM into mini-epochs; True where any sample is."""
    flat = mask2d.reshape(-1)
    w = int(round(env_fs * seconds))
    n = len(flat) // w
    return flat[: n * w].reshape(n, w)


def score_night(pack, method: Method, band="native", exclude=(), masks=None):
    """Apply one method to one night. Returns a dict of its measures. If a dict
    is passed as `masks`, it receives the per-unit decisions behind them:
    "tonic" (one per tonic epoch), "phasic" (per phasic mini-epoch) and "any"
    (per SINBAR mini-epoch), on consecutive units of the night's REM."""
    meta = pack["meta"]
    env_fs, epoch_sec = meta["env_fs"], meta["epoch_sec"]
    ep, env, _, nrem = _select(pack, band, exclude)
    out = {"method": method.name,
           "rem_minutes": round(len(ep) * epoch_sec / 60.0, 1)}
    if env is None:
        out["error"] = f"band {band} not stored"
        return out
    if len(ep) < 2:
        out["error"] = "less than one minute of REM"
        return out
    if method.mode != "relative" and not meta.get("unit_known", True):
        out["error"] = "floor correction needs the EMG in physical units"
        return out

    bt = _background(env, nrem, env_fs, method.tonic_baseline)
    bp = _background(env, nrem, env_fs, method.phasic_baseline)
    if not (np.isfinite(bt) and bt > 0 and np.isfinite(bp) and bp > 0):
        out["error"] = "background could not be estimated"
        return out

    # the noise estimate the floor-corrected rule subtracts (default: b itself)
    nt, nph = bt, bp
    if method.mode == "floor" and method.noise_from != "rem":
        bn = _background(env, nrem, env_fs, "p5_nrem")
        if not (np.isfinite(bn) and bn > 0):
            out["error"] = "no N2/N3 noise estimate"
            return out
        nt, nph = (bn, bn) if method.noise_from == "nrem" else (min(bt, bn), min(bp, bn))
        out["noise_basis_uV"] = round(nph, 3)

    # ── tonic ────────────────────────────────────────────────────────────────
    thr_t = threshold(method, method.tonic_mult, nt)
    if method.tonic_abs_uv is not None and meta.get("unit_known", True):
        thr_t = min(thr_t, method.tonic_abs_uv)      # "2x background OR > 10 uV"
    above = _mini(env > thr_t, env_fs, method.tonic_epoch_s)
    tonic = above.mean(axis=1) > 0.5

    # ── phasic ───────────────────────────────────────────────────────────────
    thr_p = threshold(method, method.phasic_mult, nph)
    burst = _runs(env, ep, env_fs, thr_p, 0.1, method.phasic_max_s)
    phasic = _mini(burst, env_fs, method.phasic_mini_s).any(axis=1)

    out.update(background_tonic_uV=round(bt, 3), background_phasic_uV=round(bp, 3),
               tonic_threshold_uV=round(thr_t, 3), phasic_threshold_uV=round(thr_p, 3),
               tonic_pct=round(100.0 * tonic.mean(), 2) if len(tonic) else np.nan,
               phasic_pct=round(100.0 * phasic.mean(), 2) if len(phasic) else np.nan)

    # ── AASM: an epoch is phasic when >= 5 of its ten 3 s mini-epochs are ──
    if method.phasic_epoch_rule:
        w = int(round(env_fs * method.phasic_mini_s))
        per = burst.shape[1]
        minis = burst[:, : (per // w) * w].reshape(len(burst), -1, w).any(axis=2)
        hits = minis.sum(axis=1) >= method.phasic_epoch_rule
        out["phasic_epoch_pct"] = round(100.0 * hits.mean(), 2)

    # ── SINBAR "any": tonic, phasic or both, above the same amplitude ──────
    any_ = None
    if method.any_mini_s:
        active = _runs(env, ep, env_fs, threshold(method, method.any_mult, nph), 0.1, None)
        any_ = _mini(active, env_fs, method.any_mini_s).any(axis=1)
        out["any_pct"] = round(100.0 * any_.mean(), 2)

    if masks is not None:
        masks.update(tonic=tonic, phasic=phasic, any=any_)
    hits = [out.get(k, np.nan) >= v for k, v in method.cutoffs.items()
            if np.isfinite(out.get(k, np.nan))]
    out["positive"] = bool(any(hits)) if hits else None
    return out


def rem_atonia_index(pack, band="native", exclude=(), bin_width=1.0,
                     n_bins=20, noise="moving_min", window_s=60):
    """
    REM Atonia Index. Average rectified amplitude per 1 s mini-epoch of REM,
    noise-corrected, then binned by amplitude:

        RAI = A0 / (1 - A1)

    with A0 the fraction of mini-epochs <= 1 uV and A1 the fraction in
    (1, 2] uV. Runs from 0 (no atonia) to 1 (complete atonia). Uses the
    MEAN-RECTIFIED rows, which must be in microvolts: the bins are absolute.

    noise = "moving_min": subtract the minimum of the `window_s` mini-epochs
            surrounding each one (Ferri et al. 2010, "improved" RAI);
            "global_min": subtract the night's single minimum (2008 version).
    """
    meta = pack["meta"]
    ep, _, env_mav, _ = _select(pack, band, exclude)
    if env_mav is None:
        return {"method": "RAI", "error": f"no mean-rectified row for band {band}"}
    if not meta.get("unit_known", True):
        return {"method": "RAI", "error": "chin EMG has no physical units"}
    if len(ep) < 2:
        return {"method": "RAI", "error": "less than one minute of REM"}
    amp, _ = mini_epoch_rms(env_mav.reshape(-1), meta["env_fs"], 1.0, how="mean")
    if noise == "moving_min":
        from scipy.ndimage import minimum_filter1d
        amp = amp - minimum_filter1d(amp, size=int(window_s), mode="nearest")
    else:
        amp = amp - amp.min()
    edges = np.arange(0, (n_bins + 1) * bin_width, bin_width)
    hist, _ = np.histogram(amp, bins=np.r_[edges, np.inf])
    frac = hist / hist.sum()
    a0, a1 = float(frac[0]), float(frac[1])
    rai = a0 / (1.0 - a1) if (1.0 - a1) > 1e-9 else float("nan")
    return {
        "method": "RAI",
        "rai": round(rai, 4),
        "frac_silent": round(a0, 4),
        "rem_minutes": round(len(ep) * meta["epoch_sec"] / 60.0, 1),
        "positive": bool(rai < 0.8) if np.isfinite(rai) else None,
    }


def score_all(pack, band="native", exclude=()):
    """
    Run every method on one night (a pack; see pack.py).
    band = "native" uses each recording's own EMG band; "harmonized" uses the
    common 10-45 Hz band, the fair choice when comparing ACROSS recordings made
    on different equipment; "native_with_ecg" leaves the heartbeat in.
    exclude = epoch-flag names (e.g. "resp", "arousal") whose REM epochs are
    dropped before scoring.
    """
    rows = [score_night(pack, m, band, exclude) for m in METHODS.values()]
    rows.append(rem_atonia_index(pack, band, exclude))
    for r in rows:
        r["band"] = band
        r["excluded"] = "+".join(exclude) if exclude else ""
    return rows

In [ ]:
%%writefile /content/rbd-rwa/run_extract.py
"""Stream CAP recordings: download -> extract -> discard raw.

Designed for hosted notebooks, where the session can vanish at any moment.
Progress is written to a manifest after every recording, so re-running
continues from where it stopped and never re-downloads finished work. The next
few recordings download in the background while the current one is being
extracted, and raw EDFs are deleted as soon as their derived file exists, so
disk use stays flat no matter how many recordings are processed.

    python run_extract.py                  # every CAP record (default)
    python run_extract.py rbd1 rbd2 n1     # specific records
    python run_extract.py --cohorts rbd n  # whole cohorts by prefix
    python run_extract.py --keep-raw       # keep EDFs (local debugging only)
"""
import argparse
import json
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.fetch import fetch_cap, cap_records
from src.extract import extract_recording, summarise

MANIFEST = config.MANIFESTS / "cap_extract.json"
PREFETCH = 3          # recordings downloading ahead of the one being extracted


def load_manifest():
    if MANIFEST.exists():
        try:
            return json.loads(MANIFEST.read_text())
        except json.JSONDecodeError:
            pass
    return {"done": {}, "failed": {}}


def save_manifest(m):
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(json.dumps(m, indent=2))
    tmp.replace(MANIFEST)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("records", nargs="*")
    ap.add_argument("--cohorts", nargs="*", help="cohort prefixes, e.g. rbd n")
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    config.describe()
    groups = cap_records()
    if args.records:
        records = args.records
    elif args.cohorts:
        records = [r for c in args.cohorts for r in groups.get(c, [])]
    else:
        records = [r for g in groups.values() for r in g]
    print(f"\n{len(records)} records queued "
          f"(cohorts available: { {k: len(v) for k, v in groups.items()} })\n")

    m = load_manifest()
    todo = [r for r in records
            if r not in m["done"] or args.overwrite
            or not (config.DERIVED / f"{r}.npz").exists()]
    print(f"{len(m['done'])} already done, {len(todo)} to process\n")

    pool = ThreadPoolExecutor(max_workers=PREFETCH)
    pending = {}

    def prefetch(i):
        if i < len(todo) and i not in pending:
            pending[i] = pool.submit(fetch_cap, todo[i], config.SCRATCH)

    for i in range(PREFETCH):
        prefetch(i)

    t_start = time.time()
    for i, rec in enumerate(todo):
        prefetch(i + PREFETCH)
        t0 = time.time()
        print(f"[{i + 1}/{len(todo)}] {rec}", flush=True)
        edf = txt = None
        try:
            edf, txt = pending.pop(i).result()
            t_dl = time.time() - t0
            out = extract_recording(edf, txt, rec, overwrite=args.overwrite)
            print(f"    {summarise(out)}  (waited {t_dl:.0f}s for download, "
                  f"{time.time() - t0 - t_dl:.0f}s to extract)", flush=True)
            m["done"][rec] = {"file": out.name, "seconds": round(time.time() - t0, 1)}
            m["failed"].pop(rec, None)
        except Exception as e:
            print(f"    FAILED: {type(e).__name__}: {e}", flush=True)
            traceback.print_exc(limit=2)
            m["failed"][rec] = f"{type(e).__name__}: {e}"
        finally:
            if not args.keep_raw:
                # Raw is disposable once the derived file exists (or failed).
                for p in (edf, txt):
                    if p is not None:
                        Path(p).unlink(missing_ok=True)
        save_manifest(m)
    pool.shutdown(wait=True)

    derived = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    size = sum(p.stat().st_size for p in derived)
    print(f"\n{'=' * 70}")
    print(f"  {len(m['done'])} recordings extracted, {len(m['failed'])} failed "
          f"in {(time.time() - t_start) / 60:.1f} min")
    print(f"  derived data: {len(derived)} files, {size / 1e6:.0f} MB total")
    print(f"  location: {config.DERIVED}")
    if m["failed"]:
        print("\n  failures:")
        for k, v in m["failed"].items():
            print(f"    {k}: {v}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_cohort.py
"""Build packs for a whole open cohort: download -> extract -> discard raw.

    python run_cohort.py --cohort chal18             # 994 MGH PSGs
    python run_cohort.py --cohort hmc                # 148 Haaglanden PSGs
    python run_cohort.py --cohort chal18 --limit 5   # quick test
    python run_cohort.py --cohort cap                # repack CAP's derived files
    python run_cohort.py --cohort hmc --ladder       # noise ladder (see cohorts.py)

Several records are processed at once (--workers), each in its own process,
so downloads overlap with computation. Progress is written to a manifest
after every record, so re-running continues where it stopped. Raw files are
deleted as soon as their pack exists.
"""
import argparse
import json
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.pack import save_pack


def _work(name, rec, demo, scratch, out_path, keep_raw, ladder=False):
    """Runs in a worker process: one record from download to pack."""
    from src.cohorts import COHORTS, extract_pack, extract_ladder_pack
    cohort = COHORTS[name]
    extract = extract_ladder_pack if ladder else extract_pack
    t0 = time.time()
    folder = Path(scratch) / rec
    folder.mkdir(parents=True, exist_ok=True)
    try:
        files = cohort.fetch(rec, folder)
        t_dl = time.time() - t0
        d = cohort.read(rec, files)
        pack = extract(cohort, rec, d, demo)
        save_pack(Path(out_path), pack)
        m = pack["meta"]
        return rec, {"seconds": round(time.time() - t0, 1), "download_s": round(t_dl, 1),
                     "rem_min": round(m["n_rem_epochs"] * 0.5, 1),
                     "ecg": m["ecg"].get("artifact_ratio_before", m["ecg"]["beats"])}, None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=2)[-300:]}"
    finally:
        if not keep_raw:
            shutil.rmtree(folder, ignore_errors=True)


class Mirror:
    """
    Best-effort copy of finished packs to a second place (Google Drive). The
    Drive mount in Colab can drop out under load ("Transport endpoint is not
    connected"), so work happens on local disk and each small pack is copied
    over as it is finished; if the mirror fails, the run carries on.
    """
    def __init__(self, path):
        self.path = Path(path) if path else None
        self.ok = self.path is not None
        if self.ok:
            try:
                self.path.mkdir(parents=True, exist_ok=True)
            except OSError as e:
                self._fail(e)

    def _fail(self, e):
        if self.ok:
            print(f"  (mirror unavailable, continuing without it: {e})", flush=True)
        self.ok = False

    def restore(self, out_dir):
        """Bring back packs saved by an earlier session, so they are skipped."""
        if not self.ok:
            return 0
        n = 0
        try:
            for p in self.path.glob("*.npz"):
                if not (out_dir / p.name).exists():
                    shutil.copy2(p, out_dir / p.name)
                    n += 1
        except OSError as e:
            self._fail(e)
        return n

    def push(self, p):
        if self.ok:
            try:
                shutil.copy2(p, self.path / Path(p).name)
            except OSError as e:
                self._fail(e)


def repack_cap(out_dir, mirror):
    """CAP was extracted to full-night files; convert them to packs."""
    from src.extract import load_derived
    from src.pack import pack_from_full
    files = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    for p in files:
        out = out_dir / p.name
        if out.exists():
            continue
        try:
            pack = pack_from_full(load_derived(p))
        except Exception as e:                       # unreadable file: report, go on
            print(f"  {p.name}: cannot read ({type(e).__name__}: {e})", flush=True)
            continue
        pack["meta"]["cohort"] = "cap"
        save_pack(out, pack)
        mirror.push(out)
    print(f"cap: {len(list(out_dir.glob('*.npz')))} packs in {out_dir}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cohort", required=True)
    ap.add_argument("--limit", type=int, default=0,
                    help="stop once this many records are done (0 = all)")
    ap.add_argument("--workers", type=int, default=3)
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--mirror", default=None,
                    help="also copy each finished pack here (e.g. a Drive folder)")
    ap.add_argument("--ladder", action="store_true",
                    help="build noise-ladder packs (DURABLE/ladder) instead of packs")
    args = ap.parse_args()

    kind = "ladder" if args.ladder else "packs"
    out_dir = config.DURABLE / kind / args.cohort
    out_dir.mkdir(parents=True, exist_ok=True)
    mirror = Mirror(Path(args.mirror) / args.cohort if args.mirror else None)
    restored = mirror.restore(out_dir)
    if restored:
        print(f"restored {restored} packs from {mirror.path}", flush=True)
    if args.cohort == "cap" and not args.ladder:
        repack_cap(out_dir, mirror)
        return

    from src.cohorts import COHORTS
    cohort = COHORTS[args.cohort]
    manifest = config.MANIFESTS / f"{args.cohort}_{kind}.json"
    m = json.loads(manifest.read_text()) if manifest.exists() else {"done": {}, "failed": {}}
    records = cohort.records()
    demo = cohort.demographics()
    todo = [r for r in records if not (out_dir / f"{r}.npz").exists()]
    done = len(records) - len(todo)
    if args.limit:
        todo = todo[: max(0, args.limit - done)]
    config.describe()
    print(f"\n{args.cohort}: {len(records)} records, {done} done, "
          f"{len(todo)} to process with {args.workers} workers\n", flush=True)

    t_start = time.time()
    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(_work, args.cohort, r, demo.get(r), str(config.SCRATCH),
                          str(out_dir / f"{r}.npz"), args.keep_raw, args.ladder)
                for r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            rec, info, err = f.result()
            if err:
                m["failed"][rec] = err
                print(f"[{i}/{len(todo)}] {rec} FAILED {err[:160]}", flush=True)
            else:
                m["done"][rec] = info
                m["failed"].pop(rec, None)
                mirror.push(out_dir / f"{rec}.npz")
                print(f"[{i}/{len(todo)}] {rec}  REM {info['rem_min']} min  "
                      f"ECG x{info['ecg']}  ({info['download_s']}s download, "
                      f"{info['seconds']}s total)", flush=True)
            tmp = manifest.with_suffix(".tmp")
            tmp.write_text(json.dumps(m, indent=1))
            tmp.replace(manifest)

    packs = list(out_dir.glob("*.npz"))
    size = sum(p.stat().st_size for p in packs) / 1e9
    print(f"\n{args.cohort}: {len(packs)} packs ({size:.2f} GB), {len(m['failed'])} failed, "
          f"{(time.time() - t_start) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_score.py
"""Score every pack of every cohort with every RWA method.

Reads DURABLE/packs/<cohort>/*.npz (build them with run_cohort.py), writes one
tidy row per (recording, method, band, exclusion) to results/rwa_scores.csv,
and prints three summaries:

  1. RBD separation within CAP (AUC, 0.5 = chance), against CAP's healthy
     sleepers - most of whom were recorded on different equipment, so partly
     a hardware contrast - and against CAP patients recorded on the same
     system as the RBD group (512 Hz, microvolts), the fair comparison.
  2. How many people WITHOUT known RBD each method's published cut-off flags,
     per cohort: the false-positive burden a screening program would face.
  3. The spread of each metric per cohort (median, 90th, 95th percentile).

    python run_score.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import score_all

CAP_GROUPS = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
              "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB", "brux": "bruxism"}
METRICS = ["tonic_pct", "phasic_pct", "phasic_epoch_pct", "any_pct", "atonia_loss"]
MIN_REM_MIN = 10.0     # nights with less REM than this are not scored in summaries


def group_of(cohort, record):
    if cohort == "cap":
        return CAP_GROUPS.get("".join(c for c in record if not c.isdigit()), "other")
    return "clinical"


def auc(pos, neg):
    """Probability a random positive outscores a random negative (ties = 1/2)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan"), len(pos), len(neg)
    diff = pos[:, None] - neg[None, :]
    return float(((diff > 0) + 0.5 * (diff == 0)).mean()), len(pos), len(neg)


def rem_boundaries(pack):
    """Flag the first and last epoch of every REM period: transitional epochs,
    where scoring conventions differ most between laboratories."""
    flag = np.zeros(len(pack["hypnogram"]), np.int8)
    ep = pack["rem_epochs"]
    if len(ep):
        brk = np.flatnonzero(np.diff(ep) != 1)
        flag[ep[np.r_[0, brk + 1]]] = 1
        flag[ep[np.r_[brk, len(ep) - 1]]] = 1
    return flag


def score_everything():
    rows = []
    for cohort_dir in sorted((config.DURABLE / "packs").glob("*")):
        files = sorted(p for p in cohort_dir.glob("*.npz") if ".tmp" not in p.name)
        print(f"{cohort_dir.name}: {len(files)} packs", flush=True)
        for p in files:
            pack = load_pack(p)
            m = pack["meta"]
            cohort = m.get("cohort", cohort_dir.name)
            info = dict(cohort=cohort, record=m["record"], group=group_of(cohort, m["record"]),
                        age=m.get("age"), sex=m.get("sex"), ahi=m.get("ahi"),
                        chin_fs=m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                        unit_known=m.get("unit_known", True))
            exclusions = [()] + ([("resp", "arousal")] if pack["epoch_flags"] else [])
            pack["epoch_flags"] = dict(pack["epoch_flags"], boundary=rem_boundaries(pack))
            for band in ("native", "harmonized"):
                for ex in exclusions + ([("boundary",)] if band == "native" else []):
                    for r in score_all(pack, band, ex):
                        rows.append({**info, **r})
    df = pd.DataFrame(rows)
    if "rai" in df:
        df["atonia_loss"] = 1 - df["rai"]              # higher = more RWA, like the rest
    return df


def main():
    import argparse
    import shutil
    ap = argparse.ArgumentParser()
    ap.add_argument("--mirror", default=None, help="also copy the results table here")
    args = ap.parse_args()

    df = score_everything()
    for col in ["positive", "error", *METRICS]:          # absent if nothing produced them
        if col not in df:
            df[col] = np.nan
    out = config.RESULTS / "rwa_scores.csv"
    df.to_csv(out, index=False)
    print(f"\nsaved {len(df)} rows -> {out}")
    if args.mirror:
        try:
            Path(args.mirror).mkdir(parents=True, exist_ok=True)
            shutil.copy2(out, Path(args.mirror) / out.name)
            print(f"copied -> {args.mirror}")
        except OSError as e:
            print(f"(could not copy to {args.mirror}: {e})")
    print()
    base = df[(df.excluded == "") & (df.rem_minutes >= MIN_REM_MIN)]

    # 1. RBD separation within CAP ────────────────────────────────────────────
    cap = base[base.cohort == "cap"]
    if len(cap):
        same = (cap.chin_fs == 512) & cap.unit_known.astype(bool)
        print("1. RBD separation in CAP (AUC)           vs healthy      vs same-system patients")
        for (band, meth), g in cap.groupby(["band", "method"], sort=False):
            for metric in METRICS:
                if metric not in g or g[metric].notna().sum() == 0:
                    continue
                s = same.loc[g.index]
                a1, n1, h1 = auc(g.loc[g.group == "RBD", metric], g.loc[g.group == "healthy", metric])
                a2, n2, h2 = auc(g.loc[s & (g.group == "RBD"), metric],
                                 g.loc[s & ~g.group.isin(["RBD", "healthy"]), metric])
                print(f"   {band:<10} {meth:<9} {metric:<17} {a1:6.3f} ({n1}v{h1:<3})   "
                      f"{a2:6.3f} ({n2}v{h2})")
        print()

    # 2. Share flagged positive by published cut-offs ─────────────────────────
    lab = base.assign(who=np.where(base.cohort == "cap", "CAP " + base.group, base.cohort))
    pos = lab.dropna(subset=["positive"])
    pos = pos.assign(positive=pos.positive.astype(float))
    tab = pos.pivot_table(index=["band", "method"], columns="who", values="positive",
                          aggfunc="mean")
    n = pos.groupby("who").record.nunique()
    print("2. Share classified POSITIVE by each published cut-off (%), nights with >= "
          f"{MIN_REM_MIN:.0f} min REM")
    print("   n per group: " + ", ".join(f"{k} {v}" for k, v in n.items()))
    print((100 * tab).round(1).to_string())
    print()

    # 3. Distribution of each metric per cohort ────────────────────────────────
    print("3. Metric distribution per group (native band): median / p90 / p95")
    nat = lab[lab.band == "native"]
    for meth, g in nat.groupby("method", sort=False):
        for metric in METRICS:
            if metric not in g or g[metric].notna().sum() == 0:
                continue
            q = g.groupby("who")[metric].quantile([0.5, 0.9, 0.95]).unstack()
            cells = "  ".join(f"{w}: {r[0.5]:.1f}/{r[0.9]:.1f}/{r[0.95]:.1f}"
                              for w, r in q.iterrows())
            print(f"   {meth:<9} {metric:<17} {cells}")

    # 4. Respiratory events and arousals (cohorts that annotate them) ─────────
    ev = df[df.cohort == "chal18"]
    if len(ev) and (ev.excluded != "").any():
        print("\n4. Effect of removing REM epochs with respiratory events or arousals "
              "(chal18, native band, share positive %)")
        e = ev[(ev.band == "native") & (ev.rem_minutes >= MIN_REM_MIN)].dropna(subset=["positive"])
        print((100 * e.assign(positive=e.positive.astype(float))
               .pivot_table(index="method", columns="excluded", values="positive",
                            aggfunc="mean")).round(1).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_sensitivity.py
"""How much does the operational definition of "background" move the verdicts?

The visual methods judge activity against the background EMG a scorer sees by
eye. An automated version has to choose a number; this script re-scores every
night (native band, all REM) under several choices and reports, for Montreal,
SINBAR and AASM, the share of nights flagged positive per group and the RBD
separation (AUC vs same-system CAP patients). The RAI does not use a
background and is not affected.

    python run_sensitivity.py
"""
import dataclasses
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import METHODS, score_night

# name -> background used by the relative criteria. AASM's tonic criterion is
# defined against the minimum NREM amplitude and keeps that definition.
VARIANTS = ["p5_rem", "p10_rem", "p25_rem", "p50_rem", "min_rem", "p5_nrem"]
MIN_REM = 10.0
CAP_GROUPS = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
              "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB", "brux": "bruxism"}


def auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    d = pos[:, None] - neg[None, :]
    return float(((d > 0) + 0.5 * (d == 0)).mean())


def main():
    rows = []
    for cdir in sorted((config.DURABLE / "packs").glob("*")):
        for p in sorted(cdir.glob("*.npz")):
            pack = load_pack(p)
            m = pack["meta"]
            cohort = m.get("cohort", cdir.name)
            group = CAP_GROUPS.get("".join(c for c in m["record"] if not c.isdigit()), "other") \
                if cohort == "cap" else "clinical"
            for v in VARIANTS:
                for key in ("montreal", "sinbar", "aasm"):
                    meth = METHODS[key]
                    meth = dataclasses.replace(meth, phasic_baseline=v, tonic_baseline=(
                        meth.tonic_baseline if key == "aasm" else v))
                    r = score_night(pack, meth)
                    rows.append({"cohort": cohort, "record": m["record"], "group": group,
                                 "chin_fs": m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                                 "unit_known": m.get("unit_known", True), "variant": v, **r})
        print(f"{cdir.name}: done", flush=True)
    df = pd.DataFrame(rows)
    df = df[df.rem_minutes >= MIN_REM]
    df.to_csv(config.RESULTS / "sensitivity_background.csv", index=False)
    who = np.where(df.cohort == "cap", "CAP " + df.group, df.cohort)
    df = df.assign(who=who)

    print("\nShare flagged POSITIVE (%) by background definition")
    pos = df.dropna(subset=["positive"]).assign(
        positive=lambda d: d.positive.map(lambda x: float(x in (True, "True", 1, 1.0))))
    t = pos.pivot_table(index=["method", "variant"], columns="who", values="positive",
                        aggfunc="mean") * 100
    cols = [c for c in ["CAP RBD", "CAP healthy", "CAP NFLE", "CAP PLM", "chal18", "hmc",
                        "hmc_q1uv", "hmc_mgh", "hmc_low", "hmc_low_mgh", "chal18_noise"]
            if c in t.columns]
    print(t[cols].round(0).to_string())

    print("\nRBD vs same-system CAP patients (AUC) by background definition")
    same = (df.cohort == "cap") & (df.chin_fs == 512) & df.unit_known.astype(bool)
    out = []
    for (meth, v), g in df[same].groupby(["method", "variant"]):
        for metric in ("tonic_pct", "phasic_pct", "any_pct", "phasic_epoch_pct"):
            if metric in g and g[metric].notna().any():
                out.append(dict(method=meth, variant=v, metric=metric,
                                auc=auc(g.loc[g.group == "RBD", metric],
                                        g.loc[~g.group.isin(["RBD", "healthy"]), metric])))
    print(pd.DataFrame(out).pivot_table(index=["method", "metric"], columns="variant",
                                        values="auc").round(3).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_report.py
"""Turn results/rwa_scores.csv into the paper's tables.

    python run_report.py

Prints (and writes to results/report.md):

  T1  cohort description: nights scored, REM minutes, age and sex where known
  T2  share of nights each method's published cut-off flags, per group, with
      95% Wilson intervals - sensitivity in CAP's RBD patients, false-positive
      rate everywhere else
  T3  agreement between methods on the same nights (Cohen's kappa)
  T4  what predicts a false positive in the MGH cohort: age, sex, apnea-
      hypopnea index (logistic regression, one model per method)
  T5  removing REM epochs that contain respiratory events or arousals
  T6  positive predictive value of each method if used to screen a
      population in which a given share has RBD (specificity per hospital)
  T7  cut-offs re-derived for automated scoring (Youden, CAP RBD vs CAP
      patients recorded on the same system) and the external nights they flag
  T9  quiet-REM chin EMG level ("background") per group
  T10 T7 validated: leave-one-out cross-validation, and bootstrap intervals
      for the cut-offs and the external false-positive shares
  T11 positive calls by quiet-REM level within cohorts, and in MGH the odds
      ratio per halving of that level, adjusted for age, sex and AHI
  T12 simulated recordings (hardware, amplitude, noise floor) against the
      cohort they were made from, on the same nights
  T13 MGH positive calls by apnea-hypopnea index, all REM and event-free REM
  T15 the same without transitional epochs (first and last of each REM period)
  T16 the floor signature: quietest N2/N3 over quietest REM chin level
  T17 age in CAP: RBD against same-system patients under 50 and aged 50 or over
  T8  heartbeat artifact in the chin EMG before and after its removal
  T14 MGH positive calls by the residual heartbeat artifact

Nights with less than MIN_REM minutes of REM are left out, as clinical
scoring would.
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config

MIN_REM = 10.0
DECIDING = ["Montreal", "SINBAR", "RAI"]          # methods with a published cut-off
METRICS = ["tonic_pct", "phasic_pct", "phasic_epoch_pct", "any_pct", "atonia_loss"]
# Software copies of a cohort (src/cohorts.py) and the cohort each one copies.
SIMULATED = {"hmc_q1uv": "hmc", "hmc_mgh": "hmc", "hmc_low": "hmc", "hmc_low_mgh": "hmc_low",
             "chal18_noise": "chal18"}
# Recordings withdrawn by the data providers (HMC v1.1: erroneous signal data).
# RECORDS no longer lists them; this guards against older downloads.
WITHDRAWN = {("hmc", "SN014"), ("hmc", "SN064"), ("hmc", "SN135")}


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def paired_change(before, after):
    """Change in the share positive on the same nights: difference with a 95%
    Wald interval for paired proportions, and the exact McNemar test."""
    from scipy.stats import binomtest
    before, after = np.asarray(before, float), np.asarray(after, float)
    n = len(before)
    up = int(((before == 0) & (after == 1)).sum())
    down = int(((before == 1) & (after == 0)).sum())
    d = (up - down) / n
    se = np.sqrt(max(up + down - (up - down) ** 2 / n, 0.0)) / n
    p = binomtest(up, up + down).pvalue if up + down else 1.0
    return (f"diff {100 * d:+.1f} [{100 * (d - 1.96 * se):+.1f} to {100 * (d + 1.96 * se):+.1f}]  "
            f"{down} nights -, {up} +, McNemar p={p:.1g}")


def kappa(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    po = np.mean(a == b)
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


def groups(df):
    """Label each night: CAP subgroups, and the external clinical cohorts."""
    who = np.where(df.cohort == "cap", "CAP " + df.group.astype(str), df.cohort)
    return df.assign(who=who)


def main():
    df = pd.read_csv(config.RESULTS / "rwa_scores.csv")
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df = groups(df)
    # CAP publishes age and sex in a spreadsheet, not in the recordings
    from src.fetch import cap_demographics
    cap_demo = cap_demographics()
    if cap_demo:
        cap = df.cohort == "cap"
        df.loc[cap, "age"] = df.loc[cap, "record"].map(lambda r: cap_demo.get(r, {}).get("age"))
        df.loc[cap, "sex"] = df.loc[cap, "record"].map(lambda r: cap_demo.get(r, {}).get("sex"))
    base = df[(df.excluded.fillna("") == "") & (df.rem_minutes >= MIN_REM)]
    lines = []
    say = lambda s="": (print(s), lines.append(s))

    # T1 ─────────────────────────────────────────────────────────────────────
    nights = base.drop_duplicates(["cohort", "record"])
    say("T1  Nights scored (>= %.0f min REM)" % MIN_REM)
    t1 = nights.groupby("who").agg(nights=("record", "count"),
                                   rem_min_median=("rem_minutes", "median"),
                                   age_mean=("age", "mean"),
                                   male_pct=("sex", lambda s: 100 * (s == "M").mean()
                                             if s.notna().any() else np.nan),
                                   chin_fs=("chin_fs", "median"))
    say(t1.round(1).to_string())
    say()

    # T2 ─────────────────────────────────────────────────────────────────────
    say("T2  Share flagged POSITIVE by the published cut-off, % [95% CI]")
    rows = []
    for (band, meth, who), g in base[base.method.isin(DECIDING)].groupby(["band", "method", "who"]):
        v = g.positive.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        p, lo, hi = wilson(v.sum(), len(v))
        rows.append(dict(band=band, method=meth, who=who, n=len(v),
                         cell=f"{100 * p:.0f} [{100 * lo:.0f}-{100 * hi:.0f}]"))
    t2 = pd.DataFrame(rows).pivot_table(index=["band", "method"], columns="who",
                                        values="cell", aggfunc="first")
    say(t2.to_string())
    say()

    # T3 ─────────────────────────────────────────────────────────────────────
    say("T3  Agreement between methods on the same nights (Cohen's kappa, native band)")
    nat = base[(base.band == "native") & base.method.isin(DECIDING)]
    wide = nat.pivot_table(index=["who", "record"], columns="method", values="positive",
                           aggfunc="first").dropna()
    wide = wide.map(lambda x: float(x in (True, "True", 1, 1.0)))
    for who, g in wide.groupby(level="who"):
        pairs = [f"{a}-{b} {kappa(g[a], g[b]):.2f}" for i, a in enumerate(DECIDING)
                 for b in DECIDING[i + 1:] if a in g and b in g]
        disagree = (g.nunique(axis=1) > 1).mean() * 100
        say(f"  {who:<16} n={len(g):<5} {'  '.join(pairs)}   methods disagree on {disagree:.0f}% of nights")
    say()

    # T4 ─────────────────────────────────────────────────────────────────────
    mgh = base[(base.cohort == "chal18") & (base.band == "native") & base.method.isin(DECIDING)]
    try:
        import statsmodels.formula.api as smf
    except ImportError:
        smf = None
        say("T4  skipped: statsmodels is not installed\n")
    if smf is not None and len(mgh) and mgh.age.notna().any():
        say("T4  What predicts a positive call in the MGH cohort (odds ratios [95% CI])")
        for meth, g in mgh.groupby("method"):
            g = g.dropna(subset=["positive", "age", "sex", "ahi"]).copy()
            g["y"] = g.positive.map(lambda x: float(x in (True, "True", 1, 1.0)))
            g["age10"] = g.age / 10.0
            g["male"] = (g.sex == "M").astype(float)
            g["ahi10"] = g.ahi / 10.0
            if g.y.nunique() < 2:
                say(f"  {meth}: all nights {'positive' if g.y.iloc[0] else 'negative'}")
                continue
            m = smf.logit("y ~ age10 + male + ahi10", data=g).fit(disp=False)
            ci = np.exp(m.conf_int())
            cells = [f"{k} {np.exp(m.params[k]):.2f} [{ci.loc[k, 0]:.2f}-{ci.loc[k, 1]:.2f}]"
                     for k in ("age10", "male", "ahi10")]
            say(f"  {meth:<9} n={len(g):<5} per 10 y: {cells[0].split(' ', 1)[1]}   "
                f"male: {cells[1].split(' ', 1)[1]}   per 10 AHI: {cells[2].split(' ', 1)[1]}")
        say()

    # T5 ─────────────────────────────────────────────────────────────────────
    ev = df[(df.cohort == "chal18") & (df.band == "native") & df.method.isin(DECIDING)
            & (df.rem_minutes >= MIN_REM) & df.excluded.fillna("").isin(["", "resp+arousal"])]
    if len(ev) and (ev.excluded.fillna("") != "").any():
        say("T5  Excluding REM epochs with respiratory events or arousals (MGH), % positive")
        w = ev.pivot_table(index=["record", "method"], columns=ev.excluded.fillna(""),
                           values="positive", aggfunc="first").dropna()
        w = w.map(lambda x: float(x in (True, "True", 1, 1.0)))
        for meth, g in w.groupby(level="method"):
            a, b = g.iloc[:, 0], g.iloc[:, 1]
            say(f"  {meth:<9} all REM {100 * a.mean():.0f}%  ->  event-free REM {100 * b.mean():.0f}%  "
                f"(changed on {100 * (a != b).mean():.0f}% of nights)")
        say()

        # T13 ────────────────────────────────────────────────────────────────
        say("T13 MGH: % positive by apnea-hypopnea index, all REM and event-free REM")
        ev = ev.assign(y=ev.positive.map(lambda x: float(x in (True, "True", 1, 1.0))
                                         if pd.notna(x) else np.nan),
                       ahi_cat=pd.cut(ev.ahi, [-1, 5, 15, 30, np.inf],
                                      labels=["AHI<5", "5-15", "15-30", ">=30"]),
                       rem=np.where(ev.excluded.fillna("") == "", "all REM", "event-free"))
        t = ev.pivot_table(index=["method", "rem"], columns="ahi_cat", values="y",
                           aggfunc="mean", observed=False) * 100
        say(t.round(0).to_string())
        n = ev[(ev.method == "Montreal") & (ev.rem == "all REM")].ahi_cat.value_counts()
        say("  nights: " + "  ".join(f"{k} {n.get(k, 0)}" for k in t.columns))
        say()

    # T6 ─────────────────────────────────────────────────────────────────────
    say("T6  Screening: positive predictive value at a given RBD prevalence")
    say("    sensitivity from CAP RBD; specificity from each external cohort and pooled")
    for meth in DECIDING:
        g = base[(base.band == "native") & (base.method == meth)]
        y = lambda s: s.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        sens = y(g[g.who == "CAP RBD"].positive)
        for name, cohorts in (("pooled", ["chal18", "hmc"]), ("MGH", ["chal18"]), ("HMC", ["hmc"])):
            spec = 1 - y(g[g.cohort.isin(cohorts)].positive)
            if len(sens) == 0 or len(spec) == 0:
                continue
            se, sp = sens.mean(), spec.mean()
            ppv = [se * p / (se * p + (1 - sp) * (1 - p)) for p in (0.01, 0.02, 0.05, 0.10)]
            say(f"  {meth:<9} {name:<6} sens {100 * se:.0f}%  spec {100 * sp:.0f}%   PPV at 1/2/5/10% "
                "prevalence: " + " / ".join(f"{100 * v:.0f}%" for v in ppv))

    say()

    # T7 ─────────────────────────────────────────────────────────────────────
    say("T7  Cut-offs re-derived for automated scoring (Youden, CAP RBD vs same-system")
    say("    patients) and the share of external nights they flag, native band")
    for (meth, metric), g in base[base.band == "native"].melt(
            id_vars=["cohort", "record", "who", "method", "chin_fs", "unit_known"],
            value_vars=[m for m in METRICS if m in base], var_name="metric") \
            .dropna(subset=["value"]).groupby(["method", "metric"]):
        s = (g.cohort == "cap") & (g.chin_fs == 512) & g.unit_known.astype(bool)
        pos = g.loc[s & (g.who == "CAP RBD"), "value"].to_numpy()
        neg = g.loc[s & ~g.who.isin(["CAP RBD", "CAP healthy"]), "value"].to_numpy()
        if len(pos) < 5 or len(neg) < 5:
            continue
        cands = np.unique(np.r_[pos, neg])
        j = [(pos >= c).mean() - (neg >= c).mean() for c in cands]
        c = cands[int(np.argmax(j))]
        ext = g.loc[g.cohort.isin(["chal18", "hmc"])]
        fp = {k: 100 * (v.value >= c).mean() for k, v in ext.groupby("cohort")}
        say(f"  {meth:<9} {metric:<17} cut-off {c:7.2f}: sens {100 * (pos >= c).mean():3.0f}%  "
            f"same-system FP {100 * (neg >= c).mean():3.0f}%  "
            + "  ".join(f"{k} FP {v:3.0f}%" for k, v in sorted(fp.items())))
    say()

    # T9 ─────────────────────────────────────────────────────────────────────
    say("T9  Background chin EMG level (5th percentile of 1 s REM RMS, uV), median [IQR]")
    bg = base[(base.band == "native") & (base.method == "Montreal")]
    if "background_phasic_uV" in bg:
        for who, g in bg.groupby("who"):
            v = g.background_phasic_uV.dropna()
            if len(v):
                say(f"  {who:<16} n={len(v):<5} {v.median():6.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}]")
    say()

    # T10 ────────────────────────────────────────────────────────────────────
    say("T10 Re-derived cut-offs, validated: leave-one-out CV on CAP, and 2,000 bootstrap")
    say("    re-derivations for the uncertainty of the cut-off and of external false positives")
    rng = np.random.default_rng(0)

    def youden(p, n):
        c = np.unique(np.r_[p, n])
        j = (p[None, :] >= c[:, None]).mean(axis=1) - (n[None, :] >= c[:, None]).mean(axis=1)
        return c[int(np.argmax(j))]

    melted = base[base.band == "native"].melt(
        id_vars=["cohort", "record", "who", "method", "chin_fs", "unit_known"],
        value_vars=[m for m in METRICS if m in base], var_name="metric").dropna(subset=["value"])
    for (meth, metric), g in melted.groupby(["method", "metric"]):
        s = (g.cohort == "cap") & (g.chin_fs == 512) & g.unit_known.astype(bool)
        pos = g.loc[s & (g.who == "CAP RBD"), "value"].to_numpy(float)
        neg = g.loc[s & ~g.who.isin(["CAP RBD", "CAP healthy"]), "value"].to_numpy(float)
        if len(pos) < 5 or len(neg) < 5:
            continue
        cv_tp = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
        cv_fp = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
        ext = {k: v.value.to_numpy(float) for k, v in g[g.cohort.isin(["chal18", "hmc"])].groupby("cohort")}
        cuts, fps = [], {k: [] for k in ext}
        for _ in range(2000):
            c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
            cuts.append(c)
            for k, v in ext.items():
                fps[k].append((v >= c).mean())
        q = lambda a: f"{100 * np.median(a):.0f} [{100 * np.percentile(a, 2.5):.0f}-{100 * np.percentile(a, 97.5):.0f}]"
        say(f"  {meth:<9} {metric:<17} cut-off {np.median(cuts):6.2f} [{np.percentile(cuts, 2.5):.2f}-"
            f"{np.percentile(cuts, 97.5):.2f}]  CV sens {100 * cv_tp:3.0f}%  CV FP {100 * cv_fp:3.0f}%  "
            + "  ".join(f"{k} FP {q(v)}" for k, v in sorted(fps.items())))
    say()

    # T11 ────────────────────────────────────────────────────────────────────
    say("T11 Positive calls by quiet-REM level (tertiles of the background within each")
    say("    cohort, native band) and, in MGH, the odds ratio per halving of that level")
    quiet = base[(base.band == "native") & (base.method == "Montreal")][
        ["cohort", "record", "background_phasic_uV"]].rename(columns={"background_phasic_uV": "quiet"})
    nat = base[(base.band == "native") & base.method.isin(DECIDING)].merge(
        quiet, on=["cohort", "record"], how="left").dropna(subset=["quiet", "positive"])
    nat["y"] = nat.positive.map(lambda x: float(x in (True, "True", 1, 1.0)))
    for (who, meth), g in nat[nat.cohort.isin(["chal18", "hmc"])].groupby(["who", "method"]):
        t = pd.qcut(g["quiet"], 3, labels=False)
        say(f"  {who:<8} {meth:<9} " + "   ".join(
            f"{g['quiet'][t == k].median():.2f} uV: {100 * g.y[t == k].mean():3.0f}%" for k in range(3)))
    if smf is not None:
        for meth, g in nat[nat.cohort == "chal18"].groupby("method"):
            g = g.dropna(subset=["age", "sex", "ahi"]).copy()
            if g.y.nunique() < 2:
                continue
            g["halvings"] = -np.log2(g["quiet"])
            g["age10"], g["male"], g["ahi10"] = g.age / 10.0, (g.sex == "M").astype(float), g.ahi / 10.0
            m = smf.logit("y ~ halvings + age10 + male + ahi10", data=g).fit(disp=False)
            lo, hi = np.exp(m.conf_int().loc["halvings"])
            say(f"  MGH {meth:<9} OR per halving {np.exp(m.params['halvings']):.2f} [{lo:.2f}-{hi:.2f}]"
                f" adjusted for age, sex and AHI (n={len(g)})")
    say()

    # T12 ────────────────────────────────────────────────────────────────────
    say("T12 Simulated recordings against their source on the same nights (native band)")
    for sim, src in SIMULATED.items():
        pair = nat[nat.cohort == sim].merge(nat[nat.cohort == src], on=["record", "method"],
                                            suffixes=("", "_src"))
        if pair.empty:
            continue
        lvl = pair[pair.method == "Montreal"]
        say(f"  {sim:<12} vs {src:<8} n={len(lvl):<4} quiet-REM level {lvl.quiet_src.median():.2f}"
            f" -> {lvl.quiet.median():.2f} uV")
        for meth, g in pair.groupby("method"):
            say(f"      {meth:<9} {100 * g.y_src.mean():3.0f}% -> {100 * g.y.mean():3.0f}%   "
                + paired_change(g.y_src.to_numpy(), g.y.to_numpy()))
    say("    (paired difference with 95% CI; exact McNemar test on the nights that changed)")
    say()

    # T15 ────────────────────────────────────────────────────────────────────
    bd = df[(df.band == "native") & df.method.isin(DECIDING) & (df.rem_minutes >= MIN_REM)
            & df.excluded.fillna("").isin(["", "boundary"])]
    if (bd.excluded.fillna("") == "boundary").any():
        say("T15 Without the first and last epoch of every REM period (transitional epochs),")
        say("    % positive: all REM -> without them")
        bd = bd.assign(y=bd.positive.map(lambda x: float(x in (True, "True", 1, 1.0))
                                         if pd.notna(x) else np.nan))
        w = bd.pivot_table(index=["who", "record", "method"], columns=bd.excluded.fillna(""),
                           values="y", aggfunc="first").dropna()
        gap = {}
        for (who, meth), g in w.groupby(level=["who", "method"]):
            if who in SIMULATED or len(g) < 10:
                continue
            a, b = g[""].mean(), g["boundary"].mean()
            gap[who, meth] = (a, b)
            say(f"  {who:<16} {meth:<9} {100 * a:3.0f}% -> {100 * b:3.0f}%   (n={len(g)})")
        for meth in DECIDING:
            if ("chal18", meth) in gap and ("hmc", meth) in gap:
                (a1, b1), (a2, b2) = gap["chal18", meth], gap["hmc", meth]
                say(f"  MGH-HMC gap, {meth}: {100 * (a1 - a2):+.0f} -> {100 * (b1 - b2):+.0f} points")
        say()

    # T16 ────────────────────────────────────────────────────────────────────
    say("T16 Floor signature: quietest N2/N3 chin level over quietest REM level (1 = the two")
    say("    are indistinguishable, i.e. the floor is the recording's own noise), median [IQR]")
    aasm = base[(base.band == "native") & (base.method == "AASM")]
    for who, g in aasm.groupby("who"):
        r = (g.background_tonic_uV / g.background_phasic_uV).dropna()
        if len(r) >= 5:
            say(f"  {who:<16} n={len(r):<5} {r.median():.2f} [{r.quantile(.25):.2f}-{r.quantile(.75):.2f}]")
    say()

    # T17 ────────────────────────────────────────────────────────────────────
    if cap_demo:
        from run_sensitivity import auc
        yes = lambda s: s.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        say("T17 Age in CAP: RBD patients against same-system patients under 50 and aged 50 or")
        say("    over (native band, published cut-offs; RBD and patients on the main system)")
        cap = base[(base.cohort == "cap") & (base.band == "native")]
        same = (cap.chin_fs == 512) & cap.unit_known.astype(bool)
        rbd = cap[same & (cap.who == "CAP RBD")]
        pat = cap[same & ~cap.who.isin(["CAP RBD", "CAP healthy"])]
        parts = {"patients < 50": pat[pat.age < 50], "patients >= 50": pat[pat.age >= 50], "RBD": rbd}
        for name, g in parts.items():
            a = g.drop_duplicates("record").age
            say(f"  {name:<15} n={len(a):<3} age {a.mean():.0f} +/- {a.std():.0f} ({a.min():.0f}-{a.max():.0f}), "
                f"diagnoses: " + ", ".join(f"{k.replace('CAP ', '')} {v}" for k, v in
                                          g.drop_duplicates('record').who.value_counts().items()))
        for meth in DECIDING:
            say(f"  {meth:<9} flagged: " + "   ".join(
                f"{name} {100 * yes(g[g.method == meth].positive).mean():3.0f}%" for name, g in parts.items()))
        old = parts["patients >= 50"]
        for meth, metric in [("Montreal", "tonic_pct"), ("Montreal", "phasic_pct"),
                             ("SINBAR", "any_pct"), ("RAI", "atonia_loss")]:
            r_ = rbd[rbd.method == meth][metric]
            say(f"  AUC {meth:<9} {metric:<12} vs all patients {auc(r_, pat[pat.method == meth][metric]):.3f}"
                f"   vs patients >= 50 {auc(r_, old[old.method == meth][metric]):.3f}")
        say()

    # T8 ─────────────────────────────────────────────────────────────────────
    say("T8  Heartbeat artifact in chin EMG (heartbeat-locked ratio; 1 = none), median [p90]")
    import json
    residual = {}                                    # (cohort, record) -> ratio after removal
    for cdir in sorted((config.DURABLE / "packs").glob("*")):
        before, after = [], []
        for p in cdir.glob("*.npz"):
            try:
                meta = json.loads(str(np.load(p)["meta"]))
            except Exception:
                continue
            e = meta.get("ecg") or meta.get("chin_info", {}).get("ecg") or {}
            if e.get("beats"):
                before.append(e.get("artifact_ratio_before", np.nan))
                after.append(e.get("artifact_ratio_after", np.nan))
                residual[(cdir.name, meta.get("record", p.stem))] = after[-1]
        if before:
            b, a = np.array(before, float), np.array(after, float)
            say(f"  {cdir.name:<7} n={len(b):<5} before {np.nanmedian(b):.2f} [{np.nanpercentile(b, 90):.2f}]"
                f"   after {np.nanmedian(a):.2f} [{np.nanpercentile(a, 90):.2f}]")
    say()

    # T14 ────────────────────────────────────────────────────────────────────
    art = nat[nat.cohort == "chal18"].copy()
    art["art"] = [residual.get(("chal18", r), np.nan) for r in art.record]
    art = art.dropna(subset=["art"])
    if len(art):
        say("T14 MGH: % positive by residual heartbeat artifact (tertiles of the ratio after removal)")
        art["t"] = pd.qcut(art.art, 3, labels=False)
        for meth, g in art.groupby("method"):
            say(f"  {meth:<9} " + "   ".join(f"ratio {g.art[g.t == k].median():.2f}: "
                                            f"{100 * g.y[g.t == k].mean():3.0f}%" for k in range(3)))

    out = config.RESULTS / "report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_figures.py
"""The paper's figures, drawn from the tables that run_score.py and
run_sensitivity.py write.

    python run_figures.py        # -> figures/fig1..fig5 (.png and .pdf)

  fig1  share of nights each method flags, per group, with 95% intervals
  fig2  quiet-REM chin EMG level per cohort (A) and the hardware simulation (B)
  fig3  how the definition of "background" moves the verdicts and the separation
  fig4  where the published and the re-derived cut-offs fall on each group
  fig5  positive calls by quiet-REM level within the MGH and HMC cohorts
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from run_report import MIN_REM, WITHDRAWN, wilson
from run_sensitivity import auc

YES = (True, "True", 1, 1.0)
GROUPS = ["CAP RBD", "CAP patients", "CAP healthy", "chal18", "hmc"]
LABEL = {"CAP RBD": "CAP\nRBD", "CAP patients": "CAP\npatients", "CAP healthy": "CAP\nhealthy",
         "chal18": "MGH", "hmc": "HMC", "hmc_q1uv": "HMC\n1 µV", "hmc_mgh": "HMC\n1 µV\n200 Hz",
         "hmc_low": "HMC\n×0.39", "hmc_low_mgh": "HMC\n×0.39\n1 µV\n200 Hz",
         "chal18_noise": "MGH\n+1.5 µV\nnoise"}
COLOR = {"Montreal": "#1b9e77", "SINBAR": "#d95f02", "RAI": "#7570b3", "AASM": "#e7298a"}
GROUP_COLOR = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "CAP healthy": "#2ca02c",
               "chal18": "#1f77b4", "hmc": "#9467bd"}
VARIANTS = ["min_rem", "p5_rem", "p10_rem", "p25_rem", "p50_rem", "p5_nrem"]
VARIANT_LABEL = ["min", "p5\n(default)", "p10", "p25", "p50", "p5\nNREM"]

plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.fontsize": 7, "savefig.dpi": 300})


def label_groups(df):
    """CAP RBD / CAP healthy / CAP patients on CAP's main system (512 Hz), and cohorts."""
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    who = np.select([cap & (df.group == "RBD"), cap & (df.group == "healthy"),
                     same & ~df.group.isin(["RBD", "healthy"])],
                    ["CAP RBD", "CAP healthy", "CAP patients"], default="")
    return df.assign(who=np.where(cap, who, df.cohort), same=same)


def load_scores():
    df = pd.read_csv(config.RESULTS / "rwa_scores.csv")
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df = df[(df.excluded.fillna("") == "") & (df.rem_minutes >= MIN_REM) & (df.band == "native")]
    df = label_groups(df)
    return df.assign(y=df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan))


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGURES / f"{name}.{ext}")
    plt.close(fig)
    print(f"  {name}")


def dots_with_ci(ax, df, groups, methods=("Montreal", "SINBAR", "RAI"), show_n=True):
    width = 0.8 / len(methods)
    for j, meth in enumerate(methods):
        xs, ps, err = [], [], [[], []]
        for i, g in enumerate(groups):
            v = df[(df.who == g) & (df.method == meth)].y.dropna()
            p, lo, hi = wilson(v.sum(), len(v))
            xs.append(i + (j - (len(methods) - 1) / 2) * width)
            ps.append(100 * p)
            err[0].append(100 * (p - lo))
            err[1].append(100 * (hi - p))
        ax.errorbar(xs, ps, yerr=err, fmt="o", ms=4, capsize=2, lw=1, color=COLOR[meth], label=meth)
    n = [df[(df.who == g) & (df.method == methods[0])].y.notna().sum() for g in groups]
    ax.set_xticks(range(len(groups)),
                  [f"{LABEL[g]}\nn={k}" if show_n else LABEL[g] for g, k in zip(groups, n)])
    ax.set_ylim(0, 100)
    ax.set_ylabel("nights flagged positive (%)")


def fig1(df):
    fig, ax = plt.subplots(figsize=(6.5, 3.0), layout="constrained")
    dots_with_ci(ax, df, GROUPS)
    ax.axvline(0.5, color="0.85", lw=0.8, zorder=0)
    ax.legend(frameon=False, ncol=3, loc="upper right")
    ax.set_title("Published cut-offs: sensitivity in RBD, positives everywhere else")
    save(fig, "fig1_positivity")


def fig2(df):
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 3.2), layout="constrained",
                               gridspec_kw={"width_ratios": [1, 1.3]})
    lvl = df[df.method == "Montreal"]
    data = [lvl.loc[lvl.who == g, "background_phasic_uV"].dropna() for g in GROUPS]
    rng = np.random.default_rng(0)
    for i, (g, v) in enumerate(zip(GROUPS, data)):
        a.scatter(i + rng.uniform(-0.25, 0.25, len(v)), v, s=3, alpha=0.35 if len(v) > 200 else 0.7,
                  color=GROUP_COLOR[g], lw=0)
    a.boxplot(data, positions=range(len(GROUPS)), widths=0.55, showfliers=False,
              medianprops={"color": "k"}, boxprops={"lw": 0.8}, whiskerprops={"lw": 0.8})
    a.axhline(1.0, ls="--", color="k", lw=0.8, label="MGH storage step (1 µV)")
    a.legend(frameon=False, loc="upper left")
    a.set_yscale("log")
    ticks = [0.2, 0.5, 1, 2, 5, 10, 20, 50]
    a.set_yticks(ticks, [f"{t:g}" for t in ticks])
    a.set_ylim(0.15, 1.3 * max(v.max() for v in data if len(v)))
    a.set_xticks(range(len(GROUPS)), [LABEL[g] for g in GROUPS])
    a.set_ylabel("quiet-REM chin EMG level (µV)")
    a.set_title("A  Quiet-REM level")
    sims = [c for c in ["hmc", "hmc_q1uv", "hmc_mgh", "hmc_low", "hmc_low_mgh", "chal18",
                        "chal18_noise"] if (df.who == c).any()]
    dots_with_ci(b, df, sims, show_n=False)
    b.tick_params(axis="x", labelsize=7)
    b.legend(frameon=False, ncol=3, loc="upper left")
    b.set_title("B  Recording simulations")
    save(fig, "fig2_level_and_hardware")


def fig3(df):
    sens = pd.read_csv(config.RESULTS / "sensitivity_background.csv")
    sens = label_groups(sens)
    sens = sens.assign(y=sens.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan))
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.1), layout="constrained")
    x = np.arange(len(VARIANTS))
    for k, (ax, meth) in enumerate(zip(axes[:2], ["Montreal", "SINBAR"])):
        for g in GROUPS:
            s = sens[(sens.method == meth) & (sens.who == g)].groupby("variant").y.mean() * 100
            ax.plot(x, [s.get(v, np.nan) for v in VARIANTS], "o-", ms=3, lw=1,
                    color=GROUP_COLOR[g], label=LABEL[g].replace("\n", " "))
        ax.set_ylim(0, 100)
        ax.set_ylabel("nights flagged positive (%)")
        ax.set_title(f"{'AB'[k]}  {meth}")
    same = sens[sens.who.isin(["CAP RBD", "CAP patients"])]
    for meth, metric, ls in [("Montreal", "phasic_pct", "-"), ("Montreal", "tonic_pct", "--"),
                             ("SINBAR", "phasic_pct", "-"), ("SINBAR", "tonic_pct", "--"),
                             ("AASM", "phasic_pct", "-")]:
        g = same[same.method == meth]
        vals = [auc(g.loc[(g.variant == v) & (g.who == "CAP RBD"), metric],
                    g.loc[(g.variant == v) & (g.who == "CAP patients"), metric]) for v in VARIANTS]
        axes[2].plot(x, vals, "o" + ls, ms=3, lw=1, color=COLOR[meth],
                     label=f"{meth} {metric.split('_')[0]}")
    axes[2].axhline(0.5, color="0.7", lw=0.8)
    axes[2].set_ylim(0.4, 1.0)
    axes[2].set_ylabel("AUC, RBD vs other CAP patients")
    axes[2].legend(frameon=False, loc="lower left")
    axes[2].set_title("C  Separation")
    for ax in axes:
        ax.set_xticks(x, VARIANT_LABEL, fontsize=7)
        ax.set_xlabel("background definition")
    fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=5,
               frameon=False)
    save(fig, "fig3_background_definition")


def youden(pos, neg):
    c = np.unique(np.r_[pos, neg])
    j = (pos[None, :] >= c[:, None]).mean(axis=1) - (neg[None, :] >= c[:, None]).mean(axis=1)
    return c[int(np.argmax(j))]


def fig4(df):
    panels = [("Montreal", "phasic_pct", 15.0, "Montreal phasic (% of 2 s mini-epochs)"),
              ("SINBAR", "any_pct", 18.2, "SINBAR any (% of 3 s mini-epochs)")]
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2), layout="constrained")
    rng = np.random.default_rng(0)
    for k, (ax, (meth, metric, published, title)) in enumerate(zip(axes, panels)):
        d = df[df.method == meth]
        # derived as in run_report.py T7: RBD and other patients on the same system
        pos = d.loc[(d.who == "CAP RBD") & d.same, metric].dropna().to_numpy(float)
        neg = d.loc[d.who == "CAP patients", metric].dropna().to_numpy(float)
        cut = youden(pos, neg)
        ticks = []
        for i, g in enumerate(GROUPS):
            v = d.loc[d.who == g, metric].dropna().to_numpy(float)
            ax.scatter(i + rng.uniform(-0.25, 0.25, len(v)), v, s=3, lw=0, color=GROUP_COLOR[g],
                       alpha=0.35 if len(v) > 200 else 0.8)
            ticks.append(f"{LABEL[g]}\n{100 * (v >= published).mean():.0f}→"
                         f"{100 * (v >= cut).mean():.0f}%")
        ax.axhline(published, color="k", ls="--", lw=0.8, label=f"published cut-off ({published:g}%)")
        ax.axhline(cut, color="k", lw=0.8, label=f"re-derived cut-off ({cut:.1f}%)")
        ax.set_xticks(range(len(GROUPS)), ticks, fontsize=7)
        ax.set_xlabel("flagged positive: published → re-derived cut-off")
        ax.set_ylim(0, 100)
        ax.set_ylabel(title)
        ax.legend(frameon=False, loc="upper right", fontsize=6.5)
        ax.set_title(f"{'AB'[k]}  {meth}")
    save(fig, "fig4_cutoffs")


def fig5(df):
    quiet = df[df.method == "Montreal"][["cohort", "record", "background_phasic_uV"]].rename(
        columns={"background_phasic_uV": "quiet"})
    d = df[df.method.isin(["Montreal", "SINBAR", "RAI"])].merge(quiet, on=["cohort", "record"])
    d = d.dropna(subset=["quiet", "y"])
    fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), sharey=True, layout="constrained")
    for ax, cohort, name in zip(axes, ["chal18", "hmc"], ["A  MGH", "B  HMC"]):
        g = d[d.cohort == cohort]
        edges = g[g.method == "Montreal"].quiet.quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
        t = np.clip(np.searchsorted(edges, g.quiet, side="right") - 1, 0, 2)
        for j, meth in enumerate(["Montreal", "SINBAR", "RAI"]):
            m = g.method.to_numpy() == meth
            ys = [100 * g.y.to_numpy()[m & (t == k)].mean() for k in range(3)]
            ax.bar(np.arange(3) + (j - 1) * 0.27, ys, width=0.26, color=COLOR[meth], label=meth)
        ax.set_xticks(range(3), [f"lowest third\n< {edges[1]:.2f} µV", "middle third",
                                 f"highest third\n> {edges[2]:.2f} µV"])
        ax.set_xlabel("night's quiet-REM chin EMG level")
        ax.set_title(name)
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[1].legend(frameon=False, loc="upper left")
    save(fig, "fig5_level_vs_verdict")


def main():
    config.FIGURES.mkdir(parents=True, exist_ok=True)
    df = load_scores()
    print(f"figures -> {config.FIGURES}")
    fig1(df)
    fig2(df)
    if (config.RESULTS / "sensitivity_background.csv").exists():
        fig3(df)
    fig4(df)
    fig5(df)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_ladder.py
"""The noise ladder: how each method's verdict moves with the noise floor.

Every ladder pack (run_cohort.py --ladder) holds one night's chin EMG as
recorded and with white noise added in band: at fixed levels (n050 ... n300 =
0.5 ... 3 uV RMS) and at the level that brings its quiet-REM level to a common
target (h150, h200 = 1.5 and 2.0 uV). This scores every copy with every method
and reports, against the criteria fixed in paper/preregistration_noise_ladder.md:

  L1  dose-response: quiet-REM level and share flagged, per group and step
  L2  matched floor: MGH against HMC after harmonizing both (criteria 1, 2)
  L3  RBD sensitivity as the floor rises (criterion 3)
  L4  the remedy: cut-offs re-derived on harmonized CAP, leave-one-out and
      bootstrap, applied to harmonized MGH and HMC (criterion 4)
  L5  the recording systems' own noise, measured on EEG (criterion 5)

    python run_ladder.py   -> results/ladder_scores.csv, results/ladder_report.md,
                              figures/fig6_noise_ladder, figures/fig7_harmonisation
    python run_ladder.py --from-csv   the same from results/ladder_scores.csv
"""
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import score_all
from run_report import MIN_REM, WITHDRAWN, wilson
from run_score import group_of

LADDER = ["native", "n050", "n100", "n150", "n200", "n300"]
STEPS = LADDER + ["h150", "h200"]
TARGET = "h200"
DECIDING = ["Montreal", "SINBAR", "RAI"]
WHO = ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"]
REMEDY = [("Montreal", "phasic_pct"), ("Montreal", "tonic_pct"), ("SINBAR", "any_pct"),
          ("SINBAR", "phasic_pct"), ("SINBAR", "tonic_pct"), ("AASM", "phasic_epoch_pct"),
          ("RAI", "atonia_loss")]
KEY = [("Montreal", "phasic_pct"), ("Montreal", "tonic_pct"), ("SINBAR", "any_pct")]
YES = (True, "True", 1, 1.0)


def score_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    eeg = m.get("eeg") or {}
    info = dict(cohort=m["cohort"], record=m["record"], group=group_of(m["cohort"], m["record"]),
                chin_fs=m.get("chin_fs"), unit_known=m.get("unit_known", True),
                eeg_nrem=eeg.get("floor_nrem_uv"), eeg_rem=eeg.get("floor_rem_uv"),
                eeg_channel=eeg.get("channel"))
    rows = []
    for step in STEPS:
        if step != "native" and f"rms_{step}" not in m["rows"]:
            continue
        for r in score_all(pack, step):
            r.pop("band", None)
            rows.append({**info, "step": step, **r})
    return rows


def load_scores():
    files = sorted(p for p in (config.DURABLE / "ladder").glob("*/*.npz") if ".tmp" not in p.name)
    print(f"scoring {len(files)} ladder packs", flush=True)
    with ProcessPoolExecutor(max_workers=2) as ex:
        rows = [r for part in ex.map(score_file, files, chunksize=4) for r in part]
    df = pd.DataFrame(rows)
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df["atonia_loss"] = 1 - df["rai"]
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    # a night's quiet-REM level belongs to the night and step, not to the method:
    # take it from the Montreal row and attach it to every method's row
    level = df[df.method == "Montreal"].set_index(["cohort", "record", "step"]).background_phasic_uV
    df["level"] = level.reindex(pd.MultiIndex.from_frame(df[["cohort", "record", "step"]])).to_numpy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select(
        [cap & same & (df.group == "RBD"), cap & same & ~df.group.isin(["RBD", "healthy"]),
         cap & (df.group == "healthy"), df.cohort == "chal18", df.cohort == "hmc"],
        ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"], default="")
    return df


def youden(p, n):
    c = np.unique(np.r_[p, n])
    j = (p[None, :] >= c[:, None]).mean(axis=1) - (n[None, :] >= c[:, None]).mean(axis=1)
    return c[int(np.argmax(j))]


def pct(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    p, lo, hi = wilson(v.sum(), len(v))
    return p, lo, hi, len(v)


def main():
    if "--from-csv" in sys.argv:
        # report and figures again from the saved scores (no ladder packs needed)
        df = pd.read_csv(config.RESULTS / "ladder_scores.csv")
    else:
        df = load_scores()
        df.to_csv(config.RESULTS / "ladder_scores.csv", index=False)
    base = df[df.rem_minutes >= MIN_REM]
    lines = []
    say = lambda s="": (print(s), lines.append(s))

    # L1 ─────────────────────────────────────────────────────────────────────
    say("L1  Dose-response: median quiet-REM level (uV) and % flagged by the published cut-offs")
    for who in WHO:
        g = base[base.who == who]
        if g.empty:
            continue
        say(f"  {who} (n={g[g.step == 'native'].record.nunique()})")
        for step in STEPS:
            s = g[g.step == step]
            if s.empty:
                continue
            cells = "  ".join(f"{m} {100 * s[s.method == m].y.mean():5.1f}%" for m in DECIDING)
            say(f"    {step:<7} level {s[s.method == 'Montreal'].level.median():5.2f}   {cells}")
    say()

    # L2 ─────────────────────────────────────────────────────────────────────
    say("L2  Matched floor: MGH against HMC (criterion 1: <= 3 points Montreal, <= 5 SINBAR)")
    for step in ["native", "h150", "h200"]:
        cells = []
        for m in DECIDING:
            a = pct(base[(base.who == "MGH") & (base.step == step) & (base.method == m)].y)
            b = pct(base[(base.who == "HMC") & (base.step == step) & (base.method == m)].y)
            cells.append(f"{m} MGH {100 * a[0]:.0f} [{100 * a[1]:.0f}-{100 * a[2]:.0f}] vs HMC "
                         f"{100 * b[0]:.0f} [{100 * b[1]:.0f}-{100 * b[2]:.0f}] (diff {100 * (a[0] - b[0]):+.1f})")
        say(f"  {step:<7} " + ";  ".join(cells))
    say("  criterion 2: MGH's ladder interpolated at HMC's native median level")
    hmc_level = base[(base.who == "HMC") & (base.step == "native")].level.median()
    for m in DECIDING:
        g = base[(base.who == "MGH") & (base.method == m) & base.step.isin(LADDER)]
        x = np.log([g[g.step == s].level.median() for s in LADDER])
        y = [100 * g[g.step == s].y.mean() for s in LADDER]
        at = np.interp(np.log(hmc_level), x, y)
        h = 100 * base[(base.who == "HMC") & (base.step == "native") & (base.method == m)].y.mean()
        say(f"    {m:<9} MGH at {hmc_level:.2f} uV: {at:.1f}%   HMC native: {h:.1f}%   (diff {at - h:+.1f})")
    say()

    # L3 ─────────────────────────────────────────────────────────────────────
    say("L3  Sensitivity for RBD (CAP, main system) as the floor rises (criterion 3: < 70% at h200)")
    for m in DECIDING:
        cells = []
        for step in STEPS:
            p, lo, hi, n = pct(base[(base.who == "CAP RBD") & (base.step == step) & (base.method == m)].y)
            if n:
                cells.append(f"{step} {100 * p:.0f}")
        say(f"  {m:<9} " + "  ".join(cells) + f"   (n={n})")
    say()

    # L4 ─────────────────────────────────────────────────────────────────────
    say("L4  Remedy: cut-offs re-derived on CAP, applied to MGH and HMC, with and without")
    say("    harmonizing every night to 2.0 uV (criterion 4 for Montreal phasic/tonic and")
    say("    SINBAR any: |FP MGH - FP HMC| <= 5 points and leave-one-out sensitivity >= 80%)")
    rng = np.random.default_rng(0)
    remedy = []
    for step in ["native", TARGET]:
        for meth, metric in REMEDY:
            g = base[(base.step == step) & (base.method == meth)]
            pos = g.loc[g.who == "CAP RBD", metric].dropna().to_numpy(float)
            neg = g.loc[g.who == "CAP patients", metric].dropna().to_numpy(float)
            if len(pos) < 5 or len(neg) < 5:
                continue
            cut = youden(pos, neg)
            cv_s = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
            cv_f = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
            ext = {k: g.loc[g.who == k, metric].dropna().to_numpy(float) for k in ("MGH", "HMC")}
            boot = {k: [] for k in ext}
            for _ in range(2000):
                c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
                for k, v in ext.items():
                    boot[k].append((v >= c).mean())
            fp = {k: (v >= cut).mean() for k, v in ext.items()}
            ci = {k: np.percentile(v, [2.5, 97.5]) for k, v in boot.items()}
            remedy.append(dict(step=step, method=meth, metric=metric, cut=cut, cv_sens=cv_s, cv_fp=cv_f,
                               fp_mgh=fp["MGH"], fp_hmc=fp["HMC"], ci_mgh=ci["MGH"], ci_hmc=ci["HMC"]))
            r = remedy[-1]
            ok = ""
            if (meth, metric) in KEY:
                ok = "  PASS" if abs(r["fp_mgh"] - r["fp_hmc"]) <= 0.05 and cv_s >= 0.80 else "  fail"
            say(f"  {step:<7} {meth:<9} {metric:<17} cut {cut:7.2f}  CV sens {100 * cv_s:3.0f}%  "
                f"CV FP {100 * cv_f:3.0f}%  MGH FP {100 * fp['MGH']:3.0f} [{100 * ci['MGH'][0]:.0f}-"
                f"{100 * ci['MGH'][1]:.0f}]  HMC FP {100 * fp['HMC']:3.0f} [{100 * ci['HMC'][0]:.0f}-"
                f"{100 * ci['HMC'][1]:.0f}]  gap {100 * (fp['MGH'] - fp['HMC']):+.0f}{ok}")
    say()

    # L5 ─────────────────────────────────────────────────────────────────────
    say("L5  The systems' own noise: EEG 40-90 Hz floor (uV), median [IQR], and the chin's")
    say("    quiet-REM level (criterion 5: HMC/MGH EEG ratio >= 1.5 = system-wide)")
    nat = base[(base.step == "native") & (base.method == "Montreal")]
    by = {"CAP (main system)": nat[(nat.cohort == "cap") & (nat.chin_fs == 512)],
          "MGH": nat[nat.cohort == "chal18"], "HMC": nat[nat.cohort == "hmc"]}
    med = {}
    for name, g in by.items():
        e = g.eeg_nrem.dropna().astype(float)
        r = g.eeg_rem.dropna().astype(float)
        if len(e):
            med[name] = e.median()
            say(f"  {name:<18} n={len(e):<4} EEG N2/N3 {e.median():.2f} [{e.quantile(.25):.2f}-{e.quantile(.75):.2f}]"
                f"   EEG REM {r.median():.2f}   chin quiet-REM {g.level.median():.2f}")
    if "MGH" in med and "HMC" in med:
        say(f"  HMC/MGH EEG floor ratio {med['HMC'] / med['MGH']:.2f}")

    out = config.RESULTS / "ladder_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figures(base, remedy)


def figures(base, remedy):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import COLOR, save

    colors = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "MGH": "#1f77b4", "HMC": "#9467bd"}
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.9), sharey=True, layout="constrained")
    for ax, m in zip(axes, DECIDING):
        for who, col in colors.items():
            g = base[(base.who == who) & (base.method == m)]
            if g.empty:
                continue
            x = [g[g.step == s].level.median() for s in LADDER]
            y = [100 * g[g.step == s].y.mean() for s in LADDER]
            ax.plot(x, y, "o-", ms=3, lw=1, color=col, mfc="white", label=who)
            ax.plot(x[0], y[0], "o", ms=5, color=col)                  # as recorded
            h = g[g.step == TARGET]
            ax.plot(h.level.median(), 100 * h.y.mean(), "*", ms=8, color=col)
        ax.set_xscale("log")
        ax.set_xticks([0.5, 1, 2, 4], ["0.5", "1", "2", "4"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("quiet-REM level (µV)")
        ax.set_title(m)
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].set_ylim(0, 100)
    axes[0].legend(frameon=False, loc="upper right", fontsize=6.5)
    save(fig, "fig6_noise_ladder")

    rem = pd.DataFrame(remedy)
    if rem.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9), layout="constrained")
    # A: published cut-offs, as recorded vs harmonized
    ax = axes[0]
    groups = ["CAP RBD", "MGH", "HMC"]
    for j, (m, step, hatch) in enumerate([("Montreal", "native", ""), ("Montreal", TARGET, "//"),
                                          ("SINBAR", "native", ""), ("SINBAR", TARGET, "//")]):
        vals = [100 * base[(base.who == w) & (base.step == step) & (base.method == m)].y.mean()
                for w in groups]
        ax.bar(np.arange(3) + (j - 1.5) * 0.2, vals, width=0.19, color=COLOR[m], hatch=hatch,
               edgecolor="white", label=f"{m}, {'as recorded' if step == 'native' else 'harmonized 2 µV'}")
    ax.set_xticks(range(3), groups)
    ax.set_ylabel("nights flagged positive (%)")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False, fontsize=6, loc="upper right")
    ax.set_title("A  Published cut-offs")
    # B: re-derived cut-offs on harmonized data: external false positives
    ax = axes[1]
    k = rem[(rem.step == TARGET) & rem.apply(lambda r: (r.method, r.metric) in KEY, axis=1)]
    for i, r in enumerate(k.itertuples()):
        for dx, fp, ci, col in ((-0.12, r.fp_mgh, r.ci_mgh, "#1f77b4"), (0.12, r.fp_hmc, r.ci_hmc, "#9467bd")):
            ax.errorbar(i + dx, 100 * fp, yerr=[[100 * (fp - ci[0])], [100 * (ci[1] - fp)]],
                        fmt="o", ms=4, capsize=2, color=col)
    ax.plot([], [], "o", color="#1f77b4", label="MGH")
    ax.plot([], [], "o", color="#9467bd", label="HMC")
    ax.set_xticks(range(len(k)), [f"{r.method} {r.metric.split('_')[0]}\nLOO sens {100 * r.cv_sens:.0f}%"
                                  for r in k.itertuples()], fontsize=7)
    ax.set_ylabel("external nights flagged (%)")
    top = max(max(r.ci_mgh[1], r.ci_hmc[1]) for r in k.itertuples())
    ax.set_ylim(0, max(30, 100 * top + 5))
    ax.legend(frameon=False, fontsize=6.5)
    ax.set_title("B  Re-derived on harmonized CAP")
    save(fig, "fig7_harmonisation")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_floor.py
"""Floor-corrected RWA criteria: the published rules with the noise floor subtracted.

The relative criteria count activity above k x background, and background is
mostly the recording's own noise floor (run_ladder.py), so the muscle activity
a criterion requires scales with the noise of the recording system. The
floor-corrected (FC) rules fix that requirement in microvolts (src/rwa.py,
`threshold`): at a reference background B they are the published rules, below
it they become absolute, above it the noise is subtracted instead of
multiplied. A "clamp" variant (background never below B, no subtraction)
separates the two ingredients.

Scored on the ordinary packs (every night as recorded, and HMC scaled by 0.39
for gain) and on the noise-ladder packs (the same nights with noise added),
against the criteria fixed in paper/preregistration_floor_corrected.md:

  FC1  site comparison as recorded (criteria 3, 4, 5)
  FC2  noise invariance on the ladder (criteria 1, 2)
  FC3  the reference background B from 0.5 to 2.0 uV
  FC4  discrimination (AUC, RBD vs same-system CAP patients)
  FC5  cut-offs re-derived under FC (leave-one-out, bootstrap)
  FC6  gain: HMC scaled by 0.39, and the size of muscle activity per cohort
  FC7  Montreal phasic bursts up to 10 s instead of 15 s
  FC8  MGH: the floor by record batch, and the ladder's 300 against the rest
  FC9  age: RBD against same-system CAP patients under 50 and aged 50 or over
  FC10 CAP's healthy volunteers night by night (mixed recording systems)

CAP RBD and CAP patients are the nights recorded on CAP's main system (512 Hz,
microvolts): 21 RBD and 51 other patients.

    python run_floor.py -> results/floor_scores.csv, floor_ladder_scores.csv,
                           floor_report.md, figures/fig8_floor_corrected
"""
import dataclasses
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import (METHODS, NOISE_P5_RATIO, _select, floor_corrected, mini_epoch_rms,
                     rem_atonia_index, score_night)
from run_ladder import STEPS, YES, pct, youden
from run_report import MIN_REM, WITHDRAWN, paired_change
from run_score import auc, group_of

REFS = (0.5, 0.75, 1.0, 1.25, 1.5, 2.0)     # reference background B, uV
PRIMARY = 1.0
COHORTS = ("cap", "chal18", "hmc", "hmc_low")
PAIRED = "h150"                              # the ladder step the criteria use
WHO = ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"]
KEY = {"Montreal": ["tonic_pct", "phasic_pct"], "SINBAR": ["any_pct"]}


def variants():
    """(variant, reference uV, Method) for every scoring of Montreal and SINBAR."""
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out.append(("published", np.nan, m))
        out += [("FC", ref, floor_corrected(m, ref)) for ref in REFS]
        out.append(("clamp", PRIMARY, floor_corrected(m, PRIMARY, mode="clamp")))
    out.append(("max10", np.nan, dataclasses.replace(METHODS["montreal"], phasic_max_s=10.0)))
    return out


def scale(pack):
    """The size of the night's chin EMG in 1 s RMS (uV): quiet-REM level b, REM
    median and 95th percentile, N2/N3 median, and each with the noise implied by
    b (b / 0.91) removed in power ("_net")."""
    ep, env, _, nrem = _select(pack, "native")
    if env is None or len(ep) < 2:
        return {}
    r, _ = mini_epoch_rms(env.reshape(-1), pack["meta"]["env_fs"], 1.0)
    if len(r) < 10:
        return {}
    b = float(np.percentile(r, 5))
    out = {"rem_p50": float(np.median(r)), "rem_p95": float(np.percentile(r, 95))}
    if len(nrem) >= 60:
        out["nrem_p50"] = float(np.median(nrem))
        out["nrem_p5"] = float(np.percentile(nrem, 5))
    n2 = (b / NOISE_P5_RATIO) ** 2
    out.update({f"{k}_net": float(np.sqrt(max(v * v - n2, 0.0))) for k, v in list(out.items())})
    return dict(out, level_1s=b)


def score_rows(pack, band, info):
    rows = []
    for variant, ref, m in variants():
        rows.append({**info, **score_night(pack, m, band), "variant": variant, "ref_uv": ref})
    rows.append({**info, **rem_atonia_index(pack, band), "variant": "published", "ref_uv": np.nan})
    return rows


def _info(m, cohort):
    ecg = m.get("ecg") or m.get("chin_info", {}).get("ecg") or {}
    return dict(cohort=cohort, record=m["record"], group=group_of(cohort, m["record"]),
                chin_fs=m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                unit_known=m.get("unit_known", True), age=m.get("age"), pid=m.get("pid"),
                beats=int(ecg.get("beats") or 0))


def score_pack_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    info = _info(m, m.get("cohort", path.parent.name))
    rows = score_rows(pack, "native", info)
    rows[0].update(scale(pack))                  # one row per night carries the scale
    return rows


def score_ladder_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    info = _info(m, m["cohort"])
    rows = []
    for step in STEPS:
        if step == "native" or f"rms_{step}" in m["rows"]:
            rows += score_rows(pack, step, {**info, "step": step})
    return rows


def label(df):
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record).isin(gone)].copy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select(
        [same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"]),
         cap & (df.group == "healthy"), df.cohort == "chal18", df.cohort == "hmc",
         df.cohort == "hmc_low"],
        ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC", "HMC x0.39"], default="")
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    df["ref_key"] = df.ref_uv.fillna(0.0)
    return df


def score(files, fn):
    with ProcessPoolExecutor(max_workers=2) as ex:
        return pd.DataFrame([r for part in ex.map(fn, files, chunksize=4) for r in part])


def sel(df, method, variant="published", ref=PRIMARY):
    s = (df.method == method) & (df.variant == variant)
    if variant in ("FC", "clamp"):
        s &= np.isclose(df.ref_key, ref)
    return df[s]


def share(g):
    p, lo, hi, n = pct(g.y)
    return f"{100 * p:3.0f} [{100 * lo:.0f}-{100 * hi:.0f}]" if n else "   -"


def main():
    packs = [p for c in COHORTS for p in sorted((config.DURABLE / "packs" / c).glob("*.npz"))
             if ".tmp" not in p.name]
    ladder = sorted(p for p in (config.DURABLE / "ladder").glob("*/*.npz") if ".tmp" not in p.name)
    print(f"scoring {len(packs)} packs and {len(ladder)} ladder packs", flush=True)
    nat = label(score(packs, score_pack_file))
    nat.to_csv(config.RESULTS / "floor_scores.csv", index=False)
    lad = label(score(ladder, score_ladder_file))
    lad.to_csv(config.RESULTS / "floor_ladder_scores.csv", index=False)
    nat = nat[nat.rem_minutes >= MIN_REM]
    lad = lad[lad.rem_minutes >= MIN_REM]
    level = lad[(lad.method == "Montreal") & (lad.variant == "published")] \
        .set_index(["cohort", "record", "step"]).background_phasic_uV
    lad["level"] = level.reindex(pd.MultiIndex.from_frame(lad[["cohort", "record", "step"]])).to_numpy()

    lines = []
    say = lambda s="": (print(s), lines.append(s))
    verdict = {}

    # FC1 ────────────────────────────────────────────────────────────────────
    say(f"FC1 Share flagged positive as recorded (all nights, published % cut-offs), % [95% CI];")
    say(f"    FC and clamp with B = {PRIMARY} uV")
    say(f"    {'':<22}" + "".join(f"{w:<16}" for w in WHO) + "MGH-HMC")
    gaps = {}
    for meth in ("Montreal", "SINBAR", "RAI"):
        for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
            g = sel(nat, meth, variant)
            cells = [share(g[g.who == w]) for w in WHO]
            gap = 100 * (g[g.who == "MGH"].y.mean() - g[g.who == "HMC"].y.mean())
            gaps[meth, variant] = gap
            say(f"    {meth + ' ' + variant:<22}" + "".join(f"{c:<16}" for c in cells) + f"{gap:+.1f}")
    for meth, limit in (("Montreal", 5), ("SINBAR", 10)):
        sens = sel(nat, meth, "FC")
        sens = 100 * sens[sens.who == "CAP RBD"].y.mean()
        pub_fp = sel(nat, meth)
        fc_fp = sel(nat, meth, "FC")
        verdict[f"3 {meth}"] = abs(gaps[meth, "FC"]) <= limit
        verdict[f"4 {meth}"] = sens >= 85
        verdict[f"5 {meth}"] = fc_fp[fc_fp.who == "CAP patients"].y.mean() <= \
            pub_fp[pub_fp.who == "CAP patients"].y.mean()
        say(f"  criterion 3 ({meth}-FC gap <= {limit}): {gaps[meth, 'FC']:+.1f} -> "
            f"{'met' if verdict[f'3 {meth}'] else 'not met'};  criterion 4 (RBD sensitivity >= 85%): "
            f"{sens:.0f}% -> {'met' if verdict[f'4 {meth}'] else 'not met'};  criterion 5 (CAP patients "
            f"not above published): {'met' if verdict[f'5 {meth}'] else 'not met'}")
    say()

    # FC2 ────────────────────────────────────────────────────────────────────
    say("FC2 The same nights as the noise floor rises: % positive per ladder step, published")
    say(f"    cut-offs; FC and clamp with B = {PRIMARY} uV; median quiet-REM level in the first line")
    steps = [s for s in STEPS if s in set(lad.step)]
    say(f"    {'':<30}" + "".join(f"{s:>7}" for s in steps))
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC"):
        g = lad[lad.who == who]
        if g.empty:
            continue
        lv = [sel(g, "Montreal")[lambda d: d.step == s].level.median() for s in steps]
        say(f"  {who} (n={g[g.step == 'native'].record.nunique()})")
        say(f"    {'quiet-REM level, uV':<28}" + "".join(f"{v:7.2f}" for v in lv))
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
                h = sel(g, meth, variant)
                say(f"    {meth + ' ' + variant:<28}" + "".join(
                    f"{100 * h[h.step == s].y.mean():7.1f}" for s in steps))
    say(f"  Paired change, as recorded -> {PAIRED} (same nights):")
    for who in ("MGH", "CAP RBD", "CAP patients", "HMC"):
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
                h = sel(lad[lad.who == who], meth, variant)
                w = h.pivot_table(index="record", columns="step", values="y", aggfunc="first")
                if "native" not in w or PAIRED not in w:
                    continue
                w = w[["native", PAIRED]].dropna()
                d = 100 * (w[PAIRED].mean() - w["native"].mean())
                say(f"    {who:<13} {meth + ' ' + variant:<20} {100 * w['native'].mean():5.1f} -> "
                    f"{100 * w[PAIRED].mean():5.1f}   " + paired_change(w["native"], w[PAIRED]))
                if variant == "FC" and who == "MGH":
                    verdict[f"1 {meth}"] = abs(d) <= 5
                if variant == "FC" and who == "CAP RBD":
                    verdict[f"2 {meth}"] = abs(d) <= 10
    for meth in ("Montreal", "SINBAR"):
        say(f"  criterion 1 ({meth}-FC, MGH within 5 points): "
            f"{'met' if verdict.get(f'1 {meth}') else 'not met'};  criterion 2 ({meth}-FC, "
            f"CAP RBD within 10 points): {'met' if verdict.get(f'2 {meth}') else 'not met'}")
    say()

    # FC3 ────────────────────────────────────────────────────────────────────
    say("FC3 The reference background B: as recorded (sensitivity, CAP patients, MGH, HMC, gap)")
    say(f"    and on the ladder (change as recorded -> {PAIRED}, MGH and CAP RBD), points")
    for meth in ("Montreal", "SINBAR"):
        for variant, ref in [("published", np.nan)] + [("FC", r) for r in REFS] + [("clamp", PRIMARY)]:
            g = sel(nat, meth, variant, ref)
            v = {w: 100 * g[g.who == w].y.mean() for w in ("CAP RBD", "CAP patients", "MGH", "HMC")}
            h = sel(lad, meth, variant, ref)
            ch = {}
            for w in ("MGH", "CAP RBD"):
                p = h[h.who == w].pivot_table(index="record", columns="step", values="y", aggfunc="first")
                p = p[["native", PAIRED]].dropna() if {"native", PAIRED} <= set(p) else p.iloc[:0]
                ch[w] = 100 * (p[PAIRED].mean() - p["native"].mean()) if len(p) else np.nan
            name = meth + " " + variant + ("" if variant == "published" else f" {ref:g}")
            say(f"  {name:<20} RBD {v['CAP RBD']:3.0f}  CAP patients {v['CAP patients']:3.0f}  "
                f"MGH {v['MGH']:3.0f}  HMC {v['HMC']:3.0f}  gap {v['MGH'] - v['HMC']:+5.1f}   "
                f"ladder MGH {ch['MGH']:+5.1f}  RBD {ch['CAP RBD']:+5.1f}")
    say()

    # FC4 ────────────────────────────────────────────────────────────────────
    say("FC4 RBD vs same-system CAP patients (AUC), as recorded")
    for meth, metrics in KEY.items():
        for metric in metrics:
            cells = []
            for variant, ref in [("published", np.nan)] + [("FC", r) for r in REFS]:
                g = sel(nat, meth, variant, ref)
                cells.append(f"{'pub' if variant == 'published' else f'FC {ref:g}'} "
                             f"{auc(g[g.who == 'CAP RBD'][metric], g[g.who == 'CAP patients'][metric])[0]:.3f}")
            say(f"  {meth:<9} {metric:<11} " + "  ".join(cells))
    say()

    # FC5 ────────────────────────────────────────────────────────────────────
    say("FC5 Cut-offs re-derived on CAP (Youden, RBD vs same-system patients), leave-one-out")
    say("    and 2,000 bootstrap re-derivations; external shares flagged [95% interval]")
    rng = np.random.default_rng(0)
    for variant in ("published", "FC"):
        for meth, metrics in KEY.items():
            g = sel(nat, meth, variant)
            for metric in metrics:
                pos = g.loc[g.who == "CAP RBD", metric].dropna().to_numpy(float)
                neg = g.loc[g.who == "CAP patients", metric].dropna().to_numpy(float)
                ext = {k: g.loc[g.who == k, metric].dropna().to_numpy(float) for k in ("MGH", "HMC")}
                cut = youden(pos, neg)
                cv_s = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
                cv_f = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
                boot = {k: [] for k in ext}
                for _ in range(2000):
                    c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
                    for k, v in ext.items():
                        boot[k].append((v >= c).mean())
                fp = {k: 100 * (v >= cut).mean() for k, v in ext.items()}
                ci = {k: np.percentile(v, [2.5, 97.5]) * 100 for k, v in boot.items()}
                say(f"  {variant:<9} {meth:<9} {metric:<11} cut {cut:6.2f}  LOO sens {100 * cv_s:3.0f}%  "
                    f"LOO FP {100 * cv_f:3.0f}%  MGH {fp['MGH']:3.0f} [{ci['MGH'][0]:.0f}-{ci['MGH'][1]:.0f}]  "
                    f"HMC {fp['HMC']:3.0f} [{ci['HMC'][0]:.0f}-{ci['HMC'][1]:.0f}]  gap {fp['MGH'] - fp['HMC']:+.0f}")
    say()

    # FC6 ────────────────────────────────────────────────────────────────────
    say("FC6 Gain: HMC scaled by 0.39 against HMC, same nights")
    for meth in ("Montreal", "SINBAR", "RAI"):
        for variant in (("published",) if meth == "RAI" else ("published", "FC")):
            g = sel(nat, meth, variant)
            w = g[g.cohort.isin(["hmc", "hmc_low"])].pivot_table(index="record", columns="cohort",
                                                                  values="y", aggfunc="first").dropna()
            if {"hmc", "hmc_low"} <= set(w):
                say(f"  {meth + ' ' + variant:<20} {100 * w.hmc.mean():5.1f} -> {100 * w.hmc_low.mean():5.1f}   "
                    + paired_change(w.hmc, w.hmc_low))
    say("  Size of the chin EMG (1 s RMS, uV, median [IQR]); _net = noise (b / 0.91) removed")
    sc = nat[(nat.method == "Montreal") & (nat.variant == "published")]
    cols = [c for c in ("level_1s", "rem_p50_net", "rem_p95_net", "nrem_p50_net") if c in sc]
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC", "HMC x0.39"):
        g = sc[sc.who == who]
        say(f"    {who:<13} n={len(g):<4} " + "  ".join(
            f"{c} {g[c].median():.2f} [{g[c].quantile(.25):.2f}-{g[c].quantile(.75):.2f}]" for c in cols))
    m, h = sc[sc.who == "MGH"], sc[sc.who == "HMC"]
    say("    MGH/HMC ratio of medians: " + "  ".join(f"{c} {m[c].median() / h[c].median():.2f}" for c in cols))
    say()

    # FC7 ────────────────────────────────────────────────────────────────────
    say("FC7 Montreal, phasic bursts up to 10 s instead of 15 s (published cut-offs), % positive")
    a, b = sel(nat, "Montreal"), sel(nat, "Montreal", "max10")
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC"):
        w = pd.merge(a[a.who == who][["record", "y", "phasic_pct"]], b[b.who == who][["record", "y", "phasic_pct"]],
                     on="record", suffixes=("", "_10"))
        say(f"  {who:<13} {100 * w.y.mean():5.1f} -> {100 * w.y_10.mean():5.1f}   nights changed "
            f"{int((w.y != w.y_10).sum())}   phasic % median {w.phasic_pct.median():.1f} -> {w.phasic_pct_10.median():.1f}")
    say()

    # FC8 ────────────────────────────────────────────────────────────────────
    say("FC8 MGH: quiet-REM level and share positive (published) by record batch, and the")
    say("    noise-ladder subset against the rest")
    mg = sel(nat, "Montreal")
    sb = sel(nat, "SINBAR")
    mg = mg[mg.who == "MGH"].merge(sb[sb.who == "MGH"][["record", "y"]], on="record", suffixes=("", "_s"))
    mg["batch"] = mg.record.str[:4]
    for b_, g in mg.groupby("batch"):
        say(f"  {b_:<6} n={len(g):<4} level {g.background_phasic_uV.median():.2f}  Montreal "
            f"{100 * g.y.mean():3.0f}%  SINBAR {100 * g.y_s.mean():3.0f}%")
    sub = set(lad[lad.cohort == "chal18"].record)
    for name, g in (("ladder 300", mg[mg.record.isin(sub)]), ("the rest", mg[~mg.record.isin(sub)])):
        say(f"  {name:<10} n={len(g):<4} level {g.background_phasic_uV.median():.2f}  Montreal "
            f"{100 * g.y.mean():3.0f}%  SINBAR {100 * g.y_s.mean():3.0f}%  age {pd.to_numeric(g.age).mean():.0f}")
    from scipy.stats import spearmanr
    per = mg.groupby("batch").agg(level=("background_phasic_uV", "median"), m=("y", "mean"), s=("y_s", "mean"))
    if len(per) >= 4:
        say(f"  across batches, Spearman rho of median level with share positive: Montreal "
            f"{spearmanr(per.level, per.m)[0]:+.2f}, SINBAR {spearmanr(per.level, per.s)[0]:+.2f} "
            f"({len(per)} batches)")
    say()

    # FC9 ────────────────────────────────────────────────────────────────────
    from src.fetch import cap_demographics
    demo = cap_demographics()
    if demo:
        say("FC9 Age in CAP: RBD against same-system patients under 50 and aged 50 or over")
        cap = nat[nat.who.isin(["CAP RBD", "CAP patients"])].copy()
        cap["age"] = cap.record.map(lambda r: demo.get(r, {}).get("age"))
        parts = {"patients < 50": lambda d: d[(d.who == "CAP patients") & (d.age < 50)],
                 "patients >= 50": lambda d: d[(d.who == "CAP patients") & (d.age >= 50)],
                 "RBD": lambda d: d[d.who == "CAP RBD"]}
        one = cap[(cap.method == "Montreal") & (cap.variant == "published")]
        say("  " + "   ".join(f"{k}: n={len(f(one))}, age {f(one).age.mean():.0f}" for k, f in parts.items()))
        for meth in ("Montreal", "SINBAR"):
            for variant in ("published", "FC"):
                g = sel(cap, meth, variant)
                say(f"  {meth + ' ' + variant:<20} flagged: " + "   ".join(
                    f"{k} {100 * f(g).y.mean():3.0f}%" for k, f in parts.items()))
        for meth, metrics in KEY.items():
            for metric in metrics:
                for variant in ("published", "FC"):
                    g = sel(cap, meth, variant)
                    r_ = parts["RBD"](g)[metric]
                    say(f"  AUC {meth + ' ' + variant:<20} {metric:<11} vs all patients "
                        f"{auc(r_, g[g.who == 'CAP patients'][metric])[0]:.3f}   vs patients >= 50 "
                        f"{auc(r_, parts['patients >= 50'](g)[metric])[0]:.3f}")
        say()

    # FC10 ───────────────────────────────────────────────────────────────────
    say("FC10 CAP healthy volunteers, night by night (mixed recording systems): positive")
    say("     published / floor-corrected (- = not scored: no physical units)")
    h = nat[nat.who == "CAP healthy"]
    flag = lambda g: "-" if g.empty or pd.isna(g.y.iloc[0]) else str(int(g.y.iloc[0]))
    for rec, g in h.groupby("record"):
        mp = sel(g, "Montreal")
        say(f"  {rec:<4} fs {g.chin_fs.iloc[0]:>5g}  units {'yes' if bool(g.unit_known.iloc[0]) else 'no ':<3}  "
            f"level {mp.background_phasic_uV.iloc[0]:6.2f}   Montreal {flag(mp)}/{flag(sel(g, 'Montreal', 'FC'))}"
            f"   SINBAR {flag(sel(g, 'SINBAR'))}/{flag(sel(g, 'SINBAR', 'FC'))}")
    say()

    say("Pre-specified criteria: " + "  ".join(f"{k}: {'met' if v else 'not met'}" for k, v in sorted(verdict.items())))
    out = config.RESULTS / "floor_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figure(nat, lad)


def figure(nat, lad):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import COLOR, save

    colors = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "MGH": "#1f77b4", "HMC": "#9467bd"}
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8), layout="constrained",
                             gridspec_kw={"width_ratios": [1, 1, 1.25]})
    ladder = [s for s in STEPS if s.startswith("n") or s == "native"]
    for ax, meth in zip(axes[:2], ("Montreal", "SINBAR")):
        for who, col in colors.items():
            for variant, ls in (("published", ":"), ("FC", "-")):
                g = sel(lad[lad.who == who], meth, variant)
                if g.empty:
                    continue
                lv = sel(lad[lad.who == who], "Montreal")
                x = [lv[lv.step == s].level.median() for s in ladder]
                y = [100 * g[g.step == s].y.mean() for s in ladder]
                ax.plot(x, y, ls, marker="o", ms=2.5, lw=1, color=col,
                        label=who if variant == "FC" else None)
        ax.set_xscale("log")
        ax.set_xticks([0.5, 1, 2, 4], ["0.5", "1", "2", "4"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("quiet-REM level (µV)")
        ax.set_ylim(0, 100)
        ax.set_title(f"{'AB'[meth == 'SINBAR']}  {meth}")
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].plot([], [], ":", color="k", lw=1, label="published")
    axes[0].plot([], [], "-", color="k", lw=1, label="floor-corrected")
    axes[0].legend(frameon=False, fontsize=6, loc="lower left", bbox_to_anchor=(0.0, 0.38))
    ax = axes[2]
    groups = ["CAP RBD", "CAP patients", "MGH", "HMC"]
    for j, (meth, variant, hatch) in enumerate([("Montreal", "published", ""), ("Montreal", "FC", "//"),
                                               ("SINBAR", "published", ""), ("SINBAR", "FC", "//")]):
        g = sel(nat, meth, variant)
        vals = [100 * g[g.who == w].y.mean() for w in groups]
        ax.bar(np.arange(4) + (j - 1.5) * 0.2, vals, width=0.19, color=COLOR[meth], hatch=hatch,
               edgecolor="white", label=f"{meth}, {'published' if variant == 'published' else 'floor-corrected'}")
    ax.set_xticks(range(4), ["CAP\nRBD", "CAP\npatients", "MGH", "HMC"])
    ax.set_ylim(0, 130)
    ax.set_yticks(range(0, 101, 20))
    ax.set_ylabel("nights flagged positive (%)")
    ax.legend(frameon=False, fontsize=5.5, loc="upper center", ncol=2)
    ax.set_title("C  As recorded")
    save(fig, "fig8_floor_corrected")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_external.py
"""External cohorts: the noise floor and the floor-corrected rule on new systems.

Scores the packs of the development cohorts (CAP, MGH, HMC) and of the external
ones (BOAS, DOD-H, DOD-O) with the published and floor-corrected (B = 1.0 uV)
Montreal and SINBAR rules and the RAI, and reports against the predictions
fixed in paper/preregistration_external.md:

  X1  cohorts: nights, REM, age, floor, floor signature, heartbeat removal
  X2  scale: net N2/N3 and REM levels (E1)
  X3  share flagged: published, floor-corrected, SINBAR-FC at 49.1% (E2, E3, E5)
  X4  floor against share flagged across all groups without RBD (E4)
  X5  DOD-O against MGH's noise ladder (E6)
  X6  BOAS: every night, and by age (descriptive)
  X7  heartbeat check (exploratory, not pre-specified)

BOAS has several nights for some participants; its primary rows are each
participant's first recording. CAP RBD and CAP patients are the nights on
CAP's main system (512 Hz, microvolts).

    python run_external.py -> results/external_scores.csv, external_report.md,
                              figures/fig9_external
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from run_floor import YES, pct, score, score_pack_file, sel
from run_report import MIN_REM, WITHDRAWN

COHORTS = ("cap", "chal18", "hmc", "boas", "dodh", "dodo")
NAME = {"chal18": "MGH", "hmc": "HMC", "boas": "BOAS", "dodh": "DOD-H", "dodo": "DOD-O"}
HEALTHY = ["BOAS", "DOD-H"]
WITHOUT_RBD = ["CAP patients", "MGH", "HMC", "BOAS", "DOD-H", "DOD-O"]
CUT_FC_SINBAR = 49.1          # SINBAR-FC "any" cut-off re-derived on CAP (results_floor, FC5)
MGH_NREM_NET = 1.60           # MGH's median net N2/N3 level, uV (results_floor, FC6)


def label(df):
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record).isin(gone)].copy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select([same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"])],
                          ["CAP RBD", "CAP patients"],
                          default=df.cohort.map(NAME).fillna(""))
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    df["ref_key"] = df.ref_uv.fillna(0.0)
    # BOAS: a participant's first recording is primary
    df["primary"] = True
    b = df.cohort == "boas"
    if b.any():
        num = df.loc[b, "record"].str.extract(r"(\d+)")[0].astype(int)
        pid = df.loc[b, "pid"].fillna(df.loc[b, "record"]).astype(str)
        first = num.groupby(pid).transform("min")
        df.loc[b, "primary"] = (num == first).to_numpy()
    return df


def periodicity(pack, band):
    """How strongly the REM envelope repeats at a heart rate: the largest spectral peak of
    the envelope between 0.7 and 2 Hz over the median in 0.5-3 Hz (Welch, 20-s windows,
    REM stretches of at least a minute); higher = more heartbeat in the EMG."""
    from scipy.signal import welch
    from src.rwa import _segments, _select
    ep, env, _, _ = _select(pack, band)
    if env is None or len(ep) < 2:
        return np.nan
    fs = pack["meta"]["env_fs"]
    acc, n = None, 0
    for a, b in _segments(ep):
        x = env[a:b].reshape(-1).astype(float)
        if len(x) >= fs * 60:
            f, p = welch(x - x.mean(), fs=fs, nperseg=int(fs * 20))
            acc, n = (p if acc is None else acc + p), n + 1
    if not n:
        return np.nan
    band_, hr = (f >= 0.5) & (f <= 3.0), (f >= 0.7) & (f <= 2.0)
    return float(acc[hr].max() / np.median(acc[band_]))


def heartbeat_check(path):
    """Exploratory (not pre-specified): the heartbeat periodicity of the REM envelope with
    and without heartbeat removal, and the calls with the heartbeat left in."""
    from src.pack import load_pack
    from src.rwa import METHODS, floor_corrected, score_night
    pack = load_pack(path)
    m = pack["meta"]
    out = {"cohort": m.get("cohort", path.parent.name), "record": m["record"],
           "period_native": periodicity(pack, "native"),
           "period_with_ecg": periodicity(pack, "native_with_ecg")}
    for key in ("montreal", "sinbar"):
        for tag, meth in (("pub", METHODS[key]), ("fc", floor_corrected(METHODS[key], 1.0))):
            r = score_night(pack, meth, "native_with_ecg")
            out[f"{key}_{tag}_with_ecg"] = float(bool(r.get("positive"))) if r.get("positive") is not None else np.nan
    return [out]


def main():
    packs = [p for c in COHORTS for p in sorted((config.DURABLE / "packs" / c).glob("*.npz"))
             if ".tmp" not in p.name]
    print(f"scoring {len(packs)} packs", flush=True)
    df = label(score(packs, score_pack_file))
    from src.fetch import cap_demographics
    demo = cap_demographics()
    if demo:
        c = df.cohort == "cap"
        df.loc[c, "age"] = df.loc[c, "record"].map(lambda r: demo.get(r, {}).get("age"))
    df.to_csv(config.RESULTS / "external_scores.csv", index=False)
    hb = score([p for p in packs if p.parent.name in ("chal18", "hmc", "boas", "dodo", "dodh")],
               heartbeat_check)
    all_nights = df[(df.rem_minutes >= MIN_REM) & (df.who != "")]
    d = all_nights[all_nights.primary]

    lines = []
    say = lambda s="": (print(s), lines.append(s))
    verdict = {}
    groups = ["CAP RBD"] + WITHOUT_RBD
    one = d[(d.method == "Montreal") & (d.variant == "published")]
    one = one.assign(signature=one.nrem_p5 / one.level_1s)

    # X1 ─────────────────────────────────────────────────────────────────────
    say("X1  Cohorts (nights with >= 10 min REM; BOAS: first night per participant)")
    for w in groups:
        g = one[one.who == w]
        if g.empty:
            continue
        age = pd.to_numeric(g.age, errors="coerce")
        say(f"  {w:<13} n={len(g):<4} REM {g.rem_minutes.median():5.1f} min  age "
            f"{age.mean():4.0f} ({age.min():.0f}-{age.max():.0f})  fs {g.chin_fs.median():g}  "
            f"floor {g.level_1s.median():.2f} [{g.level_1s.quantile(.25):.2f}-{g.level_1s.quantile(.75):.2f}] uV  "
            f"signature {g.signature.median():.2f}  heartbeat removed {int((g.beats > 0).sum())}/{len(g)}")
    say()

    # X2 ─────────────────────────────────────────────────────────────────────
    say("X2  Scale: 1 s RMS with the noise removed (uV, median [IQR]); E1: net N2/N3 within")
    say(f"    0.67-1.5 x MGH's {MGH_NREM_NET} uV")
    for w in groups:
        g = one[one.who == w]
        if g.empty:
            continue
        v = g.nrem_p50_net
        ok = 0.67 * MGH_NREM_NET <= v.median() <= 1.5 * MGH_NREM_NET
        if w in ("BOAS", "DOD-H", "DOD-O"):
            verdict[f"E1 {w}"] = ok
        say(f"  {w:<13} N2/N3 {v.median():.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}]  "
            f"REM {g.rem_p50_net.median():.2f}  REM p95 {g.rem_p95_net.median():.2f}"
            + (f"   E1 {'met' if ok else 'not met'}" if w in ("BOAS", "DOD-H", "DOD-O") else ""))
    say()

    # X3 ─────────────────────────────────────────────────────────────────────
    say("X3  Share flagged, % [95% CI]: published cut-offs; floor-corrected (B = 1.0 uV);")
    say(f"    SINBAR-FC at the cut-off re-derived on CAP ({CUT_FC_SINBAR}%)")
    rows = {}
    for w in groups:
        g = d[d.who == w]
        if g.empty:
            continue
        cells = {}
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC")):
                cells[f"{meth} {variant}"] = sel(g, meth, variant).y
        fc_any = sel(g, "SINBAR", "FC").any_pct
        cells["SINBAR-FC >= 49.1"] = (fc_any >= CUT_FC_SINBAR).astype(float).where(fc_any.notna())
        rows[w] = {k: pct(v) for k, v in cells.items()}
        say(f"  {w}")
        for k, (p, lo, hi, n) in rows[w].items():
            say(f"    {k:<22} {100 * p:5.1f} [{100 * lo:.0f}-{100 * hi:.0f}]  (n={n})")
    floor = one.groupby("who").level_1s.median()
    # E2: quiet healthy systems flag healthy people
    for w in HEALTHY:
        if w in rows and floor.get(w, np.inf) < 1.0:
            verdict[f"E2 {w}"] = rows[w]["SINBAR published"][0] >= 0.30
    # E3: two healthy cohorts, floors differing >= 1.5x
    if all(w in rows for w in HEALTHY):
        a, b = sorted(HEALTHY, key=lambda w: floor[w])          # a = quieter
        if floor[b] / floor[a] >= 1.5:
            for meth, need in (("SINBAR", 0.0), ("Montreal", 0.05)):
                pub = rows[a][f"{meth} published"][0] - rows[b][f"{meth} published"][0]
                fc = rows[a][f"{meth} FC"][0] - rows[b][f"{meth} FC"][0]
                if meth == "SINBAR" or pub >= need:
                    verdict[f"E3 {meth}"] = pub > 0 and abs(fc) <= abs(pub) / 2
                    say(f"  E3 {meth}: {a} - {b} published {100 * pub:+.1f}, floor-corrected {100 * fc:+.1f} points")
        else:
            say(f"  E3 not applicable: healthy floors {floor[a]:.2f} and {floor[b]:.2f} uV differ < 1.5x")
    # E5: the portable cut-off stays specific
    for w, limit in (("BOAS", 0.10), ("DOD-H", 0.10), ("DOD-O", 0.20)):
        if w in rows:
            verdict[f"E5 {w}"] = rows[w]["SINBAR-FC >= 49.1"][0] <= limit
    say()

    # X4 ─────────────────────────────────────────────────────────────────────
    from scipy.stats import spearmanr
    say("X4  Across groups without RBD: median floor against share flagged (Spearman rho)")
    have = [w for w in WITHOUT_RBD if w in rows]
    x = [floor[w] for w in have]
    for meth in ("SINBAR", "Montreal"):
        pub = [rows[w][f"{meth} published"][0] for w in have]
        fc = [rows[w][f"{meth} FC"][0] for w in have]
        rp, rf = spearmanr(x, pub)[0], spearmanr(x, fc)[0]
        say(f"  {meth:<9} published rho {rp:+.2f}   floor-corrected rho {rf:+.2f}   ({len(have)} groups: "
            + ", ".join(f"{w} {floor[w]:.2f} uV" for w in have) + ")")
        if meth == "SINBAR":
            verdict["E4"] = rp < 0 and abs(rf) < abs(rp)
    say()

    # X5 ─────────────────────────────────────────────────────────────────────
    say("X5  DOD-O against MGH's noise ladder (published cut-offs; E6: within 10 points)")
    lad_path = config.RESULTS / "ladder_scores.csv"
    if "DOD-O" in rows and lad_path.exists():
        lad = pd.read_csv(lad_path)
        lad = lad[(lad.who == "MGH") & (lad.rem_minutes >= MIN_REM)]
        steps = ["native", "n050", "n100", "n150", "n200", "n300"]
        f0 = floor["DOD-O"]
        ok_all = True
        for meth in ("Montreal", "SINBAR"):
            g = lad[lad.method == meth]
            lv = [g[g.step == s].level.median() for s in steps]
            yy = [100 * g[g.step == s].y.mean() for s in steps]
            if not (min(lv) <= f0 <= max(lv)):
                say(f"  {meth}: DOD-O floor {f0:.2f} uV outside the ladder's {min(lv):.2f}-{max(lv):.2f} uV: not applicable")
                ok_all = None
                continue
            pred = float(np.interp(np.log(f0), np.log(lv), yy))
            obs = 100 * rows["DOD-O"][f"{meth} published"][0]
            say(f"  {meth:<9} predicted at {f0:.2f} uV: {pred:5.1f}%   observed {obs:5.1f}%   (diff {obs - pred:+.1f})")
            if ok_all is not None:
                ok_all = ok_all and abs(obs - pred) <= 10
        if ok_all is not None:
            verdict["E6"] = ok_all
    say()

    # X6 ─────────────────────────────────────────────────────────────────────
    b = all_nights[all_nights.cohort == "boas"]
    if len(b):
        say("X6  BOAS: every night (n=%d) and first nights by age, %% flagged" % b.record.nunique())
        for name, g in (("all nights", b), ("first nights < 50 y", d[(d.cohort == "boas") & (pd.to_numeric(d.age, errors='coerce') < 50)]),
                        ("first nights >= 50 y", d[(d.cohort == "boas") & (pd.to_numeric(d.age, errors='coerce') >= 50)])):
            cells = []
            for meth in ("Montreal", "SINBAR"):
                for variant in ("published", "FC"):
                    cells.append(f"{meth} {variant} {100 * sel(g, meth, variant).y.mean():4.0f}")
            n = g.record.nunique()
            say(f"  {name:<22} n={n:<4} " + "  ".join(cells))
        say()

    # X7 ─────────────────────────────────────────────────────────────────────
    say("X7  Heartbeat check (exploratory, not pre-specified): periodicity of the REM envelope")
    say("    at a heart rate (higher = more heartbeat), and the share flagged with it left in")
    h = hb.merge(d[(d.method == "Montreal") & (d.variant == "published")][["cohort", "record", "who"]],
                 on=["cohort", "record"])
    for w in ("MGH", "HMC", "DOD-O", "DOD-H", "BOAS"):
        g = h[h.who == w]
        if g.empty:
            continue
        cells = []
        for meth in ("Montreal", "SINBAR"):
            for variant, tag in (("published", "pub"), ("FC", "fc")):
                removed = sel(d[d.who == w], meth, variant).set_index("record").y
                left = g.set_index("record")[f"{meth.lower()}_{tag}_with_ecg"].reindex(removed.index)
                cells.append(f"{meth} {variant} {100 * removed.mean():.0f} -> {100 * left.mean():.0f}")
        say(f"  {w:<6} n={len(g):<4} periodicity after removal {g.period_native.median():.2f} "
            f"[p90 {g.period_native.quantile(.9):.2f}], left in {g.period_with_ecg.median():.2f}")
        say("         % flagged, removed -> left in: " + "   ".join(cells))
    ref = h[h.who == "MGH"].period_native.quantile(0.9)
    b = h[h.who == "BOAS"]
    if len(b):
        clean = set(b[b.period_native <= ref].record)
        say(f"  BOAS nights no more periodic than MGH's after removal (<= {ref:.2f}, MGH p90): "
            f"{len(clean)}/{len(b)}; % flagged among them:")
        say("    " + "   ".join(
            f"{meth} {variant} {100 * sel(d[(d.who == 'BOAS') & d.record.isin(clean)], meth, variant).y.mean():.0f}"
            for meth in ("Montreal", "SINBAR") for variant in ("published", "FC")))
    say()

    say("Pre-specified predictions: " + "  ".join(
        f"{k}: {'met' if v else 'not met'}" for k, v in sorted(verdict.items())))
    out = config.RESULTS / "external_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figure(one, rows, floor)


def figure(one, rows, floor):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import save

    style = {"CAP RBD": ("#d62728", "*"), "CAP patients": ("#7f7f7f", "o"), "MGH": ("#1f77b4", "o"),
             "HMC": ("#9467bd", "o"), "DOD-O": ("#17becf", "o"), "BOAS": ("#2ca02c", "s"),
             "DOD-H": ("#bcbd22", "s")}
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True, layout="constrained")
    for ax, meth in zip(axes, ("SINBAR", "Montreal")):
        for w, (col, mk) in style.items():
            if w not in rows:
                continue
            x = floor[w]
            p, f = 100 * rows[w][f"{meth} published"][0], 100 * rows[w][f"{meth} FC"][0]
            ax.annotate("", xy=(x, f), xytext=(x, p),
                        arrowprops=dict(arrowstyle="->", color=col, lw=0.8, alpha=0.7))
            ax.plot(x, p, mk, color=col, mfc="white", ms=6)
            ax.plot(x, f, mk, color=col, ms=6, label=w)
        ax.set_xscale("log")
        ax.set_xticks([0.3, 0.5, 1, 2, 3], ["0.3", "0.5", "1", "2", "3"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("median quiet-REM level of the cohort (µV)")
        ax.set_ylim(0, 100)
        ax.set_title(f"{'AB'[meth == 'Montreal']}  {meth}")
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].plot([], [], "o", color="k", mfc="white", label="published")
    axes[0].plot([], [], "o", color="k", label="floor-corrected")
    axes[0].legend(frameon=False, fontsize=6, loc="upper right")
    axes[1].legend(frameon=False, fontsize=6, loc="upper right", ncol=2)
    save(fig, "fig9_external")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/src/rbdtector.py
"""RBDtector, run unchanged on our nights (paper/preregistration_parkinsons.md).

RBDtector (Röthenbacher et al., Sci Rep 2022; validated at a second center by
Joza et al., J Sleep Res 2025) is the open-source scorer of REM sleep without
atonia used in Parkinson's and RBD research; it was developed on patients with
Parkinson's disease with and without RBD, patients with isolated RBD and
controls. Its default rule counts chin activity where the RMS of a 30 ms chunk
exceeds twice a baseline, and the baseline is the lowest 30 s RMS window of
artifact-free REM in each REM period: on an atonic night, mostly the recording
system's own noise.

`write_inputs` puts one night in RBDtector's input format (an EDF with the chin
EMG as channel 'EMG', in microvolts, and the 'Sleep profile', 'Classification
Arousals' and 'Flow Events' text files of SOMNOmedics' software). `run` calls
RBDtector's own controller on it and returns the chin percentages it reports.
RBDtector does not filter: the signal is band-passed (10-100 Hz) and notched
as an acquisition system would, and the heartbeat is left in, as RBDtector
sees it in practice. No arousals are given (one dummy event in the first
second written, which is not REM), so no REM is excluded for arousals.

The floor-corrected variant changes one thing: the baseline RBDtector computes
is replaced, before detection, by half the floor-corrected threshold
(src/rwa.py, `threshold`): with n = baseline / r the noise RMS that baseline
implies (r from `noise_ratio`, simulated), activity must exceed
max(sqrt(n^2 + 3 B^2), 1.5 n) instead of 2 x baseline, B = 1.0 uV.
"""
import datetime as dt
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO = "https://github.com/aroethen/RBDtector"
COMMIT = "7c9c73d231a29d8ccf56d7eec69ae117ae781c0c"     # master, 19 May 2022
HOME = Path("/content/RBDtector")
# Röthenbacher et al. 2022, Table 5: chin alone, after removal of arousals,
# snoring and technical artifacts (cut-offs at ~100% specificity in their data)
CUTOFF = {"any_pct": 23.5, "phasic_pct": 19.0, "tonic_pct": 7.1}
B_REF, GUARD = 1.0, 1.5
START = dt.datetime(2020, 1, 1, 22, 0, 0)
STAGE = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
PHYS_UV = 2000.0                      # EDF physical range +-2 mV: 0.06 uV steps

_state = {"variant": "published", "ratio": None, "baseline_uv": np.nan,
          "fast": True, "cache": True}


def set_mode(fast=True, cache=True):
    """fast=False restores RBDtector's own find_rem_blocks; cache=False makes the
    floor-corrected run recompute the baseline instead of reusing the published
    run's. Both False = the program as released (for the verification run)."""
    from app_logic.PSG import PSG
    _state["fast"], _state["cache"] = bool(fast), bool(cache)
    PSG.find_rem_blocks = staticmethod(_find_rem_blocks if fast else PSG._rwa_original_blocks)


def install(home=HOME):
    """Clone RBDtector at the pinned commit (once) and make it importable."""
    home = Path(home)
    if not (home / "RBDtector" / "app_logic" / "PSG.py").exists():
        if home.exists():
            shutil.rmtree(home)
        subprocess.run(["git", "clone", "-q", REPO, str(home)], check=True)
        subprocess.run(["git", "-C", str(home), "checkout", "-q", COMMIT], check=True)
    pkg = str(home / "RBDtector")
    if pkg not in sys.path:
        sys.path.insert(0, pkg)
    _patch()
    return home


def noise_ratio(fs=256.0, minutes=10.0, n=20, seed=0):
    """RBDtector's baseline (lowest 30 s RMS of a REM period) over the RMS of
    band-limited white noise alone: median over `n` simulated REM periods."""
    from .extract import bandpass
    rng = np.random.default_rng(seed)
    w, out = int(30 * fs), []
    for _ in range(n):
        x, _ = bandpass(rng.normal(0.0, 1.0, int(minutes * 60 * fs) + int(10 * fs)), fs)
        x = x[int(5 * fs):-int(5 * fs)]
        c = np.r_[0.0, np.cumsum(x * x)]
        roll = np.sqrt((c[w:] - c[:-w]) / w)
        out.append(roll.min() / np.sqrt(np.mean(x * x)))
    return float(np.median(out))


def _find_rem_blocks(MIN_REM_BLOCK_LENGTH_IN_S, is_rem_series):
    """RBDtector's PSG.find_rem_blocks (commit 7c9c73d), copied with one change:
    the REM sample nearest to a short REM piece is found with numpy instead of
    Python's min(..., key=abs) over every REM sample at 256 Hz (about 35 s per
    night). Both return the first minimum, so the output is the same."""
    from util import settings
    continuuos_rem_ge_min_block_length = is_rem_series \
        .groupby(is_rem_series.diff().ne(0).cumsum()) \
        .transform('size') \
        .ge(settings.RATE * MIN_REM_BLOCK_LENGTH_IN_S)
    min_rem_block = continuuos_rem_ge_min_block_length & is_rem_series
    rem_block_counter = 0

    def transform_to_rem_group_name(x):
        if x.all():
            nonlocal rem_block_counter
            rem_block_counter += 1
            return rem_block_counter
        else:
            return 0

    numbered_rem_blocks = min_rem_block \
        .groupby(min_rem_block.diff().ne(0).cumsum()) \
        .transform(transform_to_rem_group_name)
    small_rem_pieces = is_rem_series & ~min_rem_block
    block_indices = numbered_rem_blocks[numbered_rem_blocks != 0].index   # loop-invariant

    def nearest(diff):                       # == min(diff, key=abs)
        return diff[int(np.argmin(np.abs(diff.asi8)))]

    def assign_small_piece_to_rem_block(x):
        if x.all():
            first_index_of_piece = x.index[0]
            last_index_of_piece = x.index[-1]
            closest_to_beginning_of_piece = nearest(first_index_of_piece - block_indices)
            closest_to_end_of_piece = nearest(last_index_of_piece - block_indices)
            if closest_to_beginning_of_piece < closest_to_end_of_piece:
                block_number = numbered_rem_blocks[first_index_of_piece - closest_to_beginning_of_piece]
            else:
                block_number = numbered_rem_blocks[last_index_of_piece - closest_to_end_of_piece]
            return block_number
        else:
            return 0

    small_rem_pieces_assigned_to_numbered_rem_blocks = small_rem_pieces \
        .groupby(small_rem_pieces.diff().ne(0).cumsum()) \
        .transform(assign_small_piece_to_rem_block)
    all_rem_sleep_numbered = numbered_rem_blocks + small_rem_pieces_assigned_to_numbered_rem_blocks
    return all_rem_sleep_numbered, min_rem_block, numbered_rem_blocks, small_rem_pieces


def _patch():
    """Wrap RBDtector's baseline finder: record the baseline, and replace it by
    half the floor-corrected threshold when the variant is 'FC'. Also give
    pyedflib's signal headers the key RBDtector (2022) reads: pyedflib has since
    renamed 'sample_rate' to 'sample_frequency' (an I/O shim; detection unchanged)."""
    from pyedflib import highlevel
    if not getattr(highlevel, "_rwa_patched", False):
        read_edf = highlevel.read_edf

        def read_edf_with_rate(*args, **kwargs):
            signals, headers, header = read_edf(*args, **kwargs)
            for h in headers:
                if "sample_rate" not in h and "sample_frequency" in h:
                    h["sample_rate"] = h["sample_frequency"]
            return signals, headers, header

        highlevel.read_edf = read_edf_with_rate
        highlevel._rwa_patched = True
    from app_logic.PSG import PSG
    if getattr(PSG, "_rwa_patched", False):
        return
    original = PSG.find_baselines
    PSG._rwa_original_blocks = PSG.find_rem_blocks
    PSG.find_rem_blocks = staticmethod(_find_rem_blocks)

    def find_baselines(*args, **kwargs):
        # The floor-corrected run of a night starts from the baseline its published
        # run computed from the same input files (identical, so reused, not redone)
        if _state["variant"] == "FC" and _state["cache"] \
                and _state.get("cache_key") == _state.get("folder"):
            df_b, df_art = (d.copy() for d in _state["cache"])
        else:
            df_b, df_art = original(*args, **kwargs)
            _state["cache_key"], _state["cache"] = _state.get("folder"), (df_b.copy(), df_art.copy())
        cols = [c for c in df_b.columns if c.endswith("_baseline")]
        if cols:
            _state["baseline_uv"] = float(np.nanmedian(df_b[cols[0]].to_numpy(float)))
        if _state["variant"] == "FC":
            r = _state["ratio"] or noise_ratio()
            _state["ratio"] = r
            for c in cols:
                b = df_b[c].to_numpy(float)
                nz = b / r
                thr = np.maximum(np.sqrt(nz * nz + 3.0 * B_REF * B_REF), GUARD * nz)
                df_b[c] = thr / 2.0
        return df_b, df_art

    PSG.find_baselines = staticmethod(find_baselines)
    PSG._rwa_patched = True


def rem_span(hypnogram, margin=2):
    """Epochs [a, b) from `margin` epochs before the first REM epoch to `margin`
    epochs after the last (the whole night if there is no REM)."""
    rem = np.flatnonzero(np.asarray(hypnogram) == 4)
    if len(rem) == 0:
        return 0, len(hypnogram)
    return max(0, int(rem[0]) - margin), min(len(hypnogram), int(rem[-1]) + margin + 1)


def write_inputs(folder, x_uv, fs, hypnogram, epoch_s=30, trim=True, arousals=None):
    """One night in RBDtector's input format; returns the folder.

    arousals: scored arousal intervals, (start, end) in seconds from the start of
    the night, written to the 'Classification Arousals' file so that RBDtector
    excludes them as its published cut-off assumes (default: none).

    trim: write only the stretch from two epochs before the first REM epoch to
    two after the last. RBDtector scores REM only, the clock times stay those
    of the whole night (so its 30-ms, 3-s and 30-s grids fall where they would)
    and the first and last epochs it sees are still not REM, so the result is
    the whole night's, in about two thirds of the time (checked on test nights)."""
    import pyedflib
    from pyedflib import highlevel
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    if not float(fs).is_integer():
        raise ValueError(f"EDF needs whole samples per second, got {fs} Hz")
    fs = int(fs)
    n_sec = len(x_uv) // fs
    n_ep = min(len(hypnogram), n_sec // epoch_s)
    a, b = rem_span(hypnogram[:n_ep]) if trim else (0, n_ep)
    hypnogram = hypnogram[a:b]
    start = START + dt.timedelta(seconds=a * epoch_s)
    x = np.clip(np.asarray(x_uv[a * epoch_s * fs: b * epoch_s * fs], dtype=np.float64),
                -PHYS_UV, PHYS_UV)
    # pyEDFlib 0.1.22 calls the rate 'sample_rate', later versions 'sample_frequency'
    import inspect
    rate = "sample_frequency" if "sample_frequency" in \
        inspect.signature(highlevel.make_signal_header).parameters else "sample_rate"
    sh = highlevel.make_signal_header("EMG", dimension="uV", physical_min=-PHYS_UV,
                                      physical_max=PHYS_UV, digital_min=-32768,
                                      digital_max=32767, **{rate: fs})
    hdr = highlevel.make_header(startdate=start)
    highlevel.write_edf(str(folder / "night.edf"), [x], [sh], hdr,
                        file_type=pyedflib.FILETYPE_EDFPLUS)
    clock = lambda s: (start + dt.timedelta(seconds=s)).strftime("%H:%M:%S,%f")[:-3]
    head = start.strftime("%d.%m.%Y %H:%M:%S")
    lines = ["Signal ID: SchlafProfil\\profil", f"Start Time: {head}", "Unit: ",
             "Signal Type: Discret", "Events list: N4,N3,N2,N1,REM,Wake,Movement",
             "Rate: 30 s", ""]
    lines += [f"{clock(i * epoch_s)}; {STAGE.get(int(c), 'A')}" for i, c in enumerate(hypnogram)]
    (folder / "Sleep profile - night.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    t0, t1 = a * epoch_s, b * epoch_s
    events = sorted((float(s0) - t0, float(s1) - t0) for s0, s1 in (arousals or [])
                    if s1 > t0 and s0 < t1)
    for name, sid, rows in (("Classification Arousals - night.txt", "KorrelationMA\\MAK", events),
                            ("Flow Events - night.txt", "FlowEvents\\Events", [])):
        ev = [f"Signal ID: {sid}", f"Start Time: {head}", "Unit: s", "Signal Type: Impuls", ""]
        if rows:
            ev += [f"{clock(max(s0, 0.0))}-{clock(min(s1, t1 - t0))}; {max(s1 - s0, 0):.0f};Arousal"
                   for s0, s1 in rows]
        else:
            ev.append(f"{clock(0)}-{clock(1)}; 1;Dummy (first second, wake)")
        (folder / name).write_text("\n".join(ev) + "\n", encoding="utf-8")
    return folder


def _flatten(df_out):
    """RBDtector's output frame -> {joined labels: value}."""
    import pandas as pd
    s = df_out
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0] if s.shape[1] == 1 else s.stack()
    out = {}
    for key, val in s.items():
        parts = key if isinstance(key, tuple) else (key,)
        out["|".join(str(p) for p in parts if str(p))] = val
    return out


def run(folder, variant="published"):
    """RBDtector's controller on one input folder. Returns the chin percentages
    (tonic, phasic, any), the REM epochs it evaluated and its baseline (uV)."""
    import warnings
    warnings.filterwarnings("ignore")
    from app_logic.PSG_controller import PSGController
    _state["variant"] = variant
    _state["baseline_uv"] = np.nan
    _state["folder"] = str(folder)
    df_out, _ = PSGController.run_rbd_detection(str(folder), str(folder))
    if variant == "FC":
        _state["cache_key"] = _state["cache"] = None
    flat = _flatten(df_out)

    def pick(suffix, need="EMG"):
        hits = [v for k, v in flat.items() if k.endswith(suffix) and need in k
                and not k.startswith("Global")]
        return float(hits[0]) if hits else np.nan

    return {"tonic_pct": pick("tonic_%"), "phasic_pct": pick("phasic_%"),
            "any_pct": pick("any_%"),
            "rem_miniepochs": pick("REM_MiniEpochs_WO-Artifacts", need=""),
            "rem_macroepochs": pick("REM_MacroEpochs_WO-Artifacts", need=""),
            "baseline_uv": _state["baseline_uv"], "noise_ratio": _state["ratio"],
            "keys": ";".join(sorted(flat))[:2000]}

In [ ]:
%%writefile /content/rbd-rwa/src/floorcheck.py
"""Floor check: can this recording system measure REM sleep without atonia?

The relative RWA criteria count chin activity above k x background, and the
background of an atonic night is mostly the recording system's own noise, so
every RWA threshold should be reported together with that floor. From one
night - the chin EMG and the 30 s sleep stages - this reports:

  floor b      quiet-REM level: 5th percentile of 1 s chin RMS in REM (uV), after
               band-pass (10-100 Hz, lower at low sampling rates), mains notch
               and, when an ECG is given, heartbeat removal
  noise n      the noise RMS that floor implies (b / 0.91, white noise)
  gain check   median 1 s N2/N3 chin RMS with that noise removed (uV); 1.0-1.7 uV
               in the seven cohorts of the paper - far outside suggests a gain or
               unit problem rather than a different population
  thresholds   what 'k x background' means in uV under the published rule and
               under the floor-corrected rule (B = 1.0 uV), and the muscle
               activity each requires on top of the noise
  verdict      whether the floor-corrected rule applies: sampling rate >= 200 Hz
               and b <= B / 0.59 (1.7 uV for B = 1.0)

    from src.floorcheck import floor_report, read_stages, format_report
    print(format_report(floor_report(chin_uv, fs, read_stages("night_stages.txt"))))
or, for an EDF, `python check_floor.py night.edf night_stages.txt`.
"""
import numpy as np

from .rwa import NOISE_P5_RATIO

B_REF = 1.0                    # reference background of the floor-corrected rule, uV
GUARD = 1.5
MIN_FS = 200.0
MAX_FLOOR = B_REF / 0.59       # above this, FC cannot hold the published requirement
EPOCH_S = 30
RULES = (("SINBAR any (2 x b)", 2.0), ("Montreal phasic (4 x b)", 4.0))
LABELS = {"W": 0, "WAKE": 0, "N1": 1, "S1": 1, "N2": 2, "S2": 2, "N3": 3, "S3": 3,
          "S4": 3, "N4": 3, "R": 4, "REM": 4}


def read_stages(path, rem_code=None, nrem_codes=(2, 3)):
    """30 s stages from a text file, one per line (extra columns ignored).
    Labels W/WAKE, N1, N2, N3 (N4/S4 -> N3), R/REM are read directly; for numeric
    files give the code used for REM (`rem_code`) and for N2/N3 (`nrem_codes`).
    Returns codes 0 W, 1 N1, 2 N2, 3 N3, 4 REM, -1 anything else."""
    number = lambda t: t.replace(".", "", 1).lstrip("-").isdigit()
    out = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            tok = line.replace(",", " ").replace(";", " ").replace("\t", " ").split()
            if not tok:
                continue
            timed = tok[0][0].isdigit() and ":" in tok[0]          # 22:00:00 ...
            if not timed and any(t.endswith(":") for t in tok[:2]):
                continue                                         # header "Key: value"
            last = tok[-1].upper()
            if last in LABELS:
                out.append(LABELS[last])
            elif not timed and number(tok[0]):
                if rem_code is None:
                    raise ValueError("numeric stages: give rem_code (and nrem_codes)")
                v = int(float(tok[0]))
                out.append(4 if v == rem_code else 2 if v in nrem_codes else -1)
            else:
                out.append(-1)                                   # unknown stage keeps its slot
    return np.asarray(out, dtype=np.int8)


def thresholds(b):
    """Published and floor-corrected thresholds (uV RMS) for a floor b, and the
    muscle activity (uV RMS, added in power to the noise) each requires."""
    n = b / NOISE_P5_RATIO
    out = {}
    for name, k in RULES:
        pub = k * b
        fc = max(float(np.sqrt(n * n + (k * k - 1.0) * B_REF * B_REF)), GUARD * b)
        out[name] = {"k": k, "published_uv": pub, "fc_uv": fc,
                     "published_needs_uv": float(np.sqrt(max(pub * pub - n * n, 0.0))),
                     "fc_needs_uv": float(np.sqrt(max(fc * fc - n * n, 0.0)))}
    return out


def floor_report(chin_uv, fs, stages, ecg=None, fs_ecg=None, mains=None):
    """The floor check for one night (chin EMG in microvolts, 30 s stages)."""
    from .ecg import plausible, r_peaks, subtract_heartbeat
    from .extract import bandpass, bin_envelope
    x, band = bandpass(chin_uv, fs, mains=mains)
    beats = np.empty(0)
    if ecg is not None and fs_ecg:
        beats = r_peaks(ecg, fs_ecg)
        if plausible(beats, len(ecg) / fs_ecg):
            x = subtract_heartbeat(x, fs, beats)
        else:
            beats = np.empty(0)
    sec, _ = bin_envelope(x, fs, env_fs=1)            # 1 s RMS
    n = min(len(sec), len(stages) * EPOCH_S)
    stage = np.repeat(np.asarray(stages), EPOCH_S)[:n]
    rem, nrem = sec[:n][stage == 4], sec[:n][np.isin(stage, (2, 3))]
    if len(rem) < 600:
        raise ValueError(f"only {len(rem) / 60:.1f} min of REM: need at least 10")
    b = float(np.percentile(rem, 5))
    noise = b / NOISE_P5_RATIO
    net = float(np.sqrt(max(np.median(nrem) ** 2 - noise ** 2, 0.0))) if len(nrem) >= 60 else np.nan
    reasons = []
    if fs < MIN_FS:
        reasons.append(f"sampling rate {fs:g} Hz < {MIN_FS:g} Hz")
    if b > MAX_FLOOR:
        reasons.append(f"floor {b:.2f} uV > {MAX_FLOOR:.2f} uV")
    return {"fs": float(fs), "band_hz": [round(v, 1) for v in band],
            "heartbeat_removed": bool(len(beats)), "rem_min": round(len(rem) / 60, 1),
            "floor_uv": round(b, 3), "noise_uv": round(noise, 3),
            "net_n2n3_uv": round(net, 3) if np.isfinite(net) else None,
            "thresholds": thresholds(b), "fc_applies": not reasons, "reasons": reasons}


def format_report(r):
    """Plain-text version of `floor_report`."""
    lines = [f"Floor check  ({r['fs']:g} Hz, band {r['band_hz'][0]}-{r['band_hz'][1]} Hz, "
             f"{r['rem_min']} min REM, heartbeat {'removed' if r['heartbeat_removed'] else 'not removed'})",
             f"  quiet-REM floor b      {r['floor_uv']:.2f} uV   (noise RMS ~{r['noise_uv']:.2f} uV)",
             f"  net N2/N3 tone         {r['net_n2n3_uv'] if r['net_n2n3_uv'] is not None else '-'} uV"
             "   (1.0-1.7 uV in the paper's cohorts; far outside: check gain/units)"]
    for name, t in r["thresholds"].items():
        lines.append(f"  {name:<25} published {t['k']:g} x b = {t['published_uv']:.2f} uV "
                     f"(needs {t['published_needs_uv']:.2f} uV of muscle activity);  "
                     f"floor-corrected {t['fc_uv']:.2f} uV (needs {t['fc_needs_uv']:.2f} uV)")
    lines.append("  verdict: " + ("the floor-corrected rule applies (B = 1.0 uV)" if r["fc_applies"]
                                  else "outside the floor-corrected rule's range: " + "; ".join(r["reasons"])))
    lines.append("  report b with every RWA percentage from this system")
    return "\n".join(lines)

In [ ]:
%%writefile /content/rbd-rwa/run_rbdtector.py
"""RBDtector on the noise ladder and on three recording systems.

Pre-specified in paper/preregistration_parkinsons.md. Every night is streamed,
scored by RBDtector itself (src/rbdtector.py) with its published rule and with
the floor-corrected baseline, and discarded. The noise is the noise ladder's
own: the same seeded track and the same scale per step as run_cohort.py
--ladder, added to the night's band-passed chin EMG (heartbeat left in).

    python run_rbdtector.py --cohort cap                 # CAP RBD + same-system patients, ladder steps
    python run_rbdtector.py --cohort chal18 --sample 100 # MGH, as recorded (seeded sample)
    python run_rbdtector.py --cohort hmc                 # HMC, as recorded
    python run_rbdtector.py --cohort boas                # BOAS first nights, as recorded

-> results/rbdtector_scores.csv (or --out): one row per night, step and variant.
Re-running skips nights already scored; --mirror copies the file to Drive after
each night.
"""
import argparse
import shutil
import sys
import time
import traceback
import zlib
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config

LADDER_STEPS = ["native", "n050", "n100", "n200", "h150"]
OUT = config.RESULTS / "rbdtector_scores.csv"
MIN_REM = 10.0


def records_for(name, sample, seed=0):
    """The nights to score, taken from the external-cohort scores (same nights,
    same labels as the rest of the paper)."""
    ext = pd.read_csv(config.RESULTS / "external_scores.csv")
    ext = ext[(ext.method == "Montreal") & (ext.variant == "published")
              & (ext.rem_minutes >= MIN_REM) & ext.primary.astype(bool)]
    want = {"cap": ["CAP RBD", "CAP patients"], "chal18": ["MGH"], "hmc": ["HMC"],
            "boas": ["BOAS"]}[name]
    g = ext[ext.who.isin(want)].drop_duplicates("record")
    recs = sorted(g.record.astype(str))
    if sample and sample < len(recs):
        recs = sorted(np.random.default_rng(seed).choice(recs, sample, replace=False))
    return recs, dict(zip(g.record.astype(str), g.who))


def _work(name, rec, steps, scratch):
    """One night: download, the ladder's noise, RBDtector per step and variant."""
    import warnings
    warnings.filterwarnings("ignore")
    from src import rbdtector as rt
    from src.cohorts import COHORTS, extract_ladder_pack
    from src.extract import bandpass
    rt.install()
    cohort = COHORTS[name]
    folder = Path(scratch) / f"rbdt_{name}_{rec}"
    folder.mkdir(parents=True, exist_ok=True)
    try:
        t0 = time.time()
        d = cohort.read(rec, cohort.fetch(rec, folder))
        t_dl = time.time() - t0
        if not d["unit_known"]:
            raise ValueError("chin EMG has no physical units")
        fs, hyp = d["fs"], np.asarray(d["hypnogram"])
        xf, _ = bandpass(d["chin"], fs, mains=cohort.mains)
        added, level = {"native": 0.0}, {"native": np.nan}
        nf = None
        if steps != ["native"]:
            meta = extract_ladder_pack(cohort, rec, d)["meta"]
            level["native"] = meta["level_uv"]
            for s, v in meta["ladder"].items():
                added[s], level[s] = v["added_uv"], v["level_uv"]
            rng = np.random.default_rng(zlib.crc32(rec.encode()))
            nf, _ = bandpass(rng.normal(0.0, 1.0, len(d["chin"])), fs, mains=cohort.mains)
            edge = int(5 * fs)
            nf /= np.sqrt(np.mean(nf[edge:-edge] ** 2))
        del d
        rows = []
        for step in steps:
            if step not in added:
                continue
            x = xf if step == "native" else xf + added[step] * nf
            # RBDtector works at 256 Hz; it resamples by spline, which at 512 Hz
            # (every other sample) is plain decimation - done here to save time
            x, f = (x[::2], fs / 2) if fs == 512 else (x, fs)
            sub = rt.write_inputs(folder / step, x, f, hyp)
            for variant in ("published", "FC"):
                t1 = time.time()
                r = rt.run(sub, variant)
                rows.append({"cohort": name, "record": rec, "step": step, "variant": variant,
                             "added_uv": added[step], "level_uv": level[step], "fs": fs,
                             "seconds": round(time.time() - t1, 1), **r})
            shutil.rmtree(sub, ignore_errors=True)
        return rec, rows, None, round(t_dl, 1)
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=3)[-600:]}", 0.0
    finally:
        shutil.rmtree(folder, ignore_errors=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cohort", required=True, choices=["cap", "chal18", "hmc", "boas"])
    ap.add_argument("--sample", type=int, default=0, help="seeded random subset of nights")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--limit", type=int, default=0, help="stop after this many nights (test)")
    ap.add_argument("--mirror", default=None, help="copy the scores file here after each night")
    ap.add_argument("--out", default="rbdtector_scores.csv", help="scores file in results/ (to split work across runtimes)")
    ap.add_argument("--steps", default=None,
                    help="comma-separated ladder steps (CAP default: " + ",".join(LADDER_STEPS) + ")")
    args = ap.parse_args()
    global OUT
    OUT = config.RESULTS / args.out

    from src import rbdtector as rt
    rt.install()
    steps = args.steps.split(",") if args.steps else \
        (LADDER_STEPS if args.cohort == "cap" else ["native"])
    recs, who = records_for(args.cohort, args.sample)
    done = set()
    if OUT.exists():
        old = pd.read_csv(OUT, dtype={"record": str})
        old = old[old.cohort == args.cohort]
        done = set(zip(old.record, old.step))
    todo = {r: [s for s in steps if (r, s) not in done] for r in recs}
    todo = {r: s for r, s in todo.items() if s}
    n_done = len(recs) - len(todo)
    if args.limit:
        todo = dict(list(todo.items())[: args.limit])
    print(f"{args.cohort}: {len(recs)} nights, {n_done} fully scored, "
          f"{len(todo)} to score now at steps {steps} with {args.workers} workers", flush=True)
    t_start = time.time()
    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(_work, args.cohort, r, s, str(config.SCRATCH)) for r, s in todo.items()]
        for i, f in enumerate(as_completed(futs), 1):
            rec, rows, err, t_dl = f.result()
            if err:
                print(f"[{i}/{len(todo)}] {rec} FAILED {err}", flush=True)
                continue
            df = pd.DataFrame(rows).assign(who=who.get(rec, ""))
            df.to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
            if args.mirror:
                try:
                    Path(args.mirror).mkdir(parents=True, exist_ok=True)
                    shutil.copy2(OUT, Path(args.mirror) / OUT.name)
                except OSError as e:
                    print(f"  (mirror failed: {e})", flush=True)
            pub = df[(df.variant == "published") & (df.step == "native")]
            print(f"[{i}/{len(todo)}] {rec}  download {t_dl}s  RBDtector {df.seconds.mean():.0f}s/run  "
                  f"native any {pub.any_pct.iloc[0] if len(pub) else float('nan'):.1f}%  "
                  f"({(time.time() - t_start) / 60:.1f} min)", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_parkinsons.py
"""What the noise floor means for Parkinson's disease research.

RBD is the strongest known prodromal marker of Parkinson's disease, dementia with
Lewy bodies and multiple system atrophy, and REM sleep without atonia is how it is
confirmed; how much RWA a patient has is also used to predict who develops
Parkinson's disease and to follow progression, and RWA without RBD is studied as
an earlier stage still. This puts the recording system's effect in those terms,
against the predictions fixed in paper/preregistration_parkinsons.md:

  PD1  the same RBD nights as recorded and at HMC's floor: change in tonic, phasic
       and 'any' RWA (points), against the baseline difference that separated later
       Parkinson's disease from no disease (Postuma 2010) and five years of
       progression (Iranzo 2009)
  PD2  RBDtector, the open scorer used in Parkinson's and RBD research, run
       unchanged: on the noise ladder, across recording systems, and with the
       floor-corrected baseline
  PD3  isolated RWA by system: DREAMER's RAI < 0.85 and the other rules, in people
       without RBD (descriptive)
  PD4  recruiting for prevention trials: per 1,000 sleep studies in people aged 50
       or over (descriptive)
  PD5  the floor check for each recording system (descriptive)

    python run_parkinsons.py -> results/parkinsons_report.md, figures/fig10_parkinsons
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.floorcheck import MAX_FLOOR, MIN_FS, thresholds
from src.rwa import NOISE_P5_RATIO
from run_floor import pct, sel
from run_report import MIN_REM, paired_change

PAIRED = "h150"                    # HMC's floor (1.5 uV), the ladder step used throughout
POSTUMA_PD = 72.9 - 41.0           # baseline tonic %: later Parkinson's vs disease-free
POSTUMA_ANY = 62.7 - 41.0          # ... any neurodegenerative disease vs disease-free
IRANZO_5Y = 54.0 - 30.0            # tonic %: rise over a mean 5 years of follow-up
IRANZO_YEAR = IRANZO_5Y / 5.0
CONVERT_12Y = 0.735                # isolated RBD -> synucleinopathy within 12 years
RBDT_CUT = 23.5                    # RBDtector chin 'any' cut-off (Rothenbacher 2022)
FC_CUT = 49.1                      # SINBAR-FC 'any' cut-off re-derived on CAP (FC5)
DREAMER_RAI = 0.85
PREVALENCE = (0.01, 0.02, 0.05)
WITHOUT_RBD = ["CAP patients", "MGH", "HMC", "DOD-O", "BOAS", "DOD-H"]


def boot_ci(x, n=2000, seed=0, stat=np.mean):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    b = [stat(rng.choice(x, len(x))) for _ in range(n)]
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def paired(df, metric, steps=("native", PAIRED)):
    """record x step table of a metric, complete pairs only."""
    w = df.pivot_table(index="record", columns="step", values=metric, aggfunc="first")
    if not set(steps) <= set(w):
        return w.iloc[:0]
    return w[list(steps)].dropna()


def load():
    R = config.RESULTS
    ext = pd.read_csv(R / "external_scores.csv", low_memory=False)
    ext = ext[(ext.rem_minutes >= MIN_REM) & (ext.who.fillna("") != "") & ext.primary.astype(bool)]
    lad = pd.read_csv(R / "floor_ladder_scores.csv", low_memory=False)
    lad = lad[lad.rem_minutes >= MIN_REM]
    parts = sorted(R.glob("rbdtector_scores*.csv"))        # one file per runtime used
    rb = pd.concat([pd.read_csv(f, dtype={"record": str}) for f in parts], ignore_index=True) \
        if parts else pd.DataFrame()
    if len(rb):
        rb = rb.drop_duplicates(["cohort", "record", "step", "variant"], keep="last")
        if rb.rem_miniepochs.notna().any():                     # 20 mini-epochs per minute
            rb = rb[rb.rem_miniepochs.fillna(0) >= MIN_REM * 20]
        floor = ext[(ext.method == "Montreal") & (ext.variant == "published")] \
            .drop_duplicates(["cohort", "record"]).set_index(["cohort", "record"])
        key = pd.MultiIndex.from_frame(rb[["cohort", "record"]].astype(str))
        floor.index = pd.MultiIndex.from_arrays([floor.index.get_level_values(0).astype(str),
                                                 floor.index.get_level_values(1).astype(str)])
        rb["level_1s"] = floor.level_1s.reindex(key).to_numpy()
        rb["age"] = pd.to_numeric(floor.age.reindex(key), errors="coerce").to_numpy()
        rb["who"] = floor.who.reindex(key).to_numpy()
        rb["y"] = (rb.any_pct >= RBDT_CUT).astype(float).where(rb.any_pct.notna())
    return ext, lad, rb


def main():
    ext, lad, rb = load()
    lines, verdict = [], {}
    say = lambda s="": (print(s), lines.append(s))
    sources = []                                          # (label, frame, method, metric, variant)
    for variant in ("published", "FC"):
        sources += [(f"Montreal tonic, {variant}", lambda d, v=variant: sel(d, "Montreal", v), "tonic_pct", variant),
                    (f"Montreal phasic, {variant}", lambda d, v=variant: sel(d, "Montreal", v), "phasic_pct", variant),
                    (f"SINBAR any, {variant}", lambda d, v=variant: sel(d, "SINBAR", v), "any_pct", variant)]
        if len(rb):
            sources += [(f"RBDtector {m.split('_')[0]}, {variant}", lambda d, v=variant: d[d.variant == v], m, variant)
                        for m in ("tonic_pct", "phasic_pct", "any_pct")]

    # PD1 ────────────────────────────────────────────────────────────────────
    say(f"PD1 The same nights as recorded and at HMC's floor ({PAIRED}): change in RWA, points")
    say(f"    benchmarks (tonic %): later Parkinson's vs disease-free at baseline {POSTUMA_PD:.1f}")
    say(f"    (Postuma 2010; any neurodegeneration {POSTUMA_ANY:.1f}); rise over 5 years {IRANZO_5Y:.0f}")
    say(f"    = {IRANZO_YEAR:.1f} per year (Iranzo 2009)")
    pd1 = {}
    for who in ("CAP RBD", "CAP patients"):
        say(f"  {who}")
        for label, pick, metric, variant in sources:
            src = rb if label.startswith("RBDtector") else lad
            w = paired(pick(src[src.who == who]), metric)
            if w.empty:
                continue
            d = w[PAIRED] - w["native"]
            lo, hi = boot_ci(d)
            pd1[who, label] = (d.mean(), lo, hi, len(d))
            extra = ""
            if "tonic" in label:
                extra = (f"   = {100 * abs(d.mean()) / POSTUMA_PD:3.0f}% of the Parkinson's difference,"
                         f" {abs(d.mean()) / IRANZO_YEAR:4.1f} years of progression")
            say(f"    {label:<26} n={len(d):<3} {w['native'].mean():5.1f} -> {w[PAIRED].mean():5.1f}   "
                f"change {d.mean():+6.1f} [{lo:+.1f} to {hi:+.1f}]  median {d.median():+5.1f} "
                f"[{d.quantile(.25):+.1f} to {d.quantile(.75):+.1f}]" + extra)
    get = lambda k: pd1.get(("CAP RBD", k), (np.nan,) * 4)[0]
    verdict["PD1a Montreal tonic published <= -10"] = get("Montreal tonic, published") <= -10
    verdict["PD1b Montreal tonic FC within 3"] = abs(get("Montreal tonic, FC")) <= 3
    if len(rb):
        verdict["PD1c RBDtector tonic published <= -10"] = get("RBDtector tonic, published") <= -10
        verdict["PD1d RBDtector tonic FC within 3"] = abs(get("RBDtector tonic, FC")) <= 3
    say("  Dose-response, CAP RBD, mean tonic % per ladder step (Montreal; RBDtector where scored):")
    steps = [s for s in ("native", "n050", "n100", "n150", "n200", "n300", "h150") if s in set(lad.step)]
    for label, pick, metric, variant in sources:
        if "tonic" not in label:
            continue
        src = rb if label.startswith("RBDtector") else lad
        g = pick(src[src.who == "CAP RBD"])
        say(f"    {label:<26} " + "  ".join(f"{s} {g[g.step == s][metric].mean():5.1f}"
                                            for s in steps if (g.step == s).any()))
    say()

    # PD2 ────────────────────────────────────────────────────────────────────
    if len(rb):
        say(f"PD2 RBDtector (run unchanged; chin 'any' >= {RBDT_CUT}% = positive), published baseline")
        say("    and floor-corrected baseline (B = 1.0 uV)")
        say("  a) The noise ladder, % flagged [95% CI], same nights")
        lv = sel(lad, "Montreal")
        for who in ("CAP RBD", "CAP patients"):
            g = rb[rb.who == who]
            st = [s for s in steps if (g.step == s).any()]
            level = [lv[(lv.who == who) & (lv.step == s)].background_phasic_uV.median() for s in st]
            say(f"    {who} (n={g[g.step == 'native'].record.nunique()})   quiet-REM level "
                + "  ".join(f"{s} {v:.2f}" for s, v in zip(st, level)))
            for variant in ("published", "FC"):
                h = g[g.variant == variant]
                say(f"      {variant:<10} " + "  ".join(f"{s} {100 * h[h.step == s].y.mean():5.1f}" for s in st))
                w = paired(h, "y")
                if len(w):
                    ch = 100 * (w[PAIRED].mean() - w["native"].mean())
                    say(f"      {'':<10} native -> {PAIRED}: " + paired_change(w["native"], w[PAIRED]))
                    if who == "CAP patients":
                        if variant == "published":
                            verdict["PD2a patients published fall >= 10"] = ch <= -10
                        else:
                            verdict["PD2b patients FC within 5"] = abs(ch) <= 5
                    elif variant == "FC":
                        verdict["PD2c RBD FC within 10"] = abs(ch) <= 10
        say("  b) As recorded, by system: % flagged [95% CI]; median floor")
        nat = rb[rb.step == "native"]
        share = {}
        for who in ["CAP RBD", "CAP patients", "MGH", "HMC", "BOAS"]:
            g = nat[nat.who == who]
            if g.empty:
                continue
            cells = []
            for variant in ("published", "FC"):
                p, lo, hi, n = pct(g[g.variant == variant].y)
                share[who, variant] = p
                cells.append(f"{variant} {100 * p:5.1f} [{100 * lo:.0f}-{100 * hi:.0f}]")
            say(f"    {who:<13} n={g.record.nunique():<4} floor {g.level_1s.median():.2f} uV   " + "   ".join(cells))
        if {("MGH", "published"), ("HMC", "published")} <= set(share):
            gp = share["MGH", "published"] - share["HMC", "published"]
            gf = share["MGH", "FC"] - share["HMC", "FC"]
            say(f"    MGH - HMC: published {100 * gp:+.1f}, floor-corrected {100 * gf:+.1f} points")
            verdict["PD2d MGH-HMC published >= 20 and FC halves it"] = gp >= 0.20 and abs(gf) <= abs(gp) / 2
        if ("BOAS", "published") in share:
            verdict["PD2e BOAS published >= 50% and FC halves it"] = \
                share["BOAS", "published"] >= 0.5 and share["BOAS", "FC"] <= share["BOAS", "published"] / 2
        say("  c) Nights without RBD, pooled: Spearman rho of the night's floor with RBDtector 'any' %")
        from scipy.stats import spearmanr
        g = nat[nat.who.isin(WITHOUT_RBD)]
        rho = {}
        for variant in ("published", "FC"):
            h = g[g.variant == variant][["level_1s", "any_pct"]].dropna()
            r = spearmanr(h.level_1s, h.any_pct)[0]
            rng = np.random.default_rng(0)
            bs = [spearmanr(*h.sample(len(h), replace=True, random_state=rng.integers(1 << 31)).to_numpy().T)[0]
                  for _ in range(1000)]
            rho[variant] = (r, *np.percentile(bs, [2.5, 97.5]))
            say(f"    {variant:<10} rho {r:+.2f} [{rho[variant][1]:+.2f} to {rho[variant][2]:+.2f}]  (n={len(h)})")
        verdict["PD2f published rho < 0 (CI) and FC weaker"] = rho["published"][2] < 0 and \
            abs(rho["FC"][0]) < abs(rho["published"][0])
        say("  d) RBDtector against the paper's SINBAR 'any' (published), same nights as recorded:")
        s = sel(ext, "SINBAR")[["cohort", "record", "any_pct"]].astype({"record": str})
        m = nat[nat.variant == "published"].merge(s, on=["cohort", "record"], suffixes=("", "_sinbar"))
        if len(m) > 3:
            say(f"    Spearman rho {spearmanr(m.any_pct, m.any_pct_sinbar)[0]:+.2f} (n={len(m)});  "
                f"RBDtector baseline / quiet-REM floor, median {np.nanmedian(m.baseline_uv / m.level_1s):.2f}")
        say()

    # PD3 ────────────────────────────────────────────────────────────────────
    say("PD3 Isolated RWA (RWA without RBD) by system, % of nights labeled (descriptive)")
    say(f"    DREAMER: RAI < {DREAMER_RAI}; RAI < 0.8; SINBAR and Montreal published / floor-corrected;")
    say(f"    RBDtector published / floor-corrected where scored")
    rai = ext[ext.method == "RAI"].drop_duplicates(["cohort", "record"]).set_index(["cohort", "record"]).rai
    groups = []
    for who in WITHOUT_RBD:
        g = ext[ext.who == who]
        age = pd.to_numeric(g.age, errors="coerce")
        groups.append((who, g))
        if age.notna().sum() >= 20:
            groups.append((f"{who} >= 50 y", g[age >= 50]))
    pd3 = {}
    for name, g in groups:
        base = g[(g.method == "Montreal") & (g.variant == "published")]
        if base.empty:
            continue
        r = rai.reindex(pd.MultiIndex.from_frame(base[["cohort", "record"]])).to_numpy(float)
        r = r[np.isfinite(r)]
        cells = {"RAI<0.85": np.mean(r < DREAMER_RAI) if len(r) else np.nan,
                 "RAI<0.8": np.mean(r < 0.8) if len(r) else np.nan}
        for meth in ("SINBAR", "Montreal"):
            for variant in ("published", "FC"):
                cells[f"{meth} {variant[:3]}"] = sel(g, meth, variant).y.mean()
        if len(rb):
            h = rb[(rb.step == "native") & rb.record.isin(set(base.record.astype(str)))
                   & rb.cohort.isin(set(base.cohort))]
            for variant in ("published", "FC"):
                v = h[h.variant == variant].y
                cells[f"RBDtector {variant[:3]}"] = v.mean() if len(v) else np.nan
        pd3[name] = cells
        say(f"  {name:<20} n={len(base):<4} floor {base.level_1s.median():.2f}  " + "  ".join(
            f"{k} {100 * v:4.0f}" if np.isfinite(v) else f"{k}    -" for k, v in cells.items()))
    say("  RAI < 0.85 on the noise ladder (same nights), %:")
    rl = lad[(lad.method == "RAI")]
    for who in ("CAP patients", "MGH", "HMC", "CAP RBD"):
        g = rl[rl.who == who]
        if g.empty:
            continue
        say(f"    {who:<13} " + "  ".join(f"{s} {100 * (g[g.step == s].rai < DREAMER_RAI).mean():4.0f}"
                                         for s in steps if (g.step == s).any()))
    say()

    # PD4 ────────────────────────────────────────────────────────────────────
    say("PD4 Per 1,000 sleep studies in people aged 50 or over (descriptive): flagged, true RBD")
    say("    found, positive predictive value, studies to review per RBD found, and people expected")
    say(f"    to develop a synucleinopathy within 12 years ({100 * CONVERT_12Y:.1f}%, Postuma 2019).")
    say("    False-positive rate: share flagged among MGH patients >= 50 y (an upper bound: a few")
    say("    may have RBD); sensitivity: CAP RBD.")
    mgh = ext[ext.who == "MGH"]
    old = mgh[pd.to_numeric(mgh.age, errors="coerce") >= 50]
    rbd = ext[ext.who == "CAP RBD"]
    rules = []
    for meth in ("Montreal", "SINBAR"):
        for variant in ("published", "FC"):
            rules.append((f"{meth} {variant}", sel(old, meth, variant).y, sel(rbd, meth, variant).y))
    fa, ra = sel(old, "SINBAR", "FC").any_pct, sel(rbd, "SINBAR", "FC").any_pct
    rules.append((f"SINBAR-FC >= {FC_CUT}%", (fa >= FC_CUT).astype(float).where(fa.notna()),
                  (ra >= FC_CUT).astype(float).where(ra.notna())))
    rules.append(("RAI < 0.8", sel(old, "RAI").y, sel(rbd, "RAI").y))
    if len(rb):
        for variant in ("published", "FC"):
            h = rb[(rb.step == "native") & (rb.variant == variant)]
            rules.append((f"RBDtector {variant}", h[(h.who == "MGH") & (h.age >= 50)].y,
                          h[h.who == "CAP RBD"].y))
    say(f"    MGH >= 50 y: n={old.record.nunique()};  CAP RBD: n={rbd.record.nunique()}")
    pd4 = []
    for name, fp_y, se_y in rules:
        fp, se = np.nanmean(fp_y), np.nanmean(se_y)
        for p in PREVALENCE:
            tp, f = 1000 * p * se, 1000 * (1 - p) * fp
            pd4.append(dict(rule=name, prevalence=p, fp=fp, se=se, flagged=tp + f, found=tp,
                            ppv=tp / (tp + f) if tp + f else np.nan, per_case=(tp + f) / tp if tp else np.nan,
                            convert=tp * CONVERT_12Y, n_fp=int(np.isfinite(fp_y).sum())))
        say(f"  {name:<20} FP {100 * fp:4.1f}% (n={int(np.isfinite(fp_y).sum())})  sens {100 * se:4.0f}%   " + "   ".join(
            f"{100 * p:.0f}%: flag {r['flagged']:4.0f}, find {r['found']:4.1f}, PPV {100 * r['ppv']:4.1f}%, "
            f"{r['per_case']:4.0f}/case, {r['convert']:4.1f} convert"
            for p, r in zip(PREVALENCE, pd4[-len(PREVALENCE):])))
    say("  The same at 2% prevalence with HMC's false-positive rate (all ages; HMC gives none):")
    hmc = ext[ext.who == "HMC"]
    for meth in ("Montreal", "SINBAR"):
        for variant in ("published", "FC"):
            fp, se = sel(hmc, meth, variant).y.mean(), sel(rbd, meth, variant).y.mean()
            tp, f = 20 * se, 980 * fp
            say(f"    {meth + ' ' + variant:<20} flag {tp + f:4.0f}, find {tp:4.1f}, PPV {100 * tp / (tp + f):4.1f}%")
    say()

    # PD5 ────────────────────────────────────────────────────────────────────
    say("PD5 Floor check by recording system (descriptive): floor b (5th percentile of 1-s REM RMS),")
    say(f"    nights inside the floor-corrected rule's range (>= {MIN_FS:g} Hz and b <= {MAX_FLOOR:.2f} uV),")
    say("    net N2/N3 tone (gain check), and the muscle activity 2 x b requires on top of the noise")
    one = ext[(ext.method == "Montreal") & (ext.variant == "published")]
    pd5 = {}
    for who in ["CAP RBD"] + WITHOUT_RBD:
        g = one[one.who == who]
        if g.empty:
            continue
        b = g.level_1s.median()
        t = thresholds(b)["SINBAR any (2 x b)"]
        inside = ((g.chin_fs >= MIN_FS) & (g.level_1s <= MAX_FLOOR)).mean()
        pd5[who] = dict(fs=g.chin_fs.median(), floor=b, inside=inside, net=g.nrem_p50_net.median(),
                        needs_pub=t["published_needs_uv"], needs_fc=t["fc_needs_uv"])
        say(f"  {who:<13} n={len(g):<4} fs {g.chin_fs.median():5g}  floor {b:.2f} "
            f"[{g.level_1s.quantile(.25):.2f}-{g.level_1s.quantile(.75):.2f}] uV  inside {100 * inside:3.0f}%  "
            f"net N2/N3 {g.nrem_p50_net.median():.2f} uV  2 x b needs {t['published_needs_uv']:.2f} uV "
            f"(floor-corrected {t['fc_needs_uv']:.2f})")
    say()

    say("Pre-specified predictions: " + "  ".join(f"{k}: {'met' if v else 'not met'}"
                                                 for k, v in verdict.items()))
    out = config.RESULTS / "parkinsons_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    pd.DataFrame(pd4).to_csv(config.RESULTS / "parkinsons_recruitment.csv", index=False)
    print(f"\nwritten -> {out}")
    figure(pd1, lad, rb, steps)


def figure(pd1, lad, rb, steps):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import save

    fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.0), layout="constrained",
                             gridspec_kw={"width_ratios": [1.45, 1, 1]})
    # A: recording vs disease, points of tonic RWA
    ax = axes[0]
    bars = [("Later Parkinson's vs no disease\n(baseline; Postuma 2010)", POSTUMA_PD, None, "#7f7f7f"),
            ("5-year rise in RBD\n(Iranzo 2009)", IRANZO_5Y, None, "#7f7f7f")]
    for label, key, col in (("Montreal, published", "Montreal tonic, published", "#d62728"),
                            ("Montreal, floor-corrected", "Montreal tonic, FC", "#1f77b4"),
                            ("RBDtector, published", "RBDtector tonic, published", "#ff7f0e"),
                            ("RBDtector, floor-corrected", "RBDtector tonic, FC", "#17becf")):
        if ("CAP RBD", key) in pd1:
            m, lo, hi, n = pd1["CAP RBD", key]
            a, b = sorted((abs(lo), abs(hi)))
            if np.sign(lo) != np.sign(hi):           # interval includes no change
                a = 0.0
            bars.append((label, abs(m), (abs(m) - a, b - abs(m)), col))
    n_rec = len(bars) - 2                            # literature bars on top, then a gap
    y = np.r_[n_rec + 1.6, n_rec + 0.6, np.arange(n_rec)[::-1]]
    for yi, (label, v, err, col) in zip(y, bars):
        ax.barh(yi, v, color=col, height=0.6)
        if err is not None:
            ax.errorbar(v, yi, xerr=[[err[0]], [err[1]]], fmt="none", ecolor="k", lw=0.8, capsize=2)
    ax.set_yticks(y, [b[0] for b in bars], fontsize=6)
    if n_rec:
        ax.text(0.98, n_rec - 0.2, "same RBD nights at HMC's floor (1.5 µV):",
                transform=ax.get_yaxis_transform(), fontsize=5.5, ha="right", va="center", style="italic")
    ax.set_ylim(-0.6, n_rec + 2.0)
    ax.set_xlabel("tonic RWA, percentage points")
    ax.set_title("A  Disease vs recording", fontsize=8)
    # B: RBDtector on the noise ladder
    ax = axes[1]
    if len(rb):
        lv = sel(lad, "Montreal")
        for who, col in (("CAP RBD", "#d62728"), ("CAP patients", "#7f7f7f")):
            g = rb[rb.who == who]
            st = [s for s in steps if s != "h150" and (g.step == s).any()]
            x = [lv[(lv.who == who) & (lv.step == s)].background_phasic_uV.median() for s in st]
            for variant, ls in (("published", ":"), ("FC", "-")):
                h = g[g.variant == variant]
                ax.plot(x, [100 * h[h.step == s].y.mean() for s in st], ls, marker="o", ms=2.5, lw=1,
                        color=col, label=who if variant == "FC" else None)
        ax.plot([], [], ":", color="k", lw=1, label="published")
        ax.plot([], [], "-", color="k", lw=1, label="floor-corrected")
        ax.set_xscale("log")
        ax.set_xticks([0.5, 1, 2], ["0.5", "1", "2"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.legend(frameon=False, fontsize=5.5, loc="lower left")
    ax.set_ylim(0, 100)
    ax.set_xlabel("quiet-REM level (µV)")
    ax.set_ylabel("nights flagged by RBDtector (%)")
    ax.set_title("B  RBDtector + noise", fontsize=8)
    # C: RBDtector across systems
    ax = axes[2]
    if len(rb):
        nat = rb[rb.step == "native"]
        style = {"CAP RBD": ("#d62728", "*"), "CAP patients": ("#7f7f7f", "o"), "MGH": ("#1f77b4", "o"),
                 "HMC": ("#9467bd", "o"), "BOAS": ("#2ca02c", "s")}
        for who, (col, mk) in style.items():
            g = nat[nat.who == who]
            if g.empty:
                continue
            x = g.level_1s.median()
            p, f = (100 * g[g.variant == v].y.mean() for v in ("published", "FC"))
            ax.annotate("", xy=(x, f), xytext=(x, p), arrowprops=dict(arrowstyle="->", color=col, lw=0.8))
            ax.plot(x, p, mk, color=col, mfc="white", ms=5)
            ax.plot(x, f, mk, color=col, ms=5, label=who)
        ax.set_xscale("log")
        ax.set_xticks([0.3, 0.5, 1, 2], ["0.3", "0.5", "1", "2"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.legend(frameon=False, fontsize=5.5, loc="lower left")
    ax.set_ylim(0, 100)
    ax.set_xlabel("quiet-REM level (µV)")
    ax.set_title("C  RBDtector by system", fontsize=8)
    save(fig, "fig10_parkinsons")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_rbdtector_noise.py
"""What RBDtector counts as chin activity in noise alone (exploratory).

Not pre-specified: added after prediction PD2b was missed. Under the
floor-corrected baseline, RBDtector flagged 11.8 points more of CAP's same-system
patients at HMC's floor (1.5 uV) than as recorded, where 5 had been allowed. At
that floor the floor-corrected threshold is only 1.5 times the noise RMS, and
RBDtector judges 30-ms chunks - about 8 samples at 256 Hz, whose RMS in noise
alone varies far more than that of the 1-s windows the guard was chosen for.

This scores nights of band-limited white noise and nothing else (no muscle, no
heartbeat) at quiet-REM levels from 0.5 to 2 uV, with RBDtector's published
baseline and with the floor-corrected one at the pre-specified guard (1.5) and at
2.0. Whatever it reports is noise counted as muscle activity.

    python run_rbdtector_noise.py   -> results/rbdtector_noise.csv
"""
import shutil
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config

LEVELS = (0.5, 1.0, 1.5, 2.0)      # noise RMS, uV (CAP patients 0.48, HMC 1.52)
SEEDS = (0, 1, 2)
FS = 256
# Wake, then four cycles of 25 min N2 and 20 min REM (80 min of REM), then N2
HYPNOGRAM = np.r_[[0] * 10, ([2] * 50 + [4] * 40) * 4, [2] * 10]
RULES = (("published", "published", None), ("FC", "FC", 1.5), ("FC, guard 2.0", "FC", 2.0))


def _night(level, seed):
    import warnings
    warnings.filterwarnings("ignore")
    from src import rbdtector as rt
    from src.extract import bandpass
    rt.install()
    rng = np.random.default_rng(1000 + seed)
    x, _ = bandpass(rng.normal(0.0, 1.0, len(HYPNOGRAM) * 30 * FS), FS, mains=50.0)
    edge = 5 * FS
    x *= level / np.sqrt(np.mean(x[edge:-edge] ** 2))
    folder = config.SCRATCH / f"rbdt_noise_{level}_{seed}"
    rows = []
    try:
        sub = rt.write_inputs(folder, x, FS, HYPNOGRAM)
        for label, variant, guard in RULES:
            rt.GUARD = guard or 1.5
            t0 = time.time()
            r = rt.run(sub, variant)
            r.pop("keys", None)
            rows.append({"level_uv": level, "seed": seed, "rule": label,
                         "seconds": round(time.time() - t0, 1), **r})
    finally:
        rt.GUARD = 1.5
        shutil.rmtree(folder, ignore_errors=True)
    return rows


def main():
    with ProcessPoolExecutor(max_workers=2) as ex:
        futs = [ex.submit(_night, lv, s) for lv in LEVELS for s in SEEDS]
        rows = [r for f in futs for r in f.result()]
    df = pd.DataFrame(rows)
    out = config.RESULTS / "rbdtector_noise.csv"
    df.to_csv(out, index=False)
    pd.set_option("display.width", 200)
    t = df.groupby(["rule", "level_uv"])[["any_pct", "phasic_pct", "tonic_pct", "baseline_uv"]]
    print("Noise alone, mean of", len(SEEDS), "nights (% of REM mini-epochs or epochs):")
    print(t.mean().round(2).to_string())
    print("max any_pct per rule and level:")
    print(df.groupby(["rule", "level_uv"]).any_pct.max().round(2).to_string())
    print(f"written -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/check_floor.py
"""Floor check for one recording (see src/floorcheck.py).

    python check_floor.py night.edf stages.txt                 # chin found by name
    python check_floor.py night.edf stages.txt --chin "Chin1-Chin2" --mains 60
    python check_floor.py night.edf stages.txt --rem 5         # numeric stage file

stages.txt: one 30 s stage per line from the start of the EDF (W, N1, N2, N3, R),
or numbers with --rem (and --nrem) giving the codes for REM and N2/N3.
"""
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src.ecg import find_ecg
from src.edf import read_header, read_channel
from src.extract import find_emg, read_source
from src.floorcheck import floor_report, format_report, read_stages


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("edf")
    ap.add_argument("stages")
    ap.add_argument("--chin", default=None, help="chin EMG channel label (default: found by name)")
    ap.add_argument("--mains", type=float, default=50.0, help="mains frequency to notch (50 or 60)")
    ap.add_argument("--rem", type=int, default=None, help="REM code in a numeric stage file")
    ap.add_argument("--nrem", default="2,3", help="N2/N3 codes in a numeric stage file")
    ap.add_argument("--json", action="store_true")
    args = ap.parse_args()

    hdr = read_header(args.edf)
    src = hdr.find(args.chin) if args.chin else find_emg(hdr)[0]
    if src is None:
        sys.exit(f"no chin EMG found among {[c.label for c in hdr.channels]}; use --chin")
    chin, fs, unit_known, label, _ = read_source(hdr, src)
    if not unit_known:
        sys.exit(f"{label}: no physical unit in the EDF header - the floor needs microvolts")
    ecg_src = find_ecg(hdr)
    ecg, fs_e = read_source(hdr, ecg_src)[:2] if ecg_src is not None else (None, None)
    stages = read_stages(args.stages, args.rem, tuple(int(v) for v in args.nrem.split(",")))
    r = floor_report(chin, fs, stages, ecg, fs_e, args.mains)
    print(json.dumps(r, indent=1) if args.json else f"{Path(args.edf).name}, {label}\n" + format_report(r))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_parkinsons_colab.sh
#!/bin/sh
# The Parkinson's analyses (paper/preregistration_parkinsons.md) on a fresh Colab
# runtime, reusing the scores already on Drive; everything new is copied back to
# Drive as it is made. Run from the code directory, e.g. in Colab's terminal:
#   cd /content/rbd-rwa && setsid nohup sh run_parkinsons_colab.sh > /content/pd.log 2>&1 < /dev/null &
# RBDtector needs about 75 s per night and step (both rules) on Colab's 2-core
# runtime, so the work can be split over two runtimes: PART=1 (the CAP ladder, then
# the analysis once part 2 is on Drive) and PART=2 (MGH, HMC and BOAS).
set -e
PART=${PART:-all}
D=/content/drive/MyDrive/rbd-rwa
export RBD_DURABLE_DIR=/content/durable PYTHONUNBUFFERED=1
R=$RBD_DURABLE_DIR/results
F=$RBD_DURABLE_DIR/figures
mkdir -p "$R" "$F" "$D/results" "$D/figures"
for f in external_scores.csv floor_ladder_scores.csv floor_scores.csv ladder_scores.csv; do
  if [ -f "$D/results/$f" ] && [ ! -f "$R/$f" ]; then cp "$D/results/$f" "$R/"; fi
done
O=rbdtector_scores.csv
if [ "$PART" = 2 ]; then O=rbdtector_scores_systems.csv; fi
if [ -f "$D/results/$O" ] && [ ! -f "$R/$O" ]; then cp "$D/results/$O" "$R/"; fi
# RBDtector runs in the environment its requirements.txt specifies (2021 versions;
# pandas 2 removed DataFrame.append, which its output writer uses)
E=/content/rbdt_env
if [ ! -x "$E/bin/python" ]; then
  pip -q install uv
  uv venv -q -p 3.9 "$E"
  uv pip install -q -p "$E/bin/python" "numpy~=1.21.2" "pandas~=1.3.3" "scipy~=1.7.1" \
    "openpyxl~=3.0.8" "pyEDFlib~=0.1.22" "wfdb~=3.4.0"
fi
RB="$E/bin/python run_rbdtector.py --out $O --mirror $D/results"
if [ "$PART" != 2 ]; then
  $RB --cohort cap
  # exploratory, added after PD2b was missed: RBDtector on noise alone (~10 minutes)
  if [ ! -f "$R/rbdtector_noise.csv" ]; then $E/bin/python run_rbdtector_noise.py; fi
  cp "$R/rbdtector_noise.csv" "$D/results/"
fi
if [ "$PART" != 1 ]; then
  $RB --cohort chal18 --sample 100
  $RB --cohort hmc
  $RB --cohort boas
fi
if [ "$PART" = 2 ]; then touch "$D/results/rbdtector_part2.done"; echo PART2_DONE; exit 0; fi
if [ "$PART" = 1 ]; then
  while [ ! -f "$D/results/rbdtector_part2.done" ]; do sleep 60; done
  cp "$D/results/rbdtector_scores_systems.csv" "$R/"
fi
python run_parkinsons.py
python run_ladder.py --from-csv
cp "$R"/parkinsons_* "$R"/ladder_report.md "$D/results/"
cp "$F"/fig10_* "$F"/fig6_* "$F"/fig7_* "$D/figures/"
echo PD_DONE

In [ ]:
%%writefile /content/rbd-rwa/src/inject.py
"""Known muscle activity added to real nights: ground truth for RWA scoring.

The open RBD recordings are 21 nights from one center on one quiet recording
system, so what they show about detection could be a property of those
patients. Here the activity is known instead. It is added to the REM sleep of
nights recorded without RBD, on every system in the study
(paper/preregistration_injection.md):

  synthetic   tonic episodes (whole 30-s epochs) or phasic bursts (0.25-2 s):
              Gaussian noise shaped by an activity envelope and passed through
              the night's own EMG filter, at 49 amplitudes from 0.25 to 16 uV in
              steps of 9% (RMS in the night's band, during activity)
  transplant  the REM chin EMG of one of CAP's RBD patients, filtered and
              heartbeat-cleaned as in the main pipeline, resampled to the
              night's rate and passed through its filter, at full and half
              amplitude

Both are added after the night's own filtering and heartbeat removal, as the
noise ladder adds its noise (cohorts.extract_ladder_pack). What the added
activity has to be found in is therefore the night's real background: its
noise, colored and varying through the night, its artifacts and its own
twitches.

Every copy is scored with the published and floor-corrected rules unchanged
(rwa.py), and their per-unit decisions are compared with where activity was
added: a unit (tonic epoch, phasic or SINBAR mini-epoch) is "injected" when the
added activity covers most of it and "clean" when there is none in it (for
mini-epochs, none within 0.1 s of it either).

The amplitude steps are fine because detection of a 20- or 30-s tonic epoch is
nearly a step function of amplitude (each decision rests on 1,000-1,500 bins).
"""
import json
import zlib
from fractions import Fraction

import numpy as np

from .config import ENV_FS
from .extract import bandpass, bin_envelope
from .pack import build_pack
from .rwa import METHODS, _mini, floor_corrected, rem_atonia_index, score_night

EPOCH_SEC = 30
REM_CODE = 4
AMPLITUDES_UV = tuple(round(0.25 * 2 ** (i / 8), 4) for i in range(49))   # 0.25 ... 16
DONOR_SCALES = (1.0, 0.5)
BURST_S = (0.25, 0.5, 1.0, 2.0)  # phasic burst durations, used in turn
RAMP_S = 0.025                   # raised-cosine onset and offset of every episode
TONIC_COVER = 0.75               # tonic unit injected: activity covers >= 75% of it
INTERIOR_MIN_S = 0.1             # mini-epoch injected: >= 0.1 s of full activity
CLEAN_MARGIN_S = 0.1             # clean: no activity in the unit (mini-epochs: or within 0.1 s)
REF_UV = 1.0                     # floor-corrected reference background (primary)
COUNTS = ("ni", "ki", "nc", "kc")  # injected units, of them detected; clean, detected


def variants():
    """(method name, variant, Method) scored on every copy of a night."""
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out += [(m.name, "published", m), (m.name, "FC", floor_corrected(m, REF_UV))]
    out.append(("AASM", "published", METHODS["aasm"]))
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  WHERE ACTIVITY IS ADDED
# ══════════════════════════════════════════════════════════════════════════════

def rem_stretches(hyp):
    """(first, last + 1) epoch numbers of each run of consecutive REM epochs."""
    rem = np.flatnonzero(np.asarray(hyp) == REM_CODE)
    if len(rem) == 0:
        return []
    brk = np.flatnonzero(np.diff(rem) != 1) + 1
    starts = rem[np.r_[0, brk]]
    stops = rem[np.r_[brk - 1, len(rem) - 1]] + 1
    return list(zip(starts.tolist(), stops.tolist()))


def _episode(g, a, b, fs):
    """g[a:b] = 1 with raised-cosine onset and offset of RAMP_S."""
    g[a:b] = 1.0
    r = min(int(round(RAMP_S * fs)), (b - a) // 2)
    if r > 0:
        up = 0.5 - 0.5 * np.cos(np.pi * (np.arange(r) + 0.5) / r)
        g[a:a + r] = up
        g[b - r:b] = up[::-1]


def tonic_envelope(hyp, fs, rng, n_samples):
    """Tonic episodes of 1-4 whole REM epochs separated by 2-6 epochs without,
    inside each run of REM epochs (never across two): about 40% of REM."""
    per = int(round(fs * EPOCH_SEC))
    g = np.zeros(n_samples, np.float32)
    for a, b in rem_stretches(hyp):
        e = a + int(rng.integers(0, 4))
        while e < b:
            stop = min(e + int(rng.integers(1, 5)), b)
            _episode(g, e * per, stop * per, fs)
            e = stop + int(rng.integers(2, 7))
    return g


def phasic_envelope(hyp, fs, rng, n_samples):
    """Phasic bursts of 0.25, 0.5, 1 and 2 s (in turn), 4-14 s apart, inside
    runs of REM epochs and at least 0.5 s from their ends."""
    g = np.zeros(n_samples, np.float32)
    k = 0
    for a, b in rem_stretches(hyp):
        t, end = a * EPOCH_SEC + rng.uniform(1.0, 6.0), b * EPOCH_SEC - 0.5
        while t + BURST_S[k % len(BURST_S)] <= end:
            d = BURST_S[k % len(BURST_S)]
            _episode(g, int(round(t * fs)), int(round((t + d) * fs)), fs)
            k += 1
            t += d + rng.uniform(4.0, 14.0)
    return g


def activity(g, fs, mains, rng):
    """Gaussian noise shaped by the envelope g and passed through the night's EMG
    filter, scaled to 1 uV RMS where g = 1: amplitude a then means a uV RMS in
    the night's band during activity."""
    s, _ = bandpass(rng.normal(0.0, 1.0, len(g)) * g, fs, mains=mains)
    full = g >= 0.999
    rms = float(np.sqrt(np.mean(s[full] ** 2))) if full.any() else 1.0
    return (s / rms).astype(np.float32)


def bin_mean(x, fs, env_fs=ENV_FS):
    """Mean of x in each envelope bin (the bins of extract.bin_envelope)."""
    n_bins = int(np.floor(len(x) * env_fs / fs))
    edges = np.floor(np.arange(n_bins + 1) * fs / env_fs).astype(np.int64)
    return (np.add.reduceat(np.asarray(x, np.float64), edges[:-1]) / np.diff(edges)) \
        .astype(np.float32)


def truth(gbins, seconds, kind, env_fs=ENV_FS):
    """
    Per unit of `seconds` over the night's concatenated REM (the units the
    methods score): 1 injected, 0 clean, -1 neither (left out). kind "tonic":
    injected when activity covers >= TONIC_COVER of the unit, clean when there
    is none in it; "burst": injected when the unit holds >= INTERIOR_MIN_S of
    full activity, clean when there is none in it or within CLEAN_MARGIN_S.
    """
    from scipy.ndimage import binary_dilation
    gbins = np.asarray(gbins, np.float32).reshape(-1)
    cover = _mini(gbins, env_fs, seconds).mean(axis=1)
    full = _mini(gbins >= 0.999, env_fs, seconds).sum(axis=1)
    m = int(round(CLEAN_MARGIN_S * env_fs)) if kind == "burst" else 0
    present = gbins > 0
    if m:
        present = binary_dilation(present, structure=np.ones(2 * m + 1, bool))
    near = _mini(present, env_fs, seconds).any(axis=1)
    inj = cover >= TONIC_COVER if kind == "tonic" \
        else full >= int(round(INTERIOR_MIN_S * env_fs))
    out = np.full(len(cover), -1, np.int8)
    out[inj] = 1
    out[~near] = 0
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  RBD DONORS (transplant)
# ══════════════════════════════════════════════════════════════════════════════

def make_donor(cohort, rec, d):
    """An RBD night's REM chin EMG as the main pipeline sees it (native band,
    mains notch, heartbeat removed), run of REM epochs by run, with its
    quiet-REM level."""
    from .cohorts import _beats, quiet_level
    from .ecg import subtract_heartbeat
    beats = _beats(d)
    fs = float(d["fs"])
    xf, band = bandpass(d["chin"], fs, mains=cohort.mains)
    if len(beats):
        xf = subtract_heartbeat(xf, fs, beats)
    per = int(round(fs * EPOCH_SEC))
    hyp = np.asarray(d["hypnogram"])[: len(xf) // per]
    rms, mav = bin_envelope(xf, fs)
    runs = [xf[a * per: b * per].astype(np.float32) for a, b in rem_stretches(hyp)]
    # the donor's own night scored as recorded: the reference on CAP's system
    pack = build_pack({"rms": rms, "mav": mav}, [], hyp, ENV_FS, EPOCH_SEC,
                      {"record": rec, "unit_known": bool(d["unit_known"])})
    own = {f"{r['method']}/{r['variant']}": {k: r.get(k) for k in
                                             ("tonic_pct", "phasic_pct", "any_pct", "rai",
                                              "positive", "background_phasic_uV")}
           for r in score_copy(pack, pack["rem"][0], pack["rem"][1])}
    return {"record": rec, "fs": fs, "band": list(band), "unit_known": bool(d["unit_known"]),
            "level_uv": round(float(quiet_level(rms, hyp)), 4),
            "rem_epochs": int(sum(len(r) for r in runs) // per),
            "lengths": [len(r) for r in runs], "own": own,
            "sig": np.concatenate(runs) if runs else np.zeros(0, np.float32)}


def save_donor(path, donor):
    meta = {k: v for k, v in donor.items() if k != "sig"}
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, sig=donor["sig"], meta=json.dumps(meta))
    tmp.replace(path)
    return path


def load_donor(path):
    z = np.load(path, allow_pickle=False)
    return dict(json.loads(str(z["meta"])), sig=z["sig"])


def donor_track(donor, hyp, fs, mains):
    """
    The donor's REM chin EMG laid over this night's REM, as one track over the
    night's concatenated REM samples. Recipient REM epoch j receives donor REM
    epoch floor(j n_d / n_r), i.e. the donor's night sampled evenly whatever the
    two REM durations; the donor's runs of REM are resampled to this night's
    rate first, and each run of this night's REM is then passed through its
    EMG filter (band and mains notch), as everything it records would be.
    """
    from scipy.signal import resample_poly
    per = int(round(fs * EPOCH_SEC))
    f = Fraction(int(round(fs)), int(round(donor["fs"]))).limit_denominator(1000)
    epochs, pos = [], 0
    for n in donor["lengths"]:
        y = donor["sig"][pos:pos + n].astype(np.float64)
        pos += n
        if f != 1:
            y = resample_poly(y, f.numerator, f.denominator)
        k = len(y) // per
        if k:
            epochs.append(y[: k * per].reshape(k, per))
    epochs = np.concatenate(epochs) if epochs else np.zeros((0, per))
    stretches = rem_stretches(hyp)
    n_r = sum(b - a for a, b in stretches)
    if n_r == 0 or len(epochs) == 0:
        return np.zeros(n_r * per, np.float32)
    track = epochs[(np.arange(n_r) * len(epochs)) // n_r].reshape(-1)
    out, i = np.empty(len(track), np.float32), 0
    for a, b in stretches:
        n = (b - a) * per
        out[i:i + n] = bandpass(track[i:i + n], fs, mains=mains)[0]
        i += n
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  SCORING THE COPIES OF ONE NIGHT
# ══════════════════════════════════════════════════════════════════════════════

def score_copy(pack, rms, mav, gbins=None, cache=None):
    """Score one copy of the night's REM (rms, mav: (n_rem_epochs, bins)) with
    every variant and the RAI. With gbins (the added activity's envelope per
    bin), also count detections on injected and clean units."""
    p = dict(pack, rem=np.stack([rms, mav]))
    rows = []
    cache = {} if cache is None else cache
    for name, variant, m in variants():
        masks = {}
        r = score_night(p, m, "native", masks=masks)
        r["variant"] = variant
        if gbins is not None and "error" not in r:
            for key, seconds, kind in (("tonic", m.tonic_epoch_s, "tonic"),
                                       ("phasic", m.phasic_mini_s, "burst"),
                                       ("any", m.any_mini_s, "burst")):
                dec = masks.get(key)
                if dec is None or not seconds:
                    continue
                if (seconds, kind) not in cache:
                    cache[seconds, kind] = truth(gbins, seconds, kind)
                t = cache[seconds, kind][: len(dec)]
                r.update({f"{key}_ni": int((t == 1).sum()), f"{key}_ki": int((dec & (t == 1)).sum()),
                          f"{key}_nc": int((t == 0).sum()), f"{key}_kc": int((dec & (t == 0)).sum())})
        rows.append(r)
    r = rem_atonia_index(p, "native")
    r["variant"] = "published"
    rows.append(r)
    return rows


def run_night(cohort, rec, d, donor=None, info=None):
    """
    Every copy of one night: as recorded; with synthetic tonic, then phasic,
    activity at each of AMPLITUDES_UV (the same placement and noise at every
    amplitude, seeded by cohort and record); and, given a donor, with the
    donor's RBD activity at each of DONOR_SCALES. Returns one row per copy and
    method variant.
    """
    from .cohorts import _beats, quiet_level
    from .ecg import subtract_heartbeat
    beats = _beats(d)
    fs, mains = float(d["fs"]), cohort.mains
    xf, band = bandpass(d["chin"], fs, mains=mains)
    if len(beats):
        xf = subtract_heartbeat(xf, fs, beats)
    rms, mav = bin_envelope(xf, fs)
    per_bins = int(round(ENV_FS * EPOCH_SEC))
    hyp = np.asarray(d["hypnogram"])[: len(rms) // per_bins]
    meta = {"record": rec, "cohort": cohort.name, "chin_fs": fs, "band": list(band),
            "unit_known": bool(d["unit_known"])}
    pack = build_pack({"rms": rms, "mav": mav}, [], hyp, ENV_FS, EPOCH_SEC, meta)
    rem_ep = pack["rem_epochs"]
    info = dict(info or {}, cohort=cohort.name, record=rec, chin_fs=fs,
                unit_known=bool(d["unit_known"]), beats=int(len(beats)),
                rem_minutes=round(len(rem_ep) * EPOCH_SEC / 60.0, 1),
                level_uv=round(float(quiet_level(rms, hyp)), 4))
    rows = []

    def add(pattern, amp, r_rms, r_mav, gbins=None, cache=None, extra=None):
        for r in score_copy(pack, r_rms, r_mav, gbins, cache):
            rows.append({**info, **(extra or {}), "pattern": pattern, "amp_uv": amp, **r})

    add("none", 0.0, pack["rem"][0], pack["rem"][1])
    if len(rem_ep) < 2 or not d["unit_known"]:
        return rows
    per = int(round(fs * EPOCH_SEC))
    idx = (rem_ep[:, None] * per + np.arange(per)[None, :]).reshape(-1)
    x_rem = xf[idx]
    shape = (len(rem_ep), per_bins)
    rng = np.random.default_rng(zlib.crc32(f"{cohort.name}/{rec}".encode()))
    for pattern, make in (("tonic", tonic_envelope), ("phasic", phasic_envelope)):
        g = make(hyp, fs, rng, len(xf))
        s = activity(g, fs, mains, rng)[idx]
        gbins, cache = bin_mean(g[idx], fs).reshape(shape), {}
        del g
        for a in AMPLITUDES_UV:
            r_rms, r_mav = bin_envelope(x_rem + a * s, fs)
            add(pattern, a, r_rms.reshape(shape), r_mav.reshape(shape), gbins, cache)
    if donor is not None:
        track = donor_track(donor, hyp, fs, mains)
        extra = {"donor": donor["record"], "donor_level_uv": donor.get("level_uv"),
                 "donor_rem_epochs": donor.get("rem_epochs")}
        for c in DONOR_SCALES:
            r_rms, r_mav = bin_envelope(x_rem + c * track, fs)
            add("transplant", c, r_rms.reshape(shape), r_mav.reshape(shape), extra=extra)
    return rows


# ══════════════════════════════════════════════════════════════════════════════
#  DETECTION THRESHOLD OF ONE NIGHT
# ══════════════════════════════════════════════════════════════════════════════

def corrected_rate(ki, ni, kc, nc):
    """Share of injected units detected, corrected for the share of clean units
    detected in the same copy: (h - p0) / (1 - p0)."""
    h = ki / ni if ni else np.nan
    p0 = kc / nc if nc else np.nan
    if not (np.isfinite(h) and np.isfinite(p0)) or p0 >= 1.0:
        return np.nan
    return (h - p0) / (1.0 - p0)


def isotonic(y):
    """Non-decreasing least-squares fit (pool adjacent violators)."""
    vals, wts = [], []
    for v in np.asarray(y, float):
        vals.append(v)
        wts.append(1.0)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            w = wts[-2] + wts[-1]
            vals[-2:] = [(vals[-2] * wts[-2] + vals[-1] * wts[-1]) / w]
            wts[-2:] = [w]
    return np.repeat(vals, np.asarray(wts, int))


def a50(amps, rates, level=0.5):
    """
    The amplitude at which the corrected detection rate, made non-decreasing in
    amplitude (isotonic fit), first reaches `level`, interpolated in log
    amplitude between grid points. Returns (amplitude, censored): censored -1 =
    reached at the smallest amplitude (value is the grid's lower end), +1 = never
    reached (value is the upper end), 0 = within.
    """
    amps, rates = np.asarray(amps, float), np.asarray(rates, float)
    ok = np.isfinite(rates)
    amps, rates = amps[ok], rates[ok]
    if len(amps) < 3:
        return np.nan, 0
    order = np.argsort(amps)
    amps, rates = amps[order], isotonic(rates[order])
    hit = np.flatnonzero(rates >= level)
    if len(hit) == 0:
        return float(amps[-1]), 1
    i = int(hit[0])
    if i == 0:
        return float(amps[0]), -1
    x0, x1 = np.log(amps[i - 1]), np.log(amps[i])
    y0, y1 = rates[i - 1], rates[i]
    x = x0 + (level - y0) * (x1 - x0) / (y1 - y0) if y1 > y0 else x1
    return float(np.exp(x)), 0


# The measures whose detection thresholds are analyzed: (method, measure, the
# pattern it is read on). The first three are the primary ones.
MEASURES = (("Montreal", "tonic", "tonic"), ("Montreal", "phasic", "phasic"),
            ("SINBAR", "any", "phasic"), ("SINBAR", "any", "tonic"),
            ("SINBAR", "tonic", "tonic"), ("SINBAR", "phasic", "phasic"),
            ("AASM", "tonic", "tonic"), ("AASM", "phasic", "phasic"))
PRIMARY = MEASURES[:3]


def a50_table(df):
    """One row per night, measure and variant: the detection threshold (A50,
    uV), its censoring, the corrected detection rate at each amplitude and the
    clean units' detection rate as recorded (p0 of the smallest amplitude)."""
    import pandas as pd
    out = []
    syn = df[df.pattern.isin(["tonic", "phasic"])]
    for (cohort, record), g in syn.groupby(["cohort", "record"], sort=False):
        for meth, measure, pattern in MEASURES:
            cols = [f"{measure}_{c}" for c in ("ki", "ni", "kc", "nc")]
            if not set(cols) <= set(g.columns):
                continue
            for variant in ("published", "FC"):
                s = g[(g.method == meth) & (g.variant == variant) & (g.pattern == pattern)]
                s = s.dropna(subset=cols).sort_values("amp_uv")
                if len(s) < 3:
                    continue
                rates = [corrected_rate(*r) for r in s[cols].to_numpy(float)]
                a, cens = a50(s.amp_uv.to_numpy(float), rates)
                first = s.iloc[0]
                out.append({"cohort": cohort, "record": record, "method": meth,
                            "variant": variant, "measure": measure, "pattern": pattern,
                            "a50": a, "censored": cens,
                            "p0": first[cols[2]] / first[cols[3]] if first[cols[3]] else np.nan,
                            "n_injected": int(first[cols[1]]),
                            **{f"r{a_:g}": r for a_, r in zip(s.amp_uv, rates)}})
    return pd.DataFrame(out)

In [ ]:
%%writefile /content/rbd-rwa/run_inject.py
"""Known muscle activity added to every night (paper/preregistration_injection.md).

    python run_inject.py --donors                    # CAP's RBD nights -> donors
    python run_inject.py --cohort cap                # CAP nights without RBD
    python run_inject.py --cohort hmc
    python run_inject.py --cohort chal18 --every 3   # MGH: every third record
    python run_inject.py --cohort boas
    python run_inject.py --cohort dodo / dodh

A donor is one RBD night's REM chin EMG, as the main pipeline sees it (src/inject.py,
make_donor); only nights on CAP's main system (512 Hz, microvolts) are used. Every
other night is streamed once and all its copies are scored at once: as recorded,
with synthetic tonic and phasic activity at 49 amplitudes, and with its assigned
donor's RBD activity at full and half amplitude. Only the scores are kept, one file
per night (DURABLE/inject/<cohort>/<record>.csv.gz); re-running skips nights done.

Donors are assigned in turn within each cohort: the cohort's records (after --every)
sorted by name, record i receives donor i mod 21, donors sorted by name. Every donor
therefore contributes about equally to every recording system.
"""
import argparse
import json
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from run_cohort import Mirror

OUT = config.DURABLE / "inject"


def _rbd(rec):
    return "".join(c for c in rec if not c.isdigit()) == "rbd"


def _donor_work(rec, scratch, out_path, keep_raw):
    from src.cohorts import COHORTS
    from src.inject import make_donor, save_donor
    cohort = COHORTS["cap"]
    folder = Path(scratch) / f"donor_{rec}"
    folder.mkdir(parents=True, exist_ok=True)
    try:
        d = cohort.read(rec, cohort.fetch(rec, folder))
        if not (float(d["fs"]) == 512.0 and d["unit_known"]):
            return rec, None, f"not on CAP's main system ({d['fs']} Hz, units known: {d['unit_known']})"
        donor = make_donor(cohort, rec, d)
        save_donor(Path(out_path), donor)
        return rec, {k: donor[k] for k in ("fs", "level_uv", "rem_epochs")}, None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=2)[-300:]}"
    finally:
        if not keep_raw:
            shutil.rmtree(folder, ignore_errors=True)


def _work(name, rec, demo, scratch, out_path, donor_path, keep_raw):
    """Runs in a worker process: one night from download to scores."""
    import pandas as pd
    from src.cohorts import COHORTS
    from src.inject import load_donor, run_night
    cohort = COHORTS[name]
    t0 = time.time()
    folder = Path(scratch) / rec
    folder.mkdir(parents=True, exist_ok=True)
    try:
        files = cohort.fetch(rec, folder)
        t_dl = time.time() - t0
        d = cohort.read(rec, files)
        donor = load_donor(donor_path) if donor_path else None
        rows = run_night(cohort, rec, d, donor, info=dict(demo or {}))
        out = Path(out_path)
        tmp = out.with_name(out.name.replace(".csv.gz", ".tmp.csv.gz"))
        pd.DataFrame(rows).to_csv(tmp, index=False, compression="gzip")
        tmp.replace(out)
        return rec, {"seconds": round(time.time() - t0, 1), "download_s": round(t_dl, 1),
                     "rem_min": rows[0]["rem_minutes"], "level_uv": rows[0]["level_uv"],
                     "donor": donor["record"] if donor else None}, None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=2)[-300:]}"
    finally:
        if not keep_raw:
            shutil.rmtree(folder, ignore_errors=True)


def _manifest(name):
    p = config.MANIFESTS / f"inject_{name}.json"
    return p, (json.loads(p.read_text()) if p.exists() else {"done": {}, "failed": {}})


def _save(path, m):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(m, indent=1))
    tmp.replace(path)


def run(jobs, workers, manifest_name, fn, report):
    path, m = _manifest(manifest_name)
    t_start = time.time()
    with ProcessPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(fn, *args) for args in jobs]
        for i, f in enumerate(as_completed(futs), 1):
            rec, info, err = f.result()
            if err:
                m["failed"][rec] = err
                print(f"[{i}/{len(jobs)}] {rec} FAILED {err[:200]}", flush=True)
            else:
                m["done"][rec] = info
                m["failed"].pop(rec, None)
                report(rec, info)
                print(f"[{i}/{len(jobs)}] {rec}  {info}", flush=True)
            _save(path, m)
    print(f"{manifest_name}: {len(m['done'])} done, {len(m['failed'])} failed, "
          f"{(time.time() - t_start) / 60:.1f} min", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cohort")
    ap.add_argument("--donors", action="store_true", help="build the RBD donors from CAP")
    ap.add_argument("--every", type=int, default=1, help="use every k-th record (MGH: 3)")
    ap.add_argument("--limit", type=int, default=0, help="stop after this many (testing)")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--mirror", default=None, help="also copy each result here (Drive)")
    args = ap.parse_args()
    config.describe()

    if args.donors:
        from src.fetch import cap_records
        out_dir = OUT / "donors"
        out_dir.mkdir(parents=True, exist_ok=True)
        mirror = Mirror(Path(args.mirror) / "donors" if args.mirror else None)
        mirror.restore(out_dir)
        recs = sorted((r for g in cap_records().values() for r in g if _rbd(r)),
                      key=lambda r: int("".join(c for c in r if c.isdigit())))
        todo = [r for r in recs if not (out_dir / f"{r}.npz").exists()]
        if args.limit:
            todo = todo[: args.limit]
        print(f"donors: {len(recs)} CAP RBD nights, {len(todo)} to extract", flush=True)
        run([(r, str(config.SCRATCH), str(out_dir / f"{r}.npz"), args.keep_raw) for r in todo],
            args.workers, "donors", _donor_work, lambda r, i: mirror.push(out_dir / f"{r}.npz"))
        return

    from src.cohorts import COHORTS
    name = args.cohort
    cohort = COHORTS[name]
    donors = sorted((p for p in (config.DURABLE / "inject" / "donors").glob("rbd*.npz")
                     if ".tmp" not in p.name),
                    key=lambda p: int("".join(c for c in p.stem if c.isdigit())))
    if not donors:
        print("no donors yet: run with --donors first; continuing without the transplant",
              flush=True)
    records = [r for r in cohort.records() if not (name == "cap" and _rbd(r))]
    records = sorted(records[:: args.every])
    donor_of = {r: (str(donors[i % len(donors)]) if donors else None)
                for i, r in enumerate(records)}
    out_dir = OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    mirror = Mirror(Path(args.mirror) / name if args.mirror else None)
    if mirror.ok:
        try:
            for p in mirror.path.glob("*.csv.gz"):
                if not (out_dir / p.name).exists():
                    shutil.copy2(p, out_dir / p.name)
        except OSError as e:
            mirror._fail(e)
    todo = [r for r in records if not (out_dir / f"{r}.csv.gz").exists()]
    if args.limit:
        todo = todo[: args.limit]
    demo = cohort.demographics()
    print(f"\n{name}: {len(records)} nights (every {args.every}), {len(todo)} to process with "
          f"{args.workers} workers; {len(donors)} donors\n", flush=True)
    run([(name, r, demo.get(r), str(config.SCRATCH), str(out_dir / f"{r}.csv.gz"),
          donor_of[r], args.keep_raw) for r in todo],
        args.workers, name, _work, lambda r, i: mirror.push(out_dir / f"{r}.csv.gz"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/sim_injection.py
"""White-noise predictions for the injection experiment.

The same injection and scoring as run_inject.py, on synthetic nights whose chin
EMG is band-limited white noise at a known floor: the detection threshold each
rule would have if a night's background were only the noise the ladder adds.
Real nights are compared with these (criterion A4, paper/preregistration_injection.md),
and the relation between floor and threshold here is the theoretical one the
real nights are expected to follow.

    python sim_injection.py           -> results/inject_whitenoise.csv (every copy)
                                         results/inject_whitenoise_a50.csv
"""
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.extract import bandpass
from src.inject import a50_table, run_night
from src.rwa import NOISE_P5_RATIO

FLOORS_UV = (0.3, 0.4, 0.5, 0.6, 0.7, 0.85, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0)
RATES = (200, 250, 256, 512)
SEEDS = (0, 1, 2)
# 10 epochs awake, four cycles of 20 min N2 and 12.5 min REM, 5 min N2: 100 REM epochs
HYPNOGRAM = np.r_[[0] * 10, ([2] * 40 + [4] * 25) * 4, [2] * 10].astype(np.int8)


class WhiteNoise:
    name, mains = "whitenoise", 50.0


def night(fs, floor, seed):
    """Raw white noise scaled so that its in-band RMS is floor / 0.91, the noise
    a quiet-REM level (5th percentile of 1-s RMS) of `floor` implies."""
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, 1.0, int(len(HYPNOGRAM) * 30 * fs))
    k = floor / NOISE_P5_RATIO / np.sqrt(np.mean(bandpass(w, fs, mains=50.0)[0] ** 2))
    return dict(chin=w * k, fs=float(fs), unit_known=True, hypnogram=HYPNOGRAM)


def one(args):
    fs, floor, seed = args
    rec = f"f{fs}_b{floor:g}_s{seed}"
    rows = run_night(WhiteNoise(), rec, night(fs, floor, seed),
                     info={"sim_fs": fs, "sim_floor_uv": floor, "seed": seed})
    return rows


def main(workers=3):
    jobs = [(fs, b, s) for fs in RATES for b in FLOORS_UV for s in SEEDS]
    print(f"{len(jobs)} synthetic nights", flush=True)
    with ProcessPoolExecutor(max_workers=workers) as ex:
        rows = [r for part in ex.map(one, jobs) for r in part]
    df = pd.DataFrame(rows)
    df.to_csv(config.RESULTS / "inject_whitenoise.csv", index=False)
    a = a50_table(df)
    info = df.drop_duplicates(["cohort", "record"]).set_index(["cohort", "record"])
    for col in ("sim_fs", "sim_floor_uv", "seed", "level_uv"):
        a[col] = info[col].reindex(pd.MultiIndex.from_frame(a[["cohort", "record"]])).to_numpy()
    a.to_csv(config.RESULTS / "inject_whitenoise_a50.csv", index=False)
    print(a.groupby(["method", "measure", "pattern", "variant", "sim_floor_uv"]).a50.median()
          .unstack("sim_floor_uv").round(2).to_string())
    return a


if __name__ == "__main__":
    main(int(sys.argv[1]) if len(sys.argv) > 1 else 3)

In [ ]:
%%writefile /content/rbd-rwa/run_inject_report.py
"""The injection experiment, against the criteria fixed in paper/preregistration_injection.md.

  I1  nights and floors by recording system
  I2  detection thresholds (A50) of synthetic activity against the floor: slopes (A1, A2)
  I3  thresholds by recording system (A3)
  I4  real backgrounds against white noise (A4)
  I5  transplanted RBD activity: sensitivity by system (B1, B3)
  I6  transplanted RBD activity against the white-noise ladder (B2)
  I7  the same RBD activity measured on different systems, paired by donor (B4)
  I8  secondary: night-level dose-response, every measure, the donors' own nights

    python run_inject_report.py -> results/inject_a50.csv, inject_transplant.csv,
                                   inject_report.md, figures/fig11_injection
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.inject import AMPLITUDES_UV, MEASURES, PRIMARY, a50_table, load_donor
from run_report import MIN_REM, WITHDRAWN, wilson
from run_score import group_of

COHORTS = ("cap", "chal18", "hmc", "boas", "dodh", "dodo")
NAME = {"chal18": "MGH", "hmc": "HMC", "boas": "BOAS", "dodh": "DOD-H", "dodo": "DOD-O"}
SYSTEMS = ["CAP patients", "MGH", "HMC", "BOAS", "DOD-H", "DOD-O"]
FC_MAX_FLOOR, FC_MIN_FS = 1.69, 200.0
DECIDING = [("Montreal", "published"), ("Montreal", "FC"), ("SINBAR", "published"),
            ("SINBAR", "FC"), ("RAI", "published")]
YES = (True, "True", 1, 1.0)
N_BOOT = 2000


# ══════════════════════════════════════════════════════════════════════════════
#  DATA
# ══════════════════════════════════════════════════════════════════════════════

def load():
    files = [p for c in COHORTS for p in sorted((config.DURABLE / "inject" / c).glob("*.csv.gz"))
             if ".tmp" not in p.name]
    print(f"reading {len(files)} nights", flush=True)
    df = pd.concat((pd.read_csv(p, low_memory=False) for p in files), ignore_index=True)
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record.astype(str)).isin(gone)].copy()
    df["group"] = [group_of(c, r) for c, r in zip(df.cohort, df.record)]
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select([same & ~df.group.isin(["RBD", "healthy"]), cap & (df.group == "healthy")],
                          ["CAP patients", "CAP healthy"], default=df.cohort.map(NAME).fillna(""))
    # BOAS: a participant's first night is primary (as in run_external.py)
    df["primary"] = True
    b = df.cohort == "boas"
    if b.any():
        num = df.loc[b, "record"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        pid = df.loc[b, "pid"].fillna(df.loc[b, "record"]).astype(str) if "pid" in df \
            else df.loc[b, "record"].astype(str)
        df.loc[b, "primary"] = (num == num.groupby(pid).transform("min")).to_numpy()
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    return df


def nights(df):
    """One row per night: system, rate, floor (quiet-REM level as recorded), REM."""
    n = df[df.pattern == "none"].drop_duplicates(["cohort", "record"])
    return n[["cohort", "record", "who", "group", "primary", "chin_fs", "unit_known",
              "rem_minutes", "level_uv", "beats"] + [c for c in ("age", "sex") if c in n]] \
        .reset_index(drop=True)


def fc_valid(fs, level):
    return (np.asarray(fs, float) >= FC_MIN_FS) & (np.asarray(level, float) <= FC_MAX_FLOOR)


def loglog_interp(x, xp, fp):
    """Piecewise-linear interpolation in log-log, extrapolated with the end slopes."""
    lx, lxp, lfp = np.log(np.asarray(x, float)), np.log(xp), np.log(fp)
    y = np.interp(lx, lxp, lfp)
    lo, hi = lx < lxp[0], lx > lxp[-1]
    s0 = (lfp[1] - lfp[0]) / (lxp[1] - lxp[0])
    s1 = (lfp[-1] - lfp[-2]) / (lxp[-1] - lxp[-2])
    y[lo] = lfp[0] + s0 * (lx[lo] - lxp[0])
    y[hi] = lfp[-1] + s1 * (lx[hi] - lxp[-1])
    return np.exp(y)


def slope_ci(x, y, seed=1):
    """OLS slope of log y on log x, with a 95% bootstrap interval over nights."""
    lx, ly = np.log(np.asarray(x, float)), np.log(np.asarray(y, float))
    if len(lx) < 5:
        return np.nan, np.nan, np.nan
    s = np.polyfit(lx, ly, 1)[0]
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(lx), len(lx))
        if np.ptp(lx[i]) > 0:
            boots.append(np.polyfit(lx[i], ly[i], 1)[0])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(s), float(lo), float(hi)


def share(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    p, lo, hi = wilson(v.sum(), len(v))
    return p, lo, hi, len(v)


def fmt_share(v):
    p, lo, hi, n = share(v)
    return f"{100 * p:3.0f} [{100 * lo:.0f}-{100 * hi:.0f}] (n={n})" if n else "-"


def donor_boot(t, value, systems, seed=2):
    """Bootstrap over donors: resample the 21 donors, recompute `value(sub)` (a dict
    system -> number) on the hybrids of the resampled donors."""
    rng = np.random.default_rng(seed)
    donors = np.array(sorted(t.donor.dropna().unique()))
    by = {d: t[t.donor == d] for d in donors}
    out = []
    for _ in range(N_BOOT):
        pick = rng.choice(donors, len(donors), replace=True)
        out.append(value(pd.concat([by[d] for d in pick], ignore_index=True)))
    return pd.DataFrame(out)


# ══════════════════════════════════════════════════════════════════════════════
#  REPORT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    df = load()
    df = df[df.unit_known.astype(bool)]
    nt = nights(df)
    keep = nt[(nt.rem_minutes >= MIN_REM)][["cohort", "record"]]
    df = df.merge(keep, on=["cohort", "record"])
    nt = nights(df)
    lines = []
    say = lambda s="": (print(s), lines.append(s))
    verdict = {}

    # I1 ─────────────────────────────────────────────────────────────────────
    say("I1  Nights (>= 10 min REM, units known) and quiet-REM level as recorded (µV)")
    for who in SYSTEMS + ["CAP healthy"]:
        g = nt[(nt.who == who) & nt.primary]
        if g.empty:
            continue
        say(f"    {who:<13} n={len(g):4d}  rate {'/'.join(f'{f:g}' for f in sorted(g.chin_fs.unique()))} Hz"
            f"  floor median {g.level_uv.median():.2f} [IQR {g.level_uv.quantile(.25):.2f}-"
            f"{g.level_uv.quantile(.75):.2f}]  FC-valid {fc_valid(g.chin_fs, g.level_uv).mean():.0%}"
            f"  REM {g.rem_minutes.median():.0f} min")
    say()

    # A50 per night ──────────────────────────────────────────────────────────
    a = a50_table(df)
    a = a.merge(nt, on=["cohort", "record"])
    a.to_csv(config.RESULTS / "inject_a50.csv", index=False)
    main_ = a[a.who.isin(SYSTEMS) & a.primary & (a.chin_fs >= FC_MIN_FS)]

    # I2 ─────────────────────────────────────────────────────────────────────
    say("I2  Detection threshold A50 (µV) against the floor: log-log slope over nights")
    say("    (six systems, >= 200 Hz; FC also floor <= 1.69 uV; A50 not censored)")
    say("    criteria A1: published slope 0.75-1.25;  A2: FC slope -0.25 to +0.25")
    for meth, measure, pattern in MEASURES:
        for variant in ("published", "FC"):
            g = main_[(main_.method == meth) & (main_.measure == measure) &
                      (main_.pattern == pattern) & (main_.variant == variant)]
            if variant == "FC":
                g = g[fc_valid(g.chin_fs, g.level_uv)]
            ok = g[(g.censored == 0) & np.isfinite(g.a50) & (g.level_uv > 0)]
            s, lo, hi = slope_ci(ok.level_uv, ok.a50)
            cens = f"censored {(g.censored != 0).sum()} of {len(g)}"
            tag = ""
            if (meth, measure, pattern) in PRIMARY:
                key = ("A1 " if variant == "published" else "A2 ") + f"{meth} {measure}"
                verdict[key] = (0.75 <= s <= 1.25) if variant == "published" else (-0.25 <= s <= 0.25)
                tag = f"  -> {'met' if verdict[key] else 'NOT met'}"
            say(f"    {meth:<9}{measure:<7}({pattern:<6}) {variant:<10} slope {s:5.2f} "
                f"[{lo:5.2f} to {hi:5.2f}]  n={len(ok)}  {cens}{tag}")
    say()

    # I3 ─────────────────────────────────────────────────────────────────────
    say("I3  Median A50 (µV) by system (censored values at the grid's ends);")
    say("    criterion A3: highest/lowest system median >= 2.5 published, <= 1.3 FC")
    say(f"    {'':<34}" + "".join(f"{w:<13}" for w in SYSTEMS) + "max/min")
    for meth, measure, pattern in PRIMARY:
        for variant in ("published", "FC"):
            g = main_[(main_.method == meth) & (main_.measure == measure) &
                      (main_.pattern == pattern) & (main_.variant == variant)]
            if variant == "FC":
                g = g[fc_valid(g.chin_fs, g.level_uv)]
            med = {w: g[g.who == w].a50.median() for w in SYSTEMS if (g.who == w).any()}
            ratio = max(med.values()) / min(med.values()) if med else np.nan
            key = f"A3 {meth} {measure} {variant}"
            verdict[key] = ratio >= 2.5 if variant == "published" else ratio <= 1.3
            say(f"    {meth + ' ' + measure + ' ' + variant:<34}" +
                "".join(f"{med.get(w, np.nan):<13.2f}" for w in SYSTEMS) +
                f"{ratio:.2f} -> {'met' if verdict[key] else 'NOT met'}")
    say()

    # I4 ─────────────────────────────────────────────────────────────────────
    say("I4  Real backgrounds against white noise: A50 / white-noise A50 at the night's floor")
    say("    criterion A4: within 0.75-1.25 in >= 80% of nights (censored nights count as outside)")
    wn_path = config.RESULTS / "inject_whitenoise_a50.csv"
    if wn_path.exists():
        wn = pd.read_csv(wn_path)
        rates = np.array(sorted(wn.sim_fs.unique()), float)
        rows = []
        for meth, measure, pattern in PRIMARY:
            for variant in ("published", "FC"):
                g = main_[(main_.method == meth) & (main_.measure == measure) &
                          (main_.pattern == pattern) & (main_.variant == variant)].copy()
                if variant == "FC":
                    g = g[fc_valid(g.chin_fs, g.level_uv)]
                w = wn[(wn.method == meth) & (wn.measure == measure) & (wn.pattern == pattern) &
                       (wn.variant == variant) & (wn.censored == 0)]
                pred = np.full(len(g), np.nan)
                for fs in g.chin_fs.unique():
                    near = rates[np.argmin(np.abs(rates - fs))]
                    c = w[w.sim_fs == near].groupby("sim_floor_uv")[["level_uv", "a50"]].mean()
                    m = (g.chin_fs == fs).to_numpy()
                    pred[m] = loglog_interp(g.level_uv[m], c.level_uv.to_numpy(), c.a50.to_numpy())
                g["ratio"] = g.a50 / pred
                inside = (g.censored == 0) & g.ratio.between(0.75, 1.25)
                key = f"A4 {meth} {measure} {variant}"
                verdict[key] = inside.mean() >= 0.80
                by = "  ".join(f"{s} {g[g.who == s].ratio.median():.2f}" for s in SYSTEMS
                               if (g.who == s).any())
                say(f"    {meth + ' ' + measure + ' ' + variant:<34} within {100 * inside.mean():3.0f}% "
                    f"-> {'met' if verdict[key] else 'NOT met'};  median ratio {by}")
                rows.append(g.assign(wn_a50=pred))
        pd.concat(rows).to_csv(config.RESULTS / "inject_a50_vs_whitenoise.csv", index=False)
    else:
        say("    results/inject_whitenoise_a50.csv missing: run sim_injection.py first")
    say()

    # Transplant ─────────────────────────────────────────────────────────────
    t = df[(df.pattern == "transplant") & df.who.isin(SYSTEMS + ["CAP healthy"]) & df.primary].copy()
    lvl = t[(t.method == "Montreal") & (t.variant == "published")] \
        .set_index(["cohort", "record", "amp_uv"]).background_phasic_uV
    t["hybrid_level"] = lvl.reindex(pd.MultiIndex.from_frame(t[["cohort", "record", "amp_uv"]])).to_numpy()
    t["fc_ok"] = fc_valid(t.chin_fs, t.hybrid_level)
    t.to_csv(config.RESULTS / "inject_transplant.csv", index=False)
    nat = df[(df.pattern == "none") & df.who.isin(SYSTEMS) & df.primary]
    donors = {}
    for p in sorted((config.DURABLE / "inject" / "donors").glob("*.npz")):
        if ".tmp" not in p.name:
            d = load_donor(p)
            donors[d["record"]] = d
    say(f"I5  Transplanted RBD activity ({len(donors)} donors from CAP's main system):")
    say("    share of hybrid nights flagged, % [95% CI]; native = the same nights as recorded")
    if not t.empty:
        for c in sorted(t.amp_uv.unique(), reverse=True):
            say(f"  donor activity x{c:g}; hybrid floor median by system: " + "  ".join(
                f"{w} {t[(t.who == w) & (t.amp_uv == c)].hybrid_level.median():.2f}"
                for w in SYSTEMS if (t.who == w).any()))
            for meth, variant in DECIDING:
                g = t[(t.amp_uv == c) & (t.method == meth) & (t.variant == variant)]
                if variant == "FC":
                    g = g[g.fc_ok]
                cells = "  ".join(f"{w} {fmt_share(g[g.who == w].y)}" for w in SYSTEMS
                                  if (g.who == w).any())
                say(f"    {meth + ' ' + variant:<20} {cells}")
        if donors:
            own = pd.DataFrame([{"donor": k, **{f"{m}": v.get("positive") for m, v in d["own"].items()}}
                                for k, d in donors.items()])
            say("    the donors' own nights on CAP (as recorded): " + "  ".join(
                f"{col} {100 * own[col].map(lambda x: float(x in YES)).mean():.0f}%"
                for col in ("Montreal/published", "Montreal/FC", "SINBAR/published",
                            "SINBAR/FC", "RAI/published") if col in own))
        say("    native (no donor) share flagged: " + "  ".join(
            f"{meth} {variant}: " + ", ".join(
                f"{w} {100 * nat[(nat.who == w) & (nat.method == meth) & (nat.variant == variant)].y.mean():.0f}"
                for w in SYSTEMS if (nat.who == w).any())
            for meth, variant in DECIDING[:4]))

        # B1: published Montreal, full amplitude, BOAS minus HMC
        def sens(sub, meth, variant, c):
            g = sub[(sub.amp_uv == c) & (sub.method == meth) & (sub.variant == variant)]
            if variant == "FC":
                g = g[g.fc_ok]
            return {w: 100 * g[g.who == w].y.mean() for w in SYSTEMS if (g.who == w).any()}
        s1 = sens(t, "Montreal", "published", 1.0)
        if "BOAS" in s1 and "HMC" in s1:
            bt = donor_boot(t, lambda sub: sens(sub, "Montreal", "published", 1.0), SYSTEMS)
            d = bt["BOAS"] - bt["HMC"]
            diff = s1["BOAS"] - s1["HMC"]
            verdict["B1"] = diff >= 20
            say(f"  B1 Montreal published, full amplitude: BOAS {s1['BOAS']:.0f}% - HMC {s1['HMC']:.0f}% = "
                f"{diff:+.1f} points [donor bootstrap {np.nanpercentile(d, 2.5):+.1f} to "
                f"{np.nanpercentile(d, 97.5):+.1f}] (criterion >= 20) -> {'met' if verdict['B1'] else 'NOT met'}")
        # B3: half amplitude, HMC minus BOAS (FC: hybrids within its range only)
        for meth, variant in (("Montreal", "published"), ("Montreal", "FC"), ("SINBAR", "FC")):
            s = sens(t, meth, variant, 0.5)
            if "BOAS" in s and "HMC" in s:
                bt = donor_boot(t, lambda sub: sens(sub, meth, variant, 0.5), SYSTEMS)
                d = bt["HMC"] - bt["BOAS"]
                diff = s["HMC"] - s["BOAS"]
                key = f"B3 {meth} {variant}"
                verdict[key] = diff <= -20 if variant == "published" else abs(diff) <= 10
                say(f"  B3 {meth} {variant}, half amplitude: HMC {s['HMC']:.0f}% - BOAS {s['BOAS']:.0f}% = "
                    f"{diff:+.1f} points [donor bootstrap {np.nanpercentile(d, 2.5):+.1f} to "
                    f"{np.nanpercentile(d, 97.5):+.1f}] (criterion "
                    f"{'<= -20' if variant == 'published' else 'within +/-10'}) -> "
                    f"{'met' if verdict[key] else 'NOT met'};  range over all systems "
                    f"{max(s.values()) - min(s.values()):.0f} points")
    say()

    # I6 ─────────────────────────────────────────────────────────────────────
    say("I6  Transplanted RBD against the white-noise ladder (same donors, published Montreal tonic %,")
    say("    full amplitude): hybrid minus the donor's ladder value at the hybrid's floor;")
    say("    criterion B2: |median| <= 10 points in each system")
    lad_path = config.RESULTS / "ladder_scores.csv"
    if lad_path.exists() and not t.empty:
        lad = pd.read_csv(lad_path, low_memory=False)
        lad = lad[(lad.cohort == "cap") & (lad.method == "Montreal") & lad.record.isin(donors.keys())]
        curves = {}
        for rec, g in lad.groupby("record"):
            g = g.dropna(subset=["level", "tonic_pct"]).sort_values("level")
            g = g.drop_duplicates("level")
            if len(g) >= 3:
                curves[rec] = (np.log(g.level.to_numpy(float)), g.tonic_pct.to_numpy(float))
        h = t[(t.amp_uv == 1.0) & (t.method == "Montreal") & (t.variant == "published")].copy()
        h["ladder_tonic"] = [np.interp(np.log(l), *curves[dn]) if dn in curves and l > 0 else np.nan
                             for dn, l in zip(h.donor, h.hybrid_level)]
        h["diff"] = h.tonic_pct - h.ladder_tonic
        oks = []
        for w in SYSTEMS:
            g = h[h.who == w].dropna(subset=["diff"])
            if g.empty:
                continue
            med = g["diff"].median()
            oks.append(abs(med) <= 10)
            say(f"    {w:<13} median {med:+5.1f} points [IQR {g['diff'].quantile(.25):+.1f} to "
                f"{g['diff'].quantile(.75):+.1f}]  n={len(g)}")
        verdict["B2"] = bool(oks) and all(oks)
        say(f"    -> {'met' if verdict['B2'] else 'NOT met'}")
        h.to_csv(config.RESULTS / "inject_transplant_vs_ladder.csv", index=False)
    else:
        say("    results/ladder_scores.csv missing: not evaluated")
    say()

    # I7 ─────────────────────────────────────────────────────────────────────
    say("I7  The same RBD activity on different systems: Montreal tonic %, mean per donor, HMC minus")
    say("    BOAS (paired by donor); criterion B4 (half amplitude): published <= -10, FC within +/-5")
    if not t.empty:
        for c in sorted(t.amp_uv.unique(), reverse=True):
            for variant in ("published", "FC"):
                g = t[(t.amp_uv == c) & (t.method == "Montreal") & (t.variant == variant)]
                if variant == "FC":
                    g = g[g.fc_ok]
                per = g.groupby(["donor", "who"]).tonic_pct.mean().unstack()
                if not {"HMC", "BOAS"} <= set(per.columns):
                    continue
                dd = (per["HMC"] - per["BOAS"]).dropna()
                if dd.empty:
                    continue
                rng = np.random.default_rng(3)
                boots = [rng.choice(dd.to_numpy(), len(dd)).mean() for _ in range(N_BOOT)]
                line = (f"    x{c:g} {variant:<10} HMC - BOAS {dd.mean():+5.1f} points "
                        f"[{np.percentile(boots, 2.5):+.1f} to {np.percentile(boots, 97.5):+.1f}], "
                        f"{len(dd)} donors;  by system: " + "  ".join(
                            f"{w} {per[w].mean():.1f}" for w in SYSTEMS if w in per))
                if c == 0.5:
                    key = f"B4 {variant}"
                    verdict[key] = dd.mean() <= -10 if variant == "published" else abs(dd.mean()) <= 5
                    line += f" -> {'met' if verdict[key] else 'NOT met'}"
                say(line)
    say()

    # I8 ─────────────────────────────────────────────────────────────────────
    say("I8  Secondary: share of nights flagged as injected activity grows (night-level dose-response),")
    say("    amplitude at which half of a system's nights are flagged (µV), by pattern")
    syn = df[df.pattern.isin(["tonic", "phasic"]) & df.who.isin(SYSTEMS) & df.primary]
    dose = (syn.groupby(["pattern", "method", "variant", "who", "amp_uv"]).y.mean()
            .rename("share").reset_index())
    dose.to_csv(config.RESULTS / "inject_dose_response.csv", index=False)
    for pattern in ("tonic", "phasic"):
        for meth, variant in DECIDING:
            cells = []
            for w in SYSTEMS:
                g = dose[(dose.pattern == pattern) & (dose.method == meth) &
                         (dose.variant == variant) & (dose.who == w)].sort_values("amp_uv")
                if g.empty:
                    continue
                hit = g[g.share >= 0.5]
                cells.append(f"{w} " + ("-" if hit.empty else
                                        ("<=0.25" if hit.amp_uv.iloc[0] == g.amp_uv.iloc[0]
                                         else f"{hit.amp_uv.iloc[0]:.2f}")))
            say(f"    {pattern:<7}{meth + ' ' + variant:<20} " + "  ".join(cells))
    say()

    say("VERDICTS")
    for k, v in verdict.items():
        say(f"    {k:<32} {'met' if v else 'NOT met'}")
    (config.RESULTS / "inject_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    (config.RESULTS / "inject_verdicts.json").write_text(json.dumps(
        {k: bool(v) for k, v in verdict.items()}, indent=1))
    try:
        figure(a, t, dose)
    except Exception as e:                       # the numbers matter more than the figure
        print(f"figure failed: {type(e).__name__}: {e}", flush=True)


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE
# ══════════════════════════════════════════════════════════════════════════════

COLORS = {"CAP patients": "#1b9e77", "MGH": "#d95f02", "HMC": "#7570b3", "BOAS": "#e7298a",
          "DOD-H": "#66a61e", "DOD-O": "#e6ab02"}


def figure(a, t, dose):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    shown = (0.3, 0.4, 0.6, 0.8, 1, 1.5, 2, 3, 4, 6, 8, 10, 15)
    plain = FuncFormatter(lambda v, _: f"{v:g}" if any(np.isclose(v, t) for t in shown) else "")
    fig, ax = plt.subplots(2, 2, figsize=(11, 9))
    wn_path = config.RESULTS / "inject_whitenoise_a50.csv"
    wn = pd.read_csv(wn_path) if wn_path.exists() else None
    for k, (meth, measure, pattern, title) in enumerate(
            [("Montreal", "tonic", "tonic", "A  Montreal tonic (20-s epochs)"),
             ("SINBAR", "any", "phasic", "B  SINBAR \"any\" (3-s mini-epochs)")]):
        axk = ax[0, k]
        for variant, marker in (("published", "o"), ("FC", "s")):
            g = a[(a.method == meth) & (a.measure == measure) & (a.pattern == pattern) &
                  (a.variant == variant) & a.who.isin(SYSTEMS) & a.primary &
                  (a.chin_fs >= FC_MIN_FS) & (a.censored == 0)]
            if variant == "FC":
                g = g[fc_valid(g.chin_fs, g.level_uv)]
            for w in SYSTEMS:
                s = g[g.who == w]
                axk.scatter(s.level_uv, s.a50, s=9, marker=marker, alpha=0.45,
                            facecolors=COLORS[w] if variant == "FC" else "none",
                            edgecolors=COLORS[w], linewidths=0.6,
                            label=w if variant == "published" else None)
            if wn is not None:
                c = wn[(wn.method == meth) & (wn.measure == measure) & (wn.pattern == pattern) &
                       (wn.variant == variant) & (wn.sim_fs == 256)].groupby("sim_floor_uv")[
                    ["level_uv", "a50"]].mean()
                if variant == "FC":
                    c = c[c.level_uv <= FC_MAX_FLOOR]
                axk.plot(c.level_uv, c.a50, color="k", lw=1.4,
                         ls="-" if variant == "published" else "--",
                         label=f"white noise, {'published' if variant == 'published' else 'floor-corrected'}")
        axk.set_xscale("log")
        axk.set_yscale("log")
        for axis in (axk.xaxis, axk.yaxis):
            axis.set_major_formatter(plain)
            axis.set_minor_formatter(plain)
        axk.set_xlabel("quiet-REM level of the night as recorded (µV)")
        axk.set_ylabel("amplitude detected in half of injected units, A50 (µV)")
        axk.set_title(title + "\nopen = published, filled = floor-corrected", fontsize=10)
        axk.legend(fontsize=7, loc="upper left")
    # C: transplant sensitivity
    axc = ax[1, 0]
    if not t.empty:
        xs = np.arange(len(SYSTEMS))
        for j, (c, variant, color, off) in enumerate(
                [(1.0, "published", "#444444", -0.27), (1.0, "FC", "#1f78b4", -0.09),
                 (0.5, "published", "#999999", 0.09), (0.5, "FC", "#a6cee3", 0.27)]):
            for i, w in enumerate(SYSTEMS):
                g = t[(t.amp_uv == c) & (t.method == "Montreal") & (t.variant == variant) & (t.who == w)]
                if variant == "FC":
                    g = g[g.fc_ok]
                if g.empty:
                    continue
                p, lo, hi, n = share(g.y)
                axc.errorbar(i + off, 100 * p, yerr=[[100 * (p - lo)], [100 * (hi - p)]], fmt="o",
                             color=color, ms=5, capsize=2,
                             label=f"x{c:g} {'published' if variant == 'published' else 'floor-corrected'}"
                             if i == 0 else None)
        axc.set_xticks(xs)
        axc.set_xticklabels(SYSTEMS, fontsize=8)
        axc.set_ylim(0, 105)
        axc.set_ylabel("hybrid nights flagged by the Montreal criteria (%)")
        axc.set_title("C  RBD activity of 21 CAP patients added to each system's nights", fontsize=10)
        axc.legend(fontsize=7, loc="lower left")
    # D: dose-response, SINBAR any, phasic pattern
    axd = ax[1, 1]
    for variant, ls in (("published", "-"), ("FC", "--")):
        for w in SYSTEMS:
            g = dose[(dose.pattern == "phasic") & (dose.method == "SINBAR") &
                     (dose.variant == variant) & (dose.who == w)].sort_values("amp_uv")
            if not g.empty:
                axd.plot(g.amp_uv, 100 * g.share, ls=ls, color=COLORS[w], lw=1.2,
                         label=w if variant == "published" else None)
    axd.set_xscale("log")
    axd.xaxis.set_major_formatter(plain)
    axd.set_xlabel("amplitude of the injected phasic bursts (µV)")
    axd.set_ylabel("nights flagged by SINBAR \"any\" >= 18.2% (%)")
    axd.set_title("D  Nights flagged as injected bursts grow\nsolid = published, dashed = floor-corrected",
                  fontsize=10)
    axd.legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGURES / f"fig11_injection.{ext}", dpi=200)
    plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_inject_colab.sh
#!/bin/sh
# The injection experiment (paper/preregistration_injection.md) on a fresh Colab
# runtime. Every night's scores are copied to Drive as they are made, so a restart
# continues where it stopped. Run from the code directory, e.g. in Colab's terminal:
#   cd /content/rbd-rwa && setsid nohup sh run_inject_colab.sh > /content/inject.log 2>&1 < /dev/null &
# About 20 s of computing per night on Colab's 2-core runtime, plus the downloads
# (the Dreem sets come from Zenodo at a few MB/s). To split the work over two
# runtimes: PART=2 (the Dreem sets) on one, PART=1 (everything else, then the
# analysis once part 2 is on Drive) on the other; both need Drive.
PART=${PART:-all}
D=/content/drive/MyDrive/rbd-rwa
export RBD_DURABLE_DIR=/content/durable PYTHONUNBUFFERED=1
R=$RBD_DURABLE_DIR/results
F=$RBD_DURABLE_DIR/figures
mkdir -p "$R" "$F" "$D/results" "$D/figures" "$D/inject"
# the noise-ladder scores of the earlier run (criterion B2)
if [ -f "$D/results/ladder_scores.csv" ] && [ ! -f "$R/ladder_scores.csv" ]; then
  cp "$D/results/ladder_scores.csv" "$R/"
fi
I="python run_inject.py --workers 3 --mirror $D/inject"
$I --donors
if [ "$PART" != 2 ]; then
  $I --cohort cap
  $I --cohort hmc
  $I --cohort boas
  $I --cohort chal18 --every 3
fi
if [ "$PART" != 1 ]; then
  $I --cohort dodo
  $I --cohort dodh
fi
if [ "$PART" = 2 ]; then touch "$D/inject/part2.done"; echo PART2_DONE; exit 0; fi
if [ "$PART" = 1 ]; then
  while [ ! -f "$D/inject/part2.done" ]; do sleep 60; done
  $I --cohort dodo          # nothing left to compute: restores part 2's scores from Drive
  $I --cohort dodh
fi
# white-noise predictions on this runtime (the same seeds as the frozen file in paper/)
if [ ! -f "$R/inject_whitenoise_a50.csv" ]; then python sim_injection.py 2; fi
python run_inject_report.py
cp "$R"/inject_* "$D/results/"
cp "$F"/fig11_* "$D/figures/" 2>/dev/null
echo INJECT_DONE

In [ ]:
%%writefile /content/rbd-rwa/src/noise_arms.py
"""Noise that is not white: the ladder's harmonized steps with other kinds of noise.

The noise ladder (cohorts.extract_ladder_pack) added band-limited white noise,
which is stationary and spectrally flat; real recording noise is neither. Here
the same nights are raised to the same quiet-REM levels (1.5 and 2.0 uV, the
ladder's steps h150 and h200) with four kinds of noise (review M6;
paper/preregistration_revision.md, R1):

  white   the ladder's own seeded white noise (reproduces the ladder's copies)
  pink    1/f noise, passed through the same EMG filter
  drift   white noise whose level wanders slowly through the night: a log-normal
          random walk with a 10-min correlation time and SD 0.3 of the log
          level, as electrode impedance does
  hmc     recorded noise: the quietest REM seconds of 30 HMC nights (every fifth
          record), runs of at least 3 s below each night's 20th percentile of
          1-s REM RMS, filtered and heartbeat-cleaned as in the main pipeline,
          spliced with 20-ms cross-fades; resampled to the night's rate and
          passed through its filter. It carries HMC's real spectrum, mains
          residue and electrode artifacts.

Every kind is added after the night's own filtering and heartbeat removal, in the
whole night (REM and N2/N3), as the ladder's noise was.
"""
import json
import zlib
from fractions import Fraction

import numpy as np

from .config import ENV_FS
from .extract import bandpass, bin_envelope

KINDS = ("white", "pink", "drift", "hmc")
TARGETS_UV = (1.5, 2.0)
LIB_FS = 256.0
EPOCH_SEC = 30


def _unit(x, fs):
    """Scale to unit RMS, ignoring 5 s at each end (filter edges)."""
    e = int(5 * fs)
    core = x[e:-e] if len(x) > 4 * e else x
    return x / np.sqrt(np.mean(core * core))


def white(n, fs, mains, rec):
    """Exactly the ladder's track (same seed, same filter)."""
    rng = np.random.default_rng(zlib.crc32(rec.encode()))
    return _unit(bandpass(rng.normal(0.0, 1.0, n), fs, mains=mains)[0], fs)


def pink(n, fs, mains, rec):
    rng = np.random.default_rng(zlib.crc32(("pink/" + rec).encode()))
    spec = np.fft.rfft(rng.normal(0.0, 1.0, n))
    f = np.fft.rfftfreq(n, 1.0 / fs)
    spec[1:] /= np.sqrt(f[1:])
    spec[0] = 0.0
    return _unit(bandpass(np.fft.irfft(spec, n), fs, mains=mains)[0], fs)


def drift(n, fs, mains, rec, tau_s=600.0, sd=0.3):
    rng = np.random.default_rng(zlib.crc32(("drift/" + rec).encode()))
    w = bandpass(rng.normal(0.0, 1.0, n), fs, mains=mains)[0]
    secs = int(np.ceil(n / fs)) + 1
    a = np.exp(-1.0 / tau_s)
    z = np.empty(secs)
    z[0] = rng.normal(0.0, sd)
    steps = rng.normal(0.0, sd * np.sqrt(1 - a * a), secs)
    for i in range(1, secs):                       # Ornstein-Uhlenbeck in the log level
        z[i] = a * z[i - 1] + steps[i]
    level = np.exp(np.interp(np.arange(n) / fs, np.arange(secs), z))
    return _unit(w * level, fs)


def build_library(pieces_by_night, fs=LIB_FS, fade_s=0.02):
    """Splice the quiet runs of several nights into one unit-RMS track."""
    r = int(round(fade_s * fs))
    up = 0.5 - 0.5 * np.cos(np.pi * (np.arange(r) + 0.5) / r)
    out = np.zeros(0)
    for pieces in pieces_by_night:
        for p in pieces:
            p = np.asarray(p, float)
            if len(p) <= 2 * r:
                continue
            if len(out) >= r:
                out[-r:] = out[-r:] * up[::-1] + p[:r] * up
                out = np.r_[out, p[r:]]
            else:
                out = np.r_[out, p]
    return (out / np.sqrt(np.mean(out * out))).astype(np.float32)


def quiet_runs(xf, fs, hyp, pct=20.0, min_s=3):
    """Runs of at least `min_s` consecutive REM seconds whose 1-s RMS is below the
    night's `pct` percentile: the night's recorded noise with little muscle."""
    per = int(round(fs * EPOCH_SEC))
    hyp = np.asarray(hyp)[: len(xf) // per]
    sec_rms, _ = bin_envelope(xf, fs, env_fs=1)
    rem = np.repeat(hyp == 4, EPOCH_SEC)[: len(sec_rms)]
    if rem.sum() < 60:
        return []
    quiet = rem & (sec_rms <= np.percentile(sec_rms[rem], pct))
    edges = np.diff(np.r_[0, quiet.astype(np.int8), 0])
    starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    out = []
    for a, b in zip(starts, stops):
        if b - a >= min_s:
            out.append(xf[int(a * fs): int(b * fs)].astype(np.float32))
    return out


def hmc(n, fs, mains, rec, library):
    """The recorded-noise library at this night's rate and filter, tiled from a
    seeded random offset to the night's length."""
    from scipy.signal import resample_poly
    rng = np.random.default_rng(zlib.crc32(("hmc/" + rec).encode()))
    lib = np.asarray(library, float)
    f = Fraction(int(round(fs)), int(LIB_FS)).limit_denominator(1000)
    if f != 1:
        lib = resample_poly(lib, f.numerator, f.denominator)
    start = int(rng.integers(0, len(lib)))
    reps = int(np.ceil((n + start) / len(lib)))
    x = np.tile(lib, reps)[start:start + n]
    return _unit(bandpass(x, fs, mains=mains)[0], fs)


def harmonized(xf, track, fs, hyp, target):
    """Envelopes of xf + s * track with s chosen so that the quiet-REM level is
    within 1% of `target` (the ladder's procedure). Returns (rms, mav, s, level)."""
    from .cohorts import quiet_level
    rms, mav = bin_envelope(xf, fs)
    level = quiet_level(rms, hyp)
    s, got = float(np.sqrt(max(target * target - level * level, 0.0))), level
    r, m = rms, mav
    for _ in range(5):
        if s <= 0:
            break
        r, m = bin_envelope(xf + s * track, fs)
        got = quiet_level(r, hyp)
        if abs(got - target) <= 0.01 * target:
            break
        s *= np.sqrt((target * target - level * level) / max(got * got - level * level, 1e-9))
    return r, m, s, got


def save_library(path, lib, meta):
    np.savez_compressed(path, lib=lib, meta=json.dumps(meta))


def load_library(path):
    z = np.load(path, allow_pickle=False)
    return z["lib"], json.loads(str(z["meta"]))

In [ ]:
%%writefile /content/rbd-rwa/run_noise_arms.py
"""The ladder's harmonized steps with pink, drifting and recorded noise (review M6).

Pre-specified in paper/preregistration_revision.md (R1, R2). The same nights as the
noise ladder (the first 300 MGH records; CAP's nights in microvolts) are streamed
once more; each is raised to a quiet-REM level of 1.5 and 2.0 uV with each kind of
noise in src/noise_arms.py and scored with the published rules and with the
floor-corrected rules under three noise estimates (REM as pre-specified, N2/N3, and
the lower of the two). Only scores are kept.

    python run_noise_arms.py --library           # recorded-noise library from 30 HMC nights
    python run_noise_arms.py --cohort chal18     # the ladder's 300 MGH nights
    python run_noise_arms.py --cohort cap        # CAP (RBD and same-system patients)
    python run_noise_arms.py --report
-> DURABLE/noise_arms/<cohort>/<record>.csv.gz, results/noise_arms_scores.csv,
   results/noise_arms_report.md
"""
import argparse
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config

OUT = config.DURABLE / "noise_arms"
LIB = config.DURABLE / "noise_library.npz"
REF_UV = 1.0


def variants():
    from src.rwa import METHODS, floor_corrected
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out.append((m.name, "published", m))
        for src in ("rem", "nrem", "min"):
            out.append((m.name, "FC" if src == "rem" else f"FC-{src}",
                        floor_corrected(m, REF_UV, noise_from=src)))
    out.append(("AASM", "published", METHODS["aasm"]))
    return out


def _library_work(rec, scratch):
    from src.cohorts import COHORTS, _beats
    from src.ecg import subtract_heartbeat
    from src.extract import bandpass
    from src.noise_arms import quiet_runs
    cohort = COHORTS["hmc"]
    folder = Path(scratch) / f"lib_{rec}"
    folder.mkdir(parents=True, exist_ok=True)
    try:
        d = cohort.read(rec, cohort.fetch(rec, folder))
        fs = float(d["fs"])
        if fs != 256.0:
            return rec, None, f"rate {fs}"
        xf, _ = bandpass(d["chin"], fs, mains=cohort.mains)
        beats = _beats(d)
        if len(beats):
            xf = subtract_heartbeat(xf, fs, beats)
        return rec, quiet_runs(xf, fs, d["hypnogram"]), None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e}"
    finally:
        shutil.rmtree(folder, ignore_errors=True)


def build_library(workers):
    from src.cohorts import COHORTS
    from src.noise_arms import build_library as splice, save_library
    recs = [r for r in COHORTS["hmc"].records() if r not in ("SN014", "SN064", "SN135")][::5][:30]
    pieces, used = [], []
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for f in as_completed([ex.submit(_library_work, r, str(config.SCRATCH)) for r in recs]):
            rec, runs, err = f.result()
            if err or not runs:
                print(f"  {rec}: skipped ({err or 'no quiet REM runs'})", flush=True)
                continue
            pieces.append((rec, runs))
            used.append(rec)
            print(f"  {rec}: {sum(len(p) for p in runs) / 256:.0f} s of quiet REM", flush=True)
    pieces.sort()
    lib = splice([p for _, p in pieces])
    save_library(LIB, lib, {"nights": sorted(used), "seconds": round(len(lib) / 256.0, 1)})
    print(f"library: {len(used)} nights, {len(lib) / 256 / 60:.1f} min -> {LIB}", flush=True)


def _work(name, rec, scratch, out_path):
    import warnings
    warnings.filterwarnings("ignore")
    from src import noise_arms as na
    from src.cohorts import COHORTS, _beats, quiet_level
    from src.config import ENV_FS
    from src.ecg import subtract_heartbeat
    from src.extract import bandpass, bin_envelope
    from src.pack import build_pack
    from src.rwa import rem_atonia_index, score_night
    cohort = COHORTS[name]
    folder = Path(scratch) / f"arms_{name}_{rec}"
    folder.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    try:
        d = cohort.read(rec, cohort.fetch(rec, folder))
        if not d["unit_known"]:
            raise ValueError("chin EMG has no physical units")
        fs, hyp = float(d["fs"]), np.asarray(d["hypnogram"])
        xf, _ = bandpass(d["chin"], fs, mains=cohort.mains)
        beats = _beats(d)
        if len(beats):
            xf = subtract_heartbeat(xf, fs, beats)
        rms, mav = bin_envelope(xf, fs)
        chin = {"rms": rms, "mav": mav}
        added = {"native": (0.0, quiet_level(rms, hyp))}
        lib = na.load_library(LIB)[0] if LIB.exists() else None
        for kind in na.KINDS:
            if kind == "hmc" and lib is None:
                continue
            track = na.hmc(len(xf), fs, cohort.mains, rec, lib) if kind == "hmc" \
                else getattr(na, kind)(len(xf), fs, cohort.mains, rec)
            for t in na.TARGETS_UV:
                r, m, s, got = na.harmonized(xf, track, fs, hyp, t)
                band = f"{kind}_h{round(t * 100):03d}"
                chin[f"rms_{band}"], chin[f"mav_{band}"] = r, m
                added[band] = (s, got)
            del track
        pack = build_pack(chin, [], hyp, ENV_FS, EPOCH_SEC_, {"record": rec, "unit_known": True})
        rows = []
        info = {"cohort": name, "record": rec, "chin_fs": fs, "beats": int(len(beats))}
        for band, (s, lvl) in added.items():
            for meth, variant, m in variants():
                rows.append({**info, "band": band, "added_uv": round(s, 4), "level_uv": round(lvl, 4),
                             **score_night(pack, m, band), "variant": variant})
            rows.append({**info, "band": band, "added_uv": round(s, 4), "level_uv": round(lvl, 4),
                         **rem_atonia_index(pack, band), "variant": "published"})
        out = Path(out_path)
        tmp = out.with_name(out.name.replace(".csv.gz", ".tmp.csv.gz"))
        pd.DataFrame(rows).to_csv(tmp, index=False, compression="gzip")
        tmp.replace(out)
        return rec, round(time.time() - t0, 1), None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=2)[-400:]}"
    finally:
        shutil.rmtree(folder, ignore_errors=True)


EPOCH_SEC_ = 30


def records(name):
    from src.cohorts import COHORTS
    recs = COHORTS[name].records()
    return recs[:300] if name == "chal18" else recs


def run_cohort(name, workers, mirror):
    out_dir = OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    m = Path(mirror) / "noise_arms" / name if mirror else None
    if m:
        try:
            m.mkdir(parents=True, exist_ok=True)
            for p in m.glob("*.csv.gz"):
                if not (out_dir / p.name).exists():
                    shutil.copy2(p, out_dir / p.name)
        except OSError as e:
            print(f"(mirror unavailable: {e})", flush=True)
            m = None
    todo = [r for r in records(name) if not (out_dir / f"{r}.csv.gz").exists()]
    print(f"{name}: {len(todo)} nights to process", flush=True)
    with ProcessPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_work, name, r, str(config.SCRATCH), str(out_dir / f"{r}.csv.gz")) for r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            rec, secs, err = f.result()
            if err:
                print(f"[{i}/{len(todo)}] {rec} FAILED {err[:300]}", flush=True)
                continue
            if m:
                try:
                    shutil.copy2(out_dir / f"{rec}.csv.gz", m / f"{rec}.csv.gz")
                except OSError:
                    pass
            print(f"[{i}/{len(todo)}] {rec} {secs}s", flush=True)


def report():
    from run_report import MIN_REM, wilson
    from run_score import group_of
    files = [p for c in ("chal18", "cap") for p in sorted((OUT / c).glob("*.csv.gz"))
             if ".tmp" not in p.name]
    df = pd.concat((pd.read_csv(p) for p in files), ignore_index=True)
    df["group"] = [group_of(c, r) for c, r in zip(df.cohort, df.record)]
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512)
    df["who"] = np.select([same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"]),
                           df.cohort == "chal18"], ["CAP RBD", "CAP patients", "MGH"], default="")
    df = df[df.who != ""]
    # one set of nights throughout: >= 10 min REM in every copy
    ok = df.groupby(["cohort", "record"]).rem_minutes.min() >= MIN_REM
    keep = ok[ok].index
    df = df.set_index(["cohort", "record"]).loc[keep].reset_index()
    df["y"] = df.positive.map(lambda x: float(x in (True, "True", 1, 1.0)) if pd.notna(x) else np.nan)
    df.to_csv(config.RESULTS / "noise_arms_scores.csv", index=False)
    lines = []
    say = lambda s="": (print(s), lines.append(s))
    say("N1  Share flagged (%) by kind of noise at a quiet-REM level of 1.5 uV (h150) and 2.0 uV;")
    say("    the same nights (>= 10 min REM in every copy); native = as recorded")
    bands = ["native"] + [f"{k}_h{t}" for t in ("150", "200") for k in ("white", "pink", "drift", "hmc")]
    for who in ("MGH", "CAP RBD", "CAP patients"):
        g = df[df.who == who]
        say(f"  {who} (n={g.record.nunique()})  level: " + "  ".join(
            f"{b} {g[(g.band == b) & (g.method == 'Montreal') & (g.variant == 'published')].level_uv.median():.2f}"
            for b in bands if (g.band == b).any()))
        for meth, variant in (("Montreal", "published"), ("SINBAR", "published"), ("RAI", "published"),
                              ("Montreal", "FC"), ("SINBAR", "FC"), ("Montreal", "FC-nrem"),
                              ("SINBAR", "FC-nrem"), ("Montreal", "FC-min"), ("SINBAR", "FC-min")):
            s = g[(g.method == meth) & (g.variant == variant)]
            cells = "  ".join(f"{b} {100 * s[s.band == b].y.mean():5.1f}" for b in bands if (s.band == b).any())
            say(f"    {meth + ' ' + variant:<18} {cells}")
    say()
    say("N2  Paired differences from white noise at the same step (points; 95% bootstrap CI over nights)")
    rng = np.random.default_rng(4)
    for who in ("MGH", "CAP RBD", "CAP patients"):
        for meth, variant in (("Montreal", "published"), ("SINBAR", "published"),
                              ("Montreal", "FC"), ("SINBAR", "FC")):
            s = df[(df.who == who) & (df.method == meth) & (df.variant == variant)]
            w = s.pivot_table(index="record", columns="band", values="y")
            cells = []
            for t in ("150", "200"):
                for k in ("pink", "drift", "hmc"):
                    a, b = f"{k}_h{t}", f"white_h{t}"
                    if a not in w or b not in w:
                        continue
                    d = (w[a] - w[b]).dropna().to_numpy()
                    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
                    cells.append(f"{k}{t} {100 * d.mean():+5.1f} [{100 * np.percentile(bs, 2.5):+.1f},"
                                 f"{100 * np.percentile(bs, 97.5):+.1f}]")
            say(f"  {who:<13}{meth + ' ' + variant:<18} " + "  ".join(cells))
    say()
    (config.RESULTS / "noise_arms_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--library", action="store_true")
    ap.add_argument("--cohort", choices=["chal18", "cap"])
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--mirror", default=None, help="Drive folder (e.g. MyDrive/rbd-rwa)")
    ap.add_argument("--report", action="store_true")
    args = ap.parse_args()
    if args.library:
        if args.mirror and (Path(args.mirror) / LIB.name).exists() and not LIB.exists():
            shutil.copy2(Path(args.mirror) / LIB.name, LIB)
        if not LIB.exists():
            build_library(args.workers)
            if args.mirror:
                shutil.copy2(LIB, Path(args.mirror) / LIB.name)
    if args.cohort:
        run_cohort(args.cohort, args.workers, args.mirror)
    if args.report:
        report()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_revision.py
"""Analyses requested by the pre-submission review (paper/revision_plan_2026-10-27.md),
specified in paper/preregistration_revision.md (R2-R4, R6) before they were computed.

From the packs (every night as recorded) and the score files of the earlier runs:

  V1  every night re-scored: published rules, floor-corrected with the pre-specified
      REM noise estimate, with the N2/N3 estimate, and with the lower of the two
  V2  Table 3 recomputed from counts (k/n, Wilson intervals)
  V3  between-site differences with 95% Newcombe intervals and TOST equivalence
      (90% interval within +/-10 and +/-5 points), all nights and in-range nights
  V4  the noise estimators: RBD sensitivity, same-system false positives, AUC
  V5  six-group rank correlations with n and exact permutation p
  V6  CAP: every night by group and recording system (105 analyzed, 88 in Table 1)
  V7  narcolepsy excluded; patients aged 50 or over
  V8  the MGH ladder on one set of nights (as recorded and every step)
  V9  AASM's continuous measures along the ladder

    python run_revision.py -> results/revision_scores.csv, results/revision_report.md
"""
import itertools
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr

from src import config
from src.pack import load_pack
from run_report import MIN_REM, WITHDRAWN, wilson
from run_score import auc, group_of

COHORTS = ("cap", "chal18", "hmc", "boas", "dodh", "dodo")
NAME = {"chal18": "MGH", "hmc": "HMC", "boas": "BOAS", "dodh": "DOD-H", "dodo": "DOD-O"}
GROUPS = ["CAP RBD", "CAP patients", "MGH", "HMC", "BOAS", "DOD-O", "DOD-H"]
WITHOUT_RBD = ["CAP patients", "MGH", "HMC", "BOAS", "DOD-O", "DOD-H"]
FC_MAX_FLOOR, FC_MIN_FS = 1.69, 200.0
CUT_FC_SINBAR = 49.1
YES = (True, "True", 1, 1.0)
RULES = [("Montreal", "published"), ("Montreal", "FC"), ("Montreal", "FC-nrem"), ("Montreal", "FC-min"),
         ("SINBAR", "published"), ("SINBAR", "FC"), ("SINBAR", "FC-nrem"), ("SINBAR", "FC-min"),
         ("RAI", "published")]


# ══════════════════════════════════════════════════════════════════════════════
#  STATISTICS
# ══════════════════════════════════════════════════════════════════════════════

def newcombe(k1, n1, k2, n2, z=1.96):
    """Difference of two independent proportions with Newcombe's hybrid score
    interval (method 10)."""
    p1, l1, u1 = wilson(k1, n1, z)
    p2, l2, u2 = wilson(k2, n2, z)
    d = p1 - p2
    return d, d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)


def tost(k1, n1, k2, n2, margin):
    """Two one-sided tests of |p1 - p2| < margin: equivalence at 5% when the 90%
    Newcombe interval lies inside (-margin, margin); p from the Wald statistic."""
    d, lo, hi = newcombe(k1, n1, k2, n2, z=1.6449)
    p1, p2 = k1 / n1, k2 / n2
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    p = max(1 - norm.cdf((d + margin) / se), 1 - norm.cdf((margin - d) / se)) if se > 0 else np.nan
    return bool(-margin < lo and hi < margin), float(p), lo, hi


def spearman_exact(x, y):
    """Spearman's rho with an exact two-sided permutation p (small n)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    rho = spearmanr(x, y).correlation
    perms = [spearmanr(x, np.asarray(p)).correlation for p in itertools.permutations(y)]
    p = np.mean(np.abs(perms) >= abs(rho) - 1e-12)
    return float(rho), float(p), len(x)


def mcnemar_exact(b, c):
    from scipy.stats import binomtest
    return binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0


def paired_ci(a, b, n_boot=2000, seed=5):
    d = np.asarray(a, float) - np.asarray(b, float)
    rng = np.random.default_rng(seed)
    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)


# ══════════════════════════════════════════════════════════════════════════════
#  SCORING
# ══════════════════════════════════════════════════════════════════════════════

def variants():
    from src.rwa import METHODS, floor_corrected
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out.append(("published", m))
        out += [("FC" if s == "rem" else f"FC-{s}", floor_corrected(m, 1.0, noise_from=s))
                for s in ("rem", "nrem", "min")]
    out.append(("published", METHODS["aasm"]))
    return out


def score_file(path):
    from run_floor import _info, scale
    from src.rwa import rem_atonia_index, score_night
    pack = load_pack(path)
    m = pack["meta"]
    info = _info(m, m.get("cohort", path.parent.name))
    info.update(scale(pack))
    rows = [{**info, **score_night(pack, meth, "native"), "variant": v} for v, meth in variants()]
    rows.append({**info, **rem_atonia_index(pack, "native"), "variant": "published"})
    return rows


def restore_packs():
    """Copy packs from Drive (MyDrive/rbd-rwa/packs) to the working disk if needed."""
    import shutil
    drive = config.DRIVE / "rbd-rwa" / "packs" if config.DRIVE else None
    for c in COHORTS:
        d = config.DURABLE / "packs" / c
        d.mkdir(parents=True, exist_ok=True)
        if drive and (drive / c).exists():
            for p in (drive / c).glob("*.npz"):
                if not (d / p.name).exists():
                    shutil.copy2(p, d / p.name)


def label(df):
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record.astype(str)).isin(gone)].copy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select([same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"]),
                           cap & (df.group == "healthy")], ["CAP RBD", "CAP patients", "CAP healthy"],
                          default=df.cohort.map(NAME).fillna(""))
    df["primary"] = True
    b = df.cohort == "boas"
    if b.any():
        num = df.loc[b, "record"].astype(str).str.extract(r"(\d+)")[0].astype(int)
        pid = df.loc[b, "pid"].fillna(df.loc[b, "record"]).astype(str)
        df.loc[b, "primary"] = (num == num.groupby(pid).transform("min")).to_numpy()
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    lvl = df[(df.method == "Montreal") & (df.variant == "published")] \
        .set_index(["cohort", "record"]).background_phasic_uV
    df["level"] = lvl.reindex(pd.MultiIndex.from_frame(df[["cohort", "record"]])).to_numpy()
    df["in_range"] = (df.chin_fs >= FC_MIN_FS) & (df.level <= FC_MAX_FLOOR)
    return df


def flagged(df, who, meth, variant, cut=None, in_range=False):
    g = df[(df.who == who) & (df.method == meth) & (df.variant == variant) & df.primary
           & (df.rem_minutes >= MIN_REM)]
    if in_range:
        g = g[g.in_range]
    if cut is not None:
        v = (g.any_pct >= cut).astype(float)
        return int(v.sum()), int(v.notna().sum())
    v = g.y.dropna()
    return int(v.sum()), len(v)


# ══════════════════════════════════════════════════════════════════════════════

def main():
    restore_packs()
    files = [p for c in COHORTS for p in sorted((config.DURABLE / "packs" / c).glob("*.npz"))
             if ".tmp" not in p.name]
    print(f"scoring {len(files)} packs", flush=True)
    with ProcessPoolExecutor(max_workers=2) as ex:
        rows = [r for part in ex.map(score_file, files, chunksize=8) for r in part]
    df = label(pd.DataFrame(rows))
    df.to_csv(config.RESULTS / "revision_scores.csv", index=False)
    lines = []
    say = lambda s="": (print(s), lines.append(s))

    # V2 ─────────────────────────────────────────────────────────────────────
    say("V2  Share flagged as recorded: k/n, % [95% Wilson]")
    for who in GROUPS:
        cells = []
        for meth, variant in RULES:
            k, n = flagged(df, who, meth, variant)
            if n:
                p, lo, hi = wilson(k, n)
                cells.append(f"{meth[0]}{variant.replace('published', 'pub')} {k}/{n} {100 * p:.1f} "
                             f"[{100 * lo:.0f}-{100 * hi:.0f}]")
        k, n = flagged(df, who, "SINBAR", "FC", cut=CUT_FC_SINBAR)
        if n:
            p, lo, hi = wilson(k, n)
            cells.append(f"SFC>=49.1 {k}/{n} {100 * p:.1f} [{100 * lo:.0f}-{100 * hi:.0f}]")
        say(f"  {who}: " + ";  ".join(cells))
    say()

    # V3 ─────────────────────────────────────────────────────────────────────
    say("V3  Between-site differences, points: estimate [95% Newcombe]; TOST at +/-10 and +/-5")
    say("    (equivalent = 90% interval inside the margin); FC rows also on in-range nights only")
    say("    (>= 200 Hz and quiet-REM level <= 1.69 uV)")
    pairs = [("MGH", "HMC"), ("BOAS", "DOD-H"), ("DOD-O", "MGH"), ("CAP patients", "MGH")]
    out = []
    for a, b in pairs:
        for meth, variant in RULES + [("SINBAR", "FC>=49.1")]:
            for rng_only in ((False, True) if variant.startswith("FC") else (False,)):
                cut = CUT_FC_SINBAR if variant == "FC>=49.1" else None
                v = "FC" if cut else variant
                k1, n1 = flagged(df, a, meth, v, cut, rng_only)
                k2, n2 = flagged(df, b, meth, v, cut, rng_only)
                if not (n1 and n2):
                    continue
                d, lo, hi = newcombe(k1, n1, k2, n2)
                e10, p10, _, _ = tost(k1, n1, k2, n2, 0.10)
                e5, p5, _, _ = tost(k1, n1, k2, n2, 0.05)
                out.append(dict(pair=f"{a} - {b}", method=meth, variant=variant, in_range=rng_only,
                                n1=n1, n2=n2, diff=100 * d, lo=100 * lo, hi=100 * hi,
                                equiv10=e10, p10=p10, equiv5=e5, p5=p5))
                say(f"    {a + ' - ' + b:<20}{meth + ' ' + variant:<20}{'in-range' if rng_only else 'all':<9}"
                    f"n {n1}/{n2}  {100 * d:+5.1f} [{100 * lo:+.1f} to {100 * hi:+.1f}]  "
                    f"TOST10 {'equivalent' if e10 else 'not shown'} (p {p10:.3f})  "
                    f"TOST5 {'equivalent' if e5 else 'not shown'} (p {p5:.3f})")
    pd.DataFrame(out).to_csv(config.RESULTS / "revision_site_differences.csv", index=False)
    say("    nights outside FC's range: " + "  ".join(
        f"{w} {100 * (1 - df[(df.who == w) & df.primary & (df.method == 'Montreal') & (df.variant == 'FC') & (df.rem_minutes >= MIN_REM)].in_range.mean()):.0f}%"
        for w in GROUPS))
    say()

    # V4 ─────────────────────────────────────────────────────────────────────
    say("V4  Noise estimate of the floor-corrected rule (B = 1.0 uV): REM (pre-specified), N2/N3, lower")
    for meth in ("Montreal", "SINBAR"):
        for variant in ("published", "FC", "FC-nrem", "FC-min"):
            k1, n1 = flagged(df, "CAP RBD", meth, variant)
            k2, n2 = flagged(df, "CAP patients", meth, variant)
            g = df[(df.method == meth) & (df.variant == variant) & (df.rem_minutes >= MIN_REM)]
            measures = ["any_pct"] if meth == "SINBAR" else ["tonic_pct", "phasic_pct"]
            aucs = "  ".join(f"AUC {mm} {auc(g[g.who == 'CAP RBD'][mm], g[g.who == 'CAP patients'][mm])[0]:.2f}"
                             for mm in measures)
            say(f"    {meth + ' ' + variant:<20} RBD {k1}/{n1} ({100 * k1 / n1:.0f}%)  same-system patients "
                f"{k2}/{n2} ({100 * k2 / n2:.0f}%)  {aucs}")
    nb = df[(df.method == "Montreal") & (df.variant == "FC-nrem") & (df.rem_minutes >= MIN_REM)]
    say("    noise basis (N2/N3 quiet level) median by group: " + "  ".join(
        f"{w} {nb[nb.who == w].noise_basis_uV.median():.2f}" for w in GROUPS)
        + "   REM quiet level: " + "  ".join(
        f"{w} {nb[nb.who == w].level.median():.2f}" for w in GROUPS))
    say()

    # V5 ─────────────────────────────────────────────────────────────────────
    say("V5  Six groups without RBD: rank correlation of median floor and share flagged")
    floors = [df[(df.who == w) & df.primary & (df.method == "Montreal") & (df.variant == "published")
                 & (df.rem_minutes >= MIN_REM)].level.median() for w in WITHOUT_RBD]
    for meth, variant in RULES:
        shares = []
        for w in WITHOUT_RBD:
            k, n = flagged(df, w, meth, variant)
            shares.append(k / n if n else np.nan)
        rho, p, n = spearman_exact(floors, shares)
        say(f"    {meth + ' ' + variant:<20} rho {rho:+.2f}  exact p {p:.3f}  (n = {n} groups)")
    say()

    # V6 ─────────────────────────────────────────────────────────────────────
    say("V6  CAP nights by group and recording system (>= 10 min REM)")
    cap = df[(df.cohort == "cap") & (df.method == "Montreal") & (df.variant == "published")]
    cap = cap.assign(system=np.where(cap.unit_known.astype(bool), cap.chin_fs.map(lambda f: f"{f:g} Hz"),
                                     "no units"), scored=cap.rem_minutes >= MIN_REM)
    say(cap.pivot_table(index="group", columns="system", values="scored", aggfunc="sum",
                        fill_value=0).to_string())
    say(f"    scored in total: {int(cap.scored.sum())}")
    say()

    # V7 ─────────────────────────────────────────────────────────────────────
    say("V7  Without narcolepsy, and patients aged 50 or over (CAP's main system)")
    from src.fetch import cap_demographics
    ages = {k: v.get("age") for k, v in cap_demographics().items()}
    g = df[df.who.isin(["CAP RBD", "CAP patients"]) & (df.rem_minutes >= MIN_REM)].copy()
    g["age"] = g.record.map(ages)
    for label_, sel in (("all", g), ("no narcolepsy", g[g.group != "narcolepsy"]),
                        ("patients >= 50", g[(g.who == "CAP RBD") | (g.age >= 50)])):
        cells = []
        for meth, variant, mm in (("Montreal", "published", "tonic_pct"), ("Montreal", "published", "phasic_pct"),
                                  ("SINBAR", "published", "any_pct"), ("Montreal", "FC", "tonic_pct"),
                                  ("SINBAR", "FC", "any_pct")):
            s = sel[(sel.method == meth) & (sel.variant == variant)]
            a, n1, n0 = auc(s[s.who == "CAP RBD"][mm], s[s.who == "CAP patients"][mm])
            fp = s[s.who == "CAP patients"].y.mean()
            cells.append(f"{meth[0]}{variant[:3]} {mm[:5]} AUC {a:.2f} (FP {100 * fp:.0f}%)")
        say(f"    {label_:<15} n patients {sel[sel.who == 'CAP patients'].record.nunique():3d}  " + "; ".join(cells))
    say()

    # V8, V9 ─────────────────────────────────────────────────────────────────
    lad_path, flad_path = config.RESULTS / "ladder_scores.csv", config.RESULTS / "floor_ladder_scores.csv"
    if lad_path.exists() and flad_path.exists():
        lad = pd.read_csv(lad_path, low_memory=False)
        flad = pd.read_csv(flad_path, low_memory=False)
        mg = lad[(lad.who == "MGH")]
        rem_ok = mg.groupby("record").rem_minutes.min() >= MIN_REM
        steps_present = mg.groupby("record").step.nunique()
        nat = df[(df.who == "MGH") & (df.method == "Montreal") & (df.variant == "published")]
        nat_ok = set(nat[nat.rem_minutes >= MIN_REM].record.astype(str))
        common = sorted(set(rem_ok[rem_ok].index.astype(str)) & nat_ok
                        & set(steps_present[steps_present == steps_present.max()].index.astype(str)))
        say(f"V8  MGH ladder: {mg.record.nunique()} records; >= 10 min REM at every step: {int(rem_ok.sum())}; "
            f"in the as-recorded scores with >= 10 min REM: {len(set(mg.record.astype(str)) & nat_ok)}; "
            f"common set: {len(common)}")
        dropped = sorted(set(mg.record.astype(str)) - set(common))
        say(f"    left out: {', '.join(dropped[:20])}" + (" ..." if len(dropped) > 20 else ""))
        for name_, src, var in (("published", lad, None), ("FC", flad, "FC")):
            s = src[(src.who == "MGH") & src.record.astype(str).isin(common)]
            if var:
                s = s[(s.variant == var) & np.isclose(s.ref_uv.fillna(0), 1.0)]
            for meth in ("Montreal", "SINBAR", "RAI"):
                w = s[s.method == meth].pivot_table(index="record", columns="step", values="y")
                if "native" not in w or "h150" not in w:
                    continue
                w = w.dropna(subset=["native", "h150"])
                b_ = int(((w.native == 1) & (w.h150 == 0)).sum())
                c_ = int(((w.native == 0) & (w.h150 == 1)).sum())
                d, lo, hi = paired_ci(w.h150, w.native)
                say(f"    {meth + ' ' + name_:<18} n={len(w)}  {100 * w.native.mean():.1f}% -> {100 * w.h150.mean():.1f}% "
                    f"at h150 ({100 * d:+.1f} [{100 * lo:+.1f} to {100 * hi:+.1f}]; {b_} pos->neg, {c_} neg->pos; "
                    f"McNemar p {mcnemar_exact(b_, c_):.1e})  steps: " + "  ".join(
                        f"{st} {100 * w[st].mean():.1f}" for st in
                        ("n050", "n100", "n150", "n200", "n300", "h200") if st in w))
        say()
        say("V9  AASM along the ladder: mean tonic % and phasic-epoch % (no published cut-off)")
        a = lad[(lad.method == "AASM") & (lad.rem_minutes >= MIN_REM)]
        for who in ("CAP RBD", "CAP patients", "MGH", "HMC"):
            s = a[a.who == who]
            if s.empty:
                continue
            say(f"    {who:<13}" + "  ".join(
                f"{st} {s[s.step == st].tonic_pct.mean():.0f}/{s[s.step == st].phasic_epoch_pct.mean():.0f}"
                for st in ("native", "n050", "n100", "n150", "n200", "n300", "h150", "h200")
                if (s.step == st).any()))
        say()
    else:
        say("V8, V9: ladder score files not found in results/ (copy them from Drive)")

    (config.RESULTS / "revision_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_rbdtector_checks.py
"""RBDtector checks asked for in review (paper/preregistration_revision.md, R5).

  verify      12 nights (6 CAP RBD, 3 CAP same-system patients, 3 MGH) scored twice:
              by our runner (vectorized find_rem_blocks, input trimmed to the REM
              stretch, 512 Hz decimated to 256 Hz, floor-corrected run reusing the
              published baseline) and by the program as released (its own
              find_rem_blocks, the whole night at the native rate, baseline
              recomputed). The tonic, phasic and "any" percentages and the baseline
              must agree.
  mgh         the 100 MGH nights of the Parkinson's analysis, each downloaded once and
              scored as before (heartbeat in, no arousals), with the arousals scored
              in the Challenge 2018 annotations passed to RBDtector (as its published
              cut-off assumes), and with the heartbeat removed from the chin EMG
  boas        the BOAS first nights as before and with the heartbeat removed (BOAS
              has no ECG; beats are found on the chin EMG, where detection passes)

    python run_rbdtector_checks.py --part verify|mgh|boas
    python run_rbdtector_checks.py --report
-> results/rbdtector_checks_<part>.csv (one row per night, check and variant),
   results/rbdtector_checks_report.md
"""
import argparse
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from run_rbdtector import records_for

PARTS = ("verify", "mgh", "boas")
out_file = lambda part: config.RESULTS / f"rbdtector_checks_{part}.csv"
VERIFY = {"cap": ["rbd1", "rbd2", "rbd3", "rbd4", "rbd5", "rbd6", "nfle1", "plm1", "ins2"]}


def _work(name, rec, part, scratch):
    import warnings
    warnings.filterwarnings("ignore")
    from src import rbdtector as rt
    from src.cohorts import COHORTS, _beats
    from src.ecg import subtract_heartbeat
    from src.extract import bandpass
    rt.install()
    cohort = COHORTS[name]
    folder = Path(scratch) / f"rbdtc_{name}_{rec}"
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    try:
        d = cohort.read(rec, cohort.fetch(rec, folder))
        if not d["unit_known"]:
            raise ValueError("chin EMG has no physical units")
        fs, hyp = float(d["fs"]), np.asarray(d["hypnogram"])
        xf, _ = bandpass(d["chin"], fs, mains=cohort.mains)
        info = {"cohort": name, "record": rec, "fs": fs}

        def score(check, x, f, trim=True, fast=True, cache=True, arousals=None):
            rt.set_mode(fast=fast, cache=cache)
            sub = rt.write_inputs(folder / check, x, f, hyp, trim=trim, arousals=arousals)
            for variant in ("published", "FC"):
                t1 = time.time()
                r = rt.run(sub, variant)
                rows.append({**info, "check": check, "variant": variant,
                             "seconds": round(time.time() - t1, 1), **r})
            shutil.rmtree(sub, ignore_errors=True)

        half = (xf[::2], fs / 2) if fs == 512 else (xf, fs)
        if part == "verify":
            score("modified", *half)                               # as in run_rbdtector.py
            score("released", xf, fs, trim=False, fast=False, cache=False)
        else:
            beats = _beats(d)
            ev = d.get("events", {}).get("arousal", [])
            info.update(beats=int(len(beats)), n_arousals=len(ev))
            score("as_before", *half)
            if part == "mgh":
                score("arousals", *half, arousals=ev)
            if len(beats):
                xc = subtract_heartbeat(xf, fs, beats)
                score("heartbeat_removed", *((xc[::2], fs / 2) if fs == 512 else (xc, fs)))
        rt.set_mode()
        return rec, rows, None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=3)[-600:]}"
    finally:
        shutil.rmtree(folder, ignore_errors=True)


def jobs(part):
    if part == "verify":
        mgh, _ = records_for("chal18", 100)
        return [("cap", r) for r in VERIFY["cap"]] + [("chal18", r) for r in mgh[:3]]
    if part == "mgh":
        mgh, _ = records_for("chal18", 100)
        return [("chal18", r) for r in mgh]
    boas, _ = records_for("boas", 0)
    return [("boas", r) for r in boas]


def report():
    df = pd.concat([pd.read_csv(out_file(p), dtype={"record": str}).assign(part=p)
                    for p in PARTS if out_file(p).exists()], ignore_index=True)
    lines = []
    say = lambda s="": (print(s), lines.append(s))
    cols = ["tonic_pct", "phasic_pct", "any_pct", "baseline_uv"]
    v = df[df.part == "verify"]
    if not v.empty:
        say("V  Our runner against RBDtector as released (same nights; max |difference|)")
        w = v.pivot_table(index=["cohort", "record", "variant"], columns="check", values=cols)
        for c in cols:
            if ("modified", c) in [(b, a) for a, b in w.columns]:
                d = (w[(c, "modified")] - w[(c, "released")]).abs()
                say(f"    {c:<12} max {d.max():.4f}  nights identical (< 0.01) {int((d < 0.01).sum())} of {d.notna().sum()}")
        say()
    for part, a, b, label in (("mgh", "as_before", "arousals", "MGH with its scored arousals"),
                              ("mgh", "as_before", "heartbeat_removed", "MGH, heartbeat removed"),
                              ("boas", "as_before", "heartbeat_removed", "BOAS, heartbeat removed")):
        g = df[df.part == part]
        if g.empty:
            continue
        say(f"   RBDtector, {label}: share flagged (chin any >= 23.5%) and median any %")
        for (cohort, variant), s in g.groupby(["cohort", "variant"]):
            w = s.pivot_table(index="record", columns="check", values="any_pct")
            if a not in w or b not in w:
                continue
            w = w.dropna(subset=[a, b])
            pa, pb = (w[a] >= 23.5).mean(), (w[b] >= 23.5).mean()
            say(f"    {cohort:<7}{variant:<10} n={len(w):3d}  flagged {100 * pa:5.1f}% -> {100 * pb:5.1f}% "
                f"({100 * (pb - pa):+.1f} points)  median any {w[a].median():5.1f} -> {w[b].median():5.1f}")
        say()
    (config.RESULTS / "rbdtector_checks_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--part", choices=list(PARTS))
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--mirror", default=None)
    ap.add_argument("--report", action="store_true")
    args = ap.parse_args()
    if args.report:
        report()
        return
    from src import rbdtector as rt
    rt.install()
    todo = jobs(args.part)
    OUT = out_file(args.part)
    if OUT.exists():
        old = pd.read_csv(OUT, dtype={"record": str})
        seen = set(zip(old.cohort, old.record))
        todo = [j for j in todo if j not in seen]
    print(f"{args.part}: {len(todo)} nights to score", flush=True)
    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(_work, c, r, args.part, str(config.SCRATCH)) for c, r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            rec, rows, err = f.result()
            if err:
                print(f"[{i}/{len(todo)}] {rec} FAILED {err}", flush=True)
                continue
            new = pd.DataFrame(rows)
            if OUT.exists():                    # same columns in every row of a part
                new = pd.concat([pd.read_csv(OUT, dtype={"record": str}), new], ignore_index=True)
            new.to_csv(OUT, index=False)
            if args.mirror:
                try:
                    shutil.copy2(OUT, Path(args.mirror) / OUT.name)
                except OSError as e:
                    print(f"  (mirror failed: {e})", flush=True)
            print(f"[{i}/{len(todo)}] {rec} done ({len(rows)} runs)", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_visual_package.py
"""Blinded visual-scoring substudy: the excerpts, the scoring sheets and the key.

Protocol: paper/visual_substudy_protocol.md (to be registered on OSF before any
scoring). Thirty nights are drawn with a fixed seed: 10 CAP RBD (CAP's main
system), 10 MGH and 10 HMC. From each night one stretch of 20 consecutive REM
epochs (the first such stretch; the longest REM period if none is that long) is
exported with 4 epochs of context on each side, twice: as recorded, and with
band-limited white noise added to the chin EMG at the level that raises the
night's quiet-REM level to 1.5 uV (the noise ladder's step h150). The 60 excerpts
are numbered in random order, so the scorer cannot tell the two copies of a night
apart or which cohort an excerpt comes from.

    python run_visual_package.py --out /content/drive/MyDrive/rbd-rwa/visual_study

-> <out>/excerpts/V01.edf ... V60.edf (chin EMG, central EEG, ECG; stages and the
   epochs to score as EDF+ annotations), <out>/scoring_minis.csv and
   <out>/scoring_epochs.csv (blank sheets), <out>/README_for_scorer.md;
   <out>_KEY/key.csv (which night and copy each excerpt is, and the automated
   scores of the same epochs) - for the organizer only, never for the scorer.
"""
import argparse
import sys
import zlib
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

SEED = 20261027
N_PER = 10
N_SCORE, CONTEXT = 20, 4
STAGE = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "R"}


def pick_nights():
    from run_rbdtector import records_for
    rng = np.random.default_rng(SEED)
    out = []
    for cohort, who in (("cap", "CAP RBD"), ("chal18", "MGH"), ("hmc", "HMC")):
        recs, labels = records_for(cohort, 0)
        recs = [r for r in recs if labels[r] == who]
        out += [(cohort, r) for r in sorted(rng.choice(recs, N_PER, replace=False))]
    return out


def scored_stretch(hyp):
    """Epoch numbers of the 20 REM epochs to score."""
    rem = np.flatnonzero(np.asarray(hyp) == 4)
    runs = np.split(rem, np.flatnonzero(np.diff(rem) != 1) + 1) if len(rem) else []
    long = [r for r in runs if len(r) >= N_SCORE]
    if long:
        return long[0][:N_SCORE]
    return max(runs, key=len) if runs else np.array([], int)


def automated(xf, fs, hyp, epochs):
    """Automated SINBAR "any" % (3-s mini-epochs) and tonic % (30-s epochs) on the
    scored epochs, published and floor-corrected, with the whole night's background."""
    from src.config import ENV_FS
    from src.extract import bin_envelope
    from src.pack import build_pack
    from src.rwa import METHODS, floor_corrected, score_night
    rms, mav = bin_envelope(xf, fs)
    pack = build_pack({"rms": rms, "mav": mav}, [], hyp, ENV_FS, 30, {"unit_known": True})
    pos = np.flatnonzero(np.isin(pack["rem_epochs"], epochs))
    out = {}
    for name, m in (("published", METHODS["sinbar"]), ("FC", floor_corrected(METHODS["sinbar"], 1.0))):
        masks = {}
        score_night(pack, m, "native", masks=masks)
        anym = masks["any"].reshape(-1, 10)[pos]
        out[f"auto_any_{name}"] = round(100 * anym.mean(), 2)
        out[f"auto_tonic_{name}"] = round(100 * masks["tonic"][pos].mean(), 2)
    return out


def write_edf(path, signals, start_s, hyp, epochs, a):
    import pyedflib
    from pyedflib import highlevel
    import inspect
    rate = "sample_frequency" if "sample_frequency" in \
        inspect.signature(highlevel.make_signal_header).parameters else "sample_rate"
    sigs, heads = [], []
    for label, x, fs in signals:
        lim = float(np.ceil(np.percentile(np.abs(x), 99.99) * 1.5 + 1))
        sigs.append(np.clip(x, -lim, lim))
        heads.append(highlevel.make_signal_header(label, dimension="uV", physical_min=-lim,
                                                  physical_max=lim, **{rate: int(fs)}))
    ann = [[float(i * 30), 30.0, f"Sleep stage {STAGE.get(int(c), '?')}"] for i, c in enumerate(hyp)]
    ann += [[float((e - a) * 30), 30.0, "SCORE THIS EPOCH"] for e in epochs]
    hdr = highlevel.make_header(annotations=ann)
    highlevel.write_edf(str(path), sigs, heads, hdr, file_type=pyedflib.FILETYPE_EDFPLUS)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    from src.cohorts import COHORTS, _beats
    from src.config import SCRATCH
    from src.ecg import subtract_heartbeat
    from src.extract import bandpass
    from src.noise_arms import harmonized, white
    out, key_dir = Path(args.out), Path(str(args.out).rstrip("/") + "_KEY")
    (out / "excerpts").mkdir(parents=True, exist_ok=True)
    key_dir.mkdir(parents=True, exist_ok=True)
    nights = pick_nights()
    ids = np.random.default_rng(SEED + 1).permutation(2 * len(nights)) + 1
    key, minis, eps = [], [], []
    for i, (name, rec) in enumerate(nights):
        cohort = COHORTS[name]
        folder = SCRATCH / f"vis_{rec}"
        folder.mkdir(parents=True, exist_ok=True)
        d = cohort.read(rec, cohort.fetch(rec, folder))
        fs, hyp = float(d["fs"]), np.asarray(d["hypnogram"])
        xf, _ = bandpass(d["chin"], fs, mains=cohort.mains)
        beats = _beats(d)
        if len(beats):
            xf = subtract_heartbeat(xf, fs, beats)
        track = white(len(xf), fs, cohort.mains, rec)
        _, _, s, level = harmonized(xf, track, fs, hyp, 1.5)
        epochs = scored_stretch(hyp)
        a = max(0, int(epochs[0]) - CONTEXT)
        b = min(len(hyp), int(epochs[-1]) + CONTEXT + 1)
        for j, (copy, add) in enumerate((("as recorded", 0.0), ("noise", s))):
            vid = f"V{ids[2 * i + j]:02d}"
            sl = lambda x, f: np.asarray(x[int(a * 30 * f): int(b * 30 * f)], float)
            chin = sl(d["chin"], fs) + add * sl(track, fs)
            signals = [("Chin EMG", chin, fs)]
            if d.get("eeg") is not None:
                signals.append(("EEG central", sl(d["eeg"], d["fs_eeg"]), d["fs_eeg"]))
            if d.get("ecg") is not None:
                signals.append(("ECG", sl(d["ecg"], d["fs_ecg"]), d["fs_ecg"]))
            write_edf(out / "excerpts" / f"{vid}.edf", signals, a * 30, hyp[a:b], epochs, a)
            auto = automated(xf + add * track, fs, hyp, epochs)
            key.append({"excerpt": vid, "cohort": name, "record": rec, "copy": copy,
                        "added_uv": round(add, 3), "level_after_uv": round(level if add else np.nan, 3),
                        "scored_epochs": len(epochs), **auto})
            for k, e in enumerate(epochs):
                t0 = (int(e) - a) * 30
                eps.append({"excerpt": vid, "epoch": k + 1, "start_s": t0, "tonic": ""})
                minis += [{"excerpt": vid, "epoch": k + 1, "mini": m + 1, "start_s": t0 + 3 * m, "any": ""}
                          for m in range(10)]
        print(f"[{i + 1}/{len(nights)}] {name} {rec}: {len(epochs)} epochs, noise {s:.2f} uV", flush=True)
        import shutil
        shutil.rmtree(folder, ignore_errors=True)
    pd.DataFrame(key).sort_values("excerpt").to_csv(key_dir / "key.csv", index=False)
    pd.DataFrame(minis).sort_values(["excerpt", "epoch", "mini"]).to_csv(out / "scoring_minis.csv", index=False)
    pd.DataFrame(eps).sort_values(["excerpt", "epoch"]).to_csv(out / "scoring_epochs.csv", index=False)
    (out / "README_for_scorer.md").write_text(README, encoding="utf-8")
    print(f"wrote {len(key)} excerpts to {out}; key in {key_dir} (do not share with the scorer)")


README = """# Visual scoring of chin EMG in REM sleep - instructions

You have 60 EDF+ excerpts (V01-V60). Each holds a stretch of REM sleep with a few epochs of
context, with the chin EMG, a central EEG channel and (where recorded) the ECG. Sleep stages
are given as annotations; the 20 epochs to score are annotated "SCORE THIS EPOCH".

Please score the chin EMG of those epochs only, following the SINBAR rules (Frauscher et al.,
Sleep 2012):

1. scoring_minis.csv: for each 3-s mini-epoch, any = 1 if it contains any chin EMG activity
   (tonic or phasic) of at least 0.1 s and at least twice the amplitude of the background EMG,
   otherwise 0. Mark artifacts you would exclude as "X".
2. scoring_epochs.csv: for each 30-s epoch, tonic = 1 if at least 50% of the epoch shows tonic
   chin activity at least twice the background (or above 10 uV), otherwise 0.

Use your usual display settings (filters, gain) and record them in a note. Score the excerpts
in numerical order, independently of each other, and do not look for other copies of the same
night. Return the two CSV files.
"""

if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_visual_analysis.py
"""Analysis of the blinded visual-scoring substudy (paper/visual_substudy_protocol.md).

    python run_visual_analysis.py --sheets <folder with the scorer's two CSVs> --key <key.csv>

Per excerpt: visual SINBAR "any" % (3-s mini-epochs scored 1, artifacts "X" left
out) and tonic % (30-s epochs), against the automated scores of the same epochs.

  H1  agreement: ICC(2,1), absolute agreement, and Bland-Altman bias and 95% limits,
      visual vs automated published and vs floor-corrected (excerpts as recorded)
  H2  the noise floor in visual scoring: per night, the change in visual "any" %
      from the excerpt as recorded to its noisy copy (mean, 95% bootstrap CI, Wilcoxon);
      the same for the automated published and floor-corrected scores
  H3  the share of the automated published change that visual scoring reproduces:
      mean visual change / mean automated published change, with bootstrap CI
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def icc_a1(x, y):
    """ICC(2,1): two-way random effects, absolute agreement, single rater (Shrout-Fleiss)."""
    m = np.c_[x, y].astype(float)
    m = m[np.isfinite(m).all(axis=1)]
    n, k = m.shape
    grand = m.mean()
    msr = k * ((m.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    msc = n * ((m.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    sse = ((m - m.mean(axis=1, keepdims=True) - m.mean(axis=0, keepdims=True) + grand) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)


def boot_mean(v, n=2000, seed=7):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    rng = np.random.default_rng(seed)
    bs = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sheets", required=True)
    ap.add_argument("--key", required=True)
    args = ap.parse_args()
    key = pd.read_csv(args.key)
    minis = pd.read_csv(Path(args.sheets) / "scoring_minis.csv", dtype={"any": str})
    eps = pd.read_csv(Path(args.sheets) / "scoring_epochs.csv", dtype={"tonic": str})
    minis = minis[minis["any"].str.strip().isin(["0", "1"])]
    eps = eps[eps["tonic"].str.strip().isin(["0", "1"])]
    vis = pd.DataFrame({
        "vis_any": minis.groupby("excerpt")["any"].apply(lambda s: 100 * (s.str.strip() == "1").mean()),
        "vis_tonic": eps.groupby("excerpt")["tonic"].apply(lambda s: 100 * (s.str.strip() == "1").mean()),
        "artifact_minis": pd.read_csv(Path(args.sheets) / "scoring_minis.csv", dtype={"any": str})
        .groupby("excerpt")["any"].apply(lambda s: (s.str.strip().str.upper() == "X").sum())})
    d = key.merge(vis, left_on="excerpt", right_index=True, how="left")
    lines = []
    say = lambda s="": (print(s), lines.append(s))
    rec = d[d["copy"] == "as recorded"]
    say("H1  Agreement on the excerpts as recorded (n = %d)" % rec.vis_any.notna().sum())
    for auto in ("published", "FC"):
        for meas in ("any", "tonic"):
            x, y = rec[f"vis_{meas}"], rec[f"auto_{meas}_{auto}"]
            diff = (y - x).dropna()
            say(f"    {meas:<6} visual vs automated {auto:<10} ICC(2,1) {icc_a1(x, y):.2f}  "
                f"bias {diff.mean():+.1f} points, 95% limits {diff.mean() - 1.96 * diff.std():+.1f} to "
                f"{diff.mean() + 1.96 * diff.std():+.1f}")
    for cohort, g in rec.groupby("cohort"):
        say(f"    {cohort:<7} visual any {g.vis_any.mean():.1f}%  automated published {g.auto_any_published.mean():.1f}%  "
            f"FC {g.auto_any_FC.mean():.1f}%")
    say()
    w = d.pivot_table(index=["cohort", "record"], columns="copy",
                      values=["vis_any", "auto_any_published", "auto_any_FC"])
    say("H2  Change from the excerpt as recorded to its noisy copy (points): mean [95% CI], Wilcoxon p")
    ch = {}
    for col, label in (("vis_any", "visual"), ("auto_any_published", "automated published"),
                       ("auto_any_FC", "automated floor-corrected")):
        c = (w[(col, "noise")] - w[(col, "as recorded")]).dropna()
        ch[label] = c
        m, lo, hi = boot_mean(c)
        p = wilcoxon(c).pvalue if (c != 0).any() else 1.0
        say(f"    {label:<28} {m:+6.1f} [{lo:+.1f} to {hi:+.1f}]  p {p:.3g}  (n = {len(c)} nights)")
        for cohort in ("cap", "chal18", "hmc"):
            cc = c.loc[cohort] if cohort in c.index.get_level_values(0) else pd.Series(dtype=float)
            if len(cc):
                say(f"        {cohort:<7} {cc.mean():+6.1f}")
    say()
    v, a = ch["visual"], ch["automated published"]
    both = pd.concat([v, a], axis=1, keys=["v", "a"]).dropna()
    rng = np.random.default_rng(8)
    ratios = []
    for _ in range(2000):
        i = rng.integers(0, len(both), len(both))
        s = both.iloc[i]
        if s.a.mean() != 0:
            ratios.append(s.v.mean() / s.a.mean())
    say(f"H3  Share of the automated published change reproduced visually: {both.v.mean() / both.a.mean():.2f} "
        f"[{np.percentile(ratios, 2.5):.2f} to {np.percentile(ratios, 97.5):.2f}]")
    out = Path(args.sheets) / "visual_report.md"
    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    d.to_csv(Path(args.sheets) / "visual_scores.csv", index=False)
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_site.py
"""Run the RWA analyses on recordings that cannot leave their center.

For a sleep center holding polysomnograms of patients with RBD or Parkinson's
disease: everything runs on the center's own computer, and the output is one row
of numbers per night (no signal, no date, no identifier beyond the center's own
code), which the center can share or keep.

    python run_site.py manifest.csv --out site_results
    python run_site.py manifest.csv --out site_results --ladder   # + a 1.5-uV copy

manifest.csv, one row per night (header required):
    code,edf,stages,group[,age,sex,chin,ecg,mains]
  code    the center's own pseudonym for the night (e.g. RBD017)
  edf     path to the EDF/EDF+ file (chin EMG in physical units, uV or mV)
  stages  30-s stages from the start of the EDF, one per line: W, N1, N2, N3, R
  group   e.g. iRBD, PD+RBD, PD-RBD, control
  age, sex           optional
  chin, ecg          optional channel labels (default: found by name)
  mains              50 or 60 (default 50)

Outputs (in --out):
  site_scores.csv    per night: REM minutes, quiet-REM level (noise floor), net
                     N2/N3 level (gain check), and for the Montreal and SINBAR chin
                     criteria (published and floor-corrected, with the REM and the
                     N2/N3 noise estimate) and the REM Atonia Index: tonic, phasic
                     and "any" percentages and the verdict. With --ladder, the same
                     night again with white noise raising its quiet-REM level to
                     1.5 uV (the paper's noise ladder), so the effect of a noisier
                     system can be measured on the center's own patients.
  site_summary.md    by group: nights, floor, share flagged by each rule.
"""
import argparse
import csv
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src.config import ENV_FS
from src.ecg import find_ecg, plausible, r_peaks, subtract_heartbeat
from src.edf import read_header
from src.extract import bandpass, bin_envelope, find_emg, read_source
from src.floorcheck import read_stages
from src.pack import build_pack
from src.rwa import METHODS, floor_corrected, rem_atonia_index, score_night

EPOCH_SEC = 30


def variants():
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out += [("published", m), ("FC", floor_corrected(m, 1.0)),
                ("FC-nrem", floor_corrected(m, 1.0, noise_from="nrem"))]
    return out


def find_chin(hdr):
    """The pipeline's chin finder, then any single channel named like a chin EMG."""
    src = find_emg(hdr)[0]
    if src is None:
        norm = lambda c: c.label.upper().replace(" ", "").replace("-", "")
        hits = [c for c in hdr.channels if any(k in norm(c) for k in ("CHIN", "MENT", "SUBM"))]
        src = hits[0] if len(hits) >= 1 else None
    return src


def one_night(row, ladder):
    hdr = read_header(row["edf"])
    src = hdr.find(row["chin"]) if row.get("chin") else find_chin(hdr)
    if src is None:
        raise ValueError(f"no chin EMG among {[c.label for c in hdr.channels]}")
    chin, fs, unit_known, label, _ = read_source(hdr, src)
    if not unit_known:
        raise ValueError(f"{label}: no physical unit in the EDF header")
    mains = float(row.get("mains") or 50)
    ecg_src = hdr.find(row["ecg"]) if row.get("ecg") else find_ecg(hdr)
    beats = np.empty(0)
    if ecg_src is not None:
        e, fs_e = read_source(hdr, ecg_src)[:2]
        beats = r_peaks(e, fs_e)
        if not plausible(beats, len(e) / fs_e):
            beats = np.empty(0)
    xf, band = bandpass(chin, fs, mains=mains)
    if len(beats):
        xf = subtract_heartbeat(xf, fs, beats)
    hyp = np.asarray(read_stages(row["stages"]))
    rows_env = {}
    rms, mav = bin_envelope(xf, fs)
    rows_env["rms"], rows_env["mav"] = rms, mav
    if ladder:
        from src.noise_arms import harmonized, white
        r, m, s, got = harmonized(xf, white(len(xf), fs, mains, row["code"]), fs, hyp, 1.5)
        rows_env["rms_white_h150"], rows_env["mav_white_h150"] = r, m
    pack = build_pack(rows_env, [], hyp, ENV_FS, EPOCH_SEC, {"record": row["code"], "unit_known": True})
    from run_floor import scale
    info = {"code": row["code"], "group": row["group"], "age": row.get("age"), "sex": row.get("sex"),
            "chin": label, "fs": fs, "band": f"{band[0]:g}-{band[1]:g}", "beats": int(len(beats)),
            **{k: round(v, 3) for k, v in scale(pack).items()}}
    out = []
    for copy in (("native", "white_h150") if ladder else ("native",)):
        for variant, m in variants():
            r = score_night(pack, m, copy)
            out.append({**info, "copy": copy, "variant": variant, **{k: r.get(k) for k in (
                "method", "rem_minutes", "background_phasic_uV", "tonic_pct", "phasic_pct", "any_pct",
                "positive", "error")}})
        r = rem_atonia_index(pack, copy)
        out.append({**info, "copy": copy, "variant": "published", "method": "RAI",
                    "rem_minutes": r.get("rem_minutes"), "rai": r.get("rai"), "positive": r.get("positive"),
                    "error": r.get("error")})
    return out


def summary(df):
    lines = ["# RWA site summary", "",
             "Share of nights flagged positive (%), by group; floor = quiet-REM level (uV, median).", ""]
    if df.empty:
        return "# RWA site summary\n\nNo night could be scored.\n"
    d = df[(df["copy"] == "native") & (df.rem_minutes >= 10)]
    for group, g in d.groupby("group"):
        floor = g[(g.method == "Montreal") & (g.variant == "published")].background_phasic_uV.median()
        lines.append(f"## {group}: {g.code.nunique()} nights, floor {floor:.2f} uV")
        for (meth, variant), s in g.groupby(["method", "variant"]):
            v = s.positive.map(lambda x: x in (True, "True")).astype(float)
            lines.append(f"- {meth} {variant}: {100 * v.mean():.0f}% ({int(v.sum())}/{len(v)})")
        lines.append("")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("manifest")
    ap.add_argument("--out", default="site_results")
    ap.add_argument("--ladder", action="store_true", help="also score a copy at a 1.5-uV floor")
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    with open(args.manifest, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    results, failed = [], []
    for i, row in enumerate(rows, 1):
        try:
            results += one_night(row, args.ladder)
            print(f"[{i}/{len(rows)}] {row['code']} done", flush=True)
        except Exception as e:
            failed.append((row["code"], f"{type(e).__name__}: {e}"))
            print(f"[{i}/{len(rows)}] {row['code']} FAILED {e}", flush=True)
    df = pd.DataFrame(results)
    df.to_csv(out / "site_scores.csv", index=False)
    (out / "site_summary.md").write_text(summary(df) + "\n" + "\n".join(
        f"- failed: {c}: {e}" for c, e in failed) + "\n", encoding="utf-8")
    print(f"wrote {out / 'site_scores.csv'} and {out / 'site_summary.md'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_revision_colab.sh
#!/bin/sh
# The revision analyses (paper/preregistration_revision.md) on a fresh Colab runtime,
# reusing the packs and scores already on Drive. Run from the code directory, e.g. in
# Colab's terminal:
#   cd /content/rbd-rwa && setsid nohup sh run_revision_colab.sh > /content/revision.log 2>&1 < /dev/null &
# PART=1: re-analyses, the noise arms and the visual-scoring excerpts (about 4 h);
# PART=2: the RBDtector checks (about 4 h, RBDtector's own environment). Without PART,
# both in turn. The injection experiment has its own runner (run_inject_colab.sh).
PART=${PART:-all}
D=/content/drive/MyDrive/rbd-rwa
export RBD_DURABLE_DIR=/content/durable PYTHONUNBUFFERED=1
R=$RBD_DURABLE_DIR/results
mkdir -p "$R" "$D/results"
for f in ladder_scores.csv floor_ladder_scores.csv floor_scores.csv external_scores.csv \
         rbdtector_scores.csv rbdtector_scores_systems.csv; do
  if [ -f "$D/results/$f" ] && [ ! -f "$R/$f" ]; then cp "$D/results/$f" "$R/"; fi
done
if [ "$PART" != 2 ]; then
  python run_revision.py && cp "$R"/revision_* "$D/results/"
  python run_noise_arms.py --library --mirror "$D"
  python run_noise_arms.py --cohort chal18 --mirror "$D"
  python run_noise_arms.py --cohort cap --mirror "$D"
  python run_noise_arms.py --report && cp "$R"/noise_arms_* "$D/results/"
  if [ ! -f "$D/visual_study_KEY/key.csv" ]; then
    pip -q install pyedflib
    python run_visual_package.py --out "$D/visual_study"
  fi
fi
if [ "$PART" != 1 ]; then
  E=/content/rbdt_env
  if [ ! -x "$E/bin/python" ]; then
    pip -q install uv
    uv venv -q -p 3.9 "$E"
    uv pip install -q -p "$E/bin/python" "numpy~=1.21.2" "pandas~=1.3.3" "scipy~=1.7.1" \
      "openpyxl~=3.0.8" "pyEDFlib~=0.1.22" "wfdb~=3.4.0"
  fi
  for part in verify mgh boas; do
    $E/bin/python run_rbdtector_checks.py --part $part --mirror "$D/results"
  done
  $E/bin/python run_rbdtector_checks.py --report && cp "$R"/rbdtector_checks* "$D/results/"
fi
echo REVISION_DONE

## 1. CAP: extract every night (~12 minutes), then pack

In [ ]:
!cd /content/rbd-rwa && python run_extract.py && python run_cohort.py --cohort cap {PACK_MIRROR}

## 2. HMC: 151 nights (about 15 GB streamed, ~10 minutes)

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort hmc --workers 3 {PACK_MIRROR}

## 3. Challenge 2018: 994 nights (about 130 GB streamed, roughly an hour)

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort chal18 --workers 3 {PACK_MIRROR}

## 4. Hardware simulation: HMC recorded the way MGH records chin EMG

The same 151 HMC nights again, quantized to 1 uV steps (`hmc_q1uv`), and also
resampled to 200 Hz (`hmc_mgh`), as the MGH chin EMG is stored. MGH's quiet
REM level (~0.6 uV) is below that step while HMC's (~1.5 uV) is above it, so
both are repeated after scaling HMC down to the MGH level (`hmc_low`,
`hmc_low_mgh`). Any change against the matching undegraded copy is caused by
the recording hardware alone (~25 minutes).

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort hmc_q1uv --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_mgh --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_low --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_low_mgh --workers 3 {PACK_MIRROR}

The reverse test: 300 MGH nights with white noise added to the chin EMG, which raises
their quiet-REM level from ~0.6 uV to HMC's ~1.5 uV (`chal18_noise`, ~15 minutes).

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort chal18_noise --limit 300 --workers 3 {PACK_MIRROR}

## 5. The noise ladder

CAP, HMC and 300 MGH nights streamed once more; each night's chin EMG is kept as
recorded and with white noise added at 0.5-3 uV, and at the level that brings its
quiet-REM level to 1.5 and 2.0 uV. The analysis and its decision criteria were fixed
in advance (paper/preregistration_noise_ladder.md). About 40 minutes; the ladder
files stay on this machine (they are large) and only the results go to Drive.

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort cap --ladder --workers 3 && python run_cohort.py --cohort hmc --ladder --workers 3 && python run_cohort.py --cohort chal18 --ladder --limit 300 --workers 3 && python run_ladder.py

## 5b. Floor-corrected criteria

The published rules with the noise floor subtracted instead of multiplied, scored on
every night as recorded and on the noise ladder, against the criteria fixed in advance
(paper/preregistration_floor_corrected.md). About 15 minutes.

In [ ]:
!cd /content/rbd-rwa && python run_floor.py

## 5c. External cohorts

Three recording systems not used to find the effect: BOAS (OpenNeuro, 128 nights of adults
from the general population), DOD-O (55 patients, Stanford) and DOD-H (25 healthy
volunteers, France), scored against the predictions fixed in advance
(paper/preregistration_external.md). BOAS takes ~15 minutes; the Dreem sets come from
Zenodo at a few MB/s and take about an hour each.

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort boas --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort dodo --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort dodh --workers 3 {PACK_MIRROR} && python run_external.py

## 5d. What the noise floor means for Parkinson's disease research

RBDtector, the open-source RWA scorer used in Parkinson's and RBD research, run unchanged
(and with the floor-corrected baseline) on the CAP noise ladder and on MGH, HMC and BOAS
as recorded; then the analyses fixed in advance in paper/preregistration_parkinsons.md
(change in RWA against the disease signals PD research uses, isolated RWA by system,
recruitment arithmetic, the floor check), and the exploratory check of RBDtector on noise
alone. RBDtector runs in the 2021 environment its requirements specify, which the script
creates. About seven hours on one runtime (about 35 s per night, step and rule); to split
it, run `PART=2 sh run_parkinsons_colab.sh` (MGH, HMC, BOAS) on a second runtime and
`PART=1 sh run_parkinsons_colab.sh` (CAP, then the analysis) here; both need Drive.

In [ ]:
!cd /content/rbd-rwa && sh run_parkinsons_colab.sh

## 5e. Known muscle activity added to every night

Ground truth that does not depend on the 21 RBD patients: muscle activity of known size is
added to the REM sleep of every night without RBD on the six recording systems (synthetic
tonic and phasic activity at 49 amplitudes), and the REM muscle activity of the 21 CAP RBD
patients is transplanted into the same nights at full and half amplitude. Scored with the
published and floor-corrected rules against the criteria fixed in advance
(paper/preregistration_injection.md). All nights are streamed again; about five hours on one
runtime. To split it, run `PART=2 sh run_inject_colab.sh` (the Dreem sets) on a second
runtime and `PART=1 sh run_inject_colab.sh` here; both need Drive.

In [ ]:
!cd /content/rbd-rwa && sh run_inject_colab.sh

## 5f. Analyses added in revision

Specified in paper/preregistration_revision.md before they were run: every between-site
difference with confidence intervals and equivalence tests, the floor-corrected rule on
in-range nights and with a noise estimate from N2/N3, the noise ladder with pink, drifting
and recorded noise, the RBDtector checks (against the program as released, with MGH's
scored arousals, with the heartbeat removed), and the blinded excerpts for the visual-scoring
substudy (paper/visual_substudy_protocol.md). About eight hours; `PART=1` (re-analyses,
noise, excerpts) and `PART=2` (RBDtector) can run on two runtimes.

In [ ]:
!cd /content/rbd-rwa && sh run_revision_colab.sh

The floor check for a recording of your own: an EDF on Drive and its 30-s stages (one per
line: W, N1, N2, N3, R). It reports the system's noise floor, whether the floor-corrected
criteria apply, and what each RWA threshold means in microvolts.

In [ ]:
# !cd /content/rbd-rwa && python check_floor.py /content/drive/MyDrive/night.edf /content/drive/MyDrive/night_stages.txt --mains 50

## 6. Score every night with every method

In [ ]:
!cd /content/rbd-rwa && python run_score.py {RESULT_MIRROR}

## 7. Background definition sensitivity, then the report tables

In [ ]:
!cd /content/rbd-rwa && python run_sensitivity.py && python run_report.py
if MIRROR:
    !mkdir -p {MIRROR}/results && cp /content/durable/results/* {MIRROR}/results/

## 8. Figures

In [ ]:
!cd /content/rbd-rwa && python run_figures.py
if MIRROR:
    !mkdir -p {MIRROR}/figures && cp /content/durable/figures/* {MIRROR}/figures/
from IPython.display import Image, display
for name in sorted(os.listdir('/content/durable/figures')):
    if name.endswith('.png'):
        display(Image(f'/content/durable/figures/{name}', width=900))